# FOWT-ARISE — Proposed Model

## FOWT-ARISE: A Physics-Informed Adaptive Reinforcement Learning Framework for Real-Time Structural Load Relief in Floating Offshore Wind Turbines Under Imperfect IoT Observations

---

## 01. Research Overview

### Central contribution

> FOWT-ARISE transforms the physics-informed FLOATBench fatigue benchmark into a sequential,
> IoT-aware decision environment in which a single RL policy learns **what structural load is
> controllable**, **which actuator to use**, **how aggressively to act**, and **when not to act** —
> while balancing fatigue reduction against power and actuation costs.

### The four novelties

| | Novelty | What it changes |
|---|---|---|
| **N1** | Physics-Informed Load-Aware State Representation | State carries load-path decomposition, governing-section fatigue, and controllable-load share alongside the ordinary operating variables |
| **N2** | Adaptive Multi-Actuator Control | **One** policy jointly commands pitch / yaw / IPC, with a *learned* control-authority gate that decides when **not** to act |
| **N3** | Fatigue–Power–Actuation Multi-Objective Reward | Reward balances load relief against power loss, actuator duty, and action smoothness, each normalised |
| **N4** | IoT-Degradation-Aware Robust RL | Trains and evaluates under Gaussian noise / dropout / persistent bias / stale observations, with a clean-vs-degraded consistency objective |

A single unified controller is trained. There are **no** three independent per-actuator policies in
any configuration, and the no-action behaviour is **learned**, never a hand-written threshold.

### Base RL algorithm, kept separate from the novelties

The underlying learner is an **offline actor–critic** in the TD3+BC / CQL family: twin critics,
target networks, target-policy smoothing, delayed policy updates, a conservative (CQL) critic
penalty, and a behaviour-support regulariser. The FLOATBench transitions were collected under a
*mixture of behaviour policies*, not the policy being trained, so this is an **offline** RL problem.
N1–N4 sit **on top of** that base algorithm and are toggled independently by the ablation study.

### Datasets consumed (never modified, never fabricated)

* **Transitions** — `transitions_{tower}.parquet`, 129,600 rows x 89 columns total across 3 towers,
  1,200 `episode_id` values per tower, i.e. **3,600 distinct trajectories** keyed by
  `(tower, episode_id)`.
* **Action sweep** — `action_sweep_{tower}.parquet`, **1,455,300 rows x 27 columns** total, a full
  factorial pitch x yaw x IPC grid evaluated at every operating condition. Used **only** for
  counterfactual evaluation and for validation-based model selection — never as a training target.

### Reading guide

Sections 02–12 prepare and audit the data. Sections 13–23 build the model and training machinery.
Sections 24–29 train and evaluate FOWT-ARISE. Sections 30–35 run the four ablations and compare
them. Sections 36–37 integrate externally supplied baselines. Sections 38–44 produce plots, SHAP
explanations, a validation checklist, and the final summary.

## 02. Configuration

**This is the only cell you need to edit.** Change the five paths at the top; everything below them
has a working default and every other cell in the notebook derives what it needs from here. Column
names, action limits, normalisation, architecture, output paths, seeds, evaluation functions and
plotting are all derived automatically and are **not** listed here.

Set `DRY_RUN = True` first to validate the whole pipeline in ~2 epochs per experiment, then set it
to `False` for the real run.

In [ ]:
# =============================================================================
# EDIT ONLY THIS BLOCK
# =============================================================================
DATASET_PATH           = "CHANGE_THIS"   # dir containing transitions_*.parquet
ACTION_SWEEP_PATH      = "CHANGE_THIS"   # dir containing action_sweep_*.parquet
OUTPUT_ROOT            = "CHANGE_THIS"   # all outputs are written under here
RESUME_FROM_CHECKPOINT = ""              # "" = fresh; else path to a checkpoint .pt
BASELINE_DIR           = ""              # "" = skip baseline comparison; else dir with RB-FOWT/CQL/IQL outputs

DRY_RUN = True                           # True => tiny run that exercises every code path
# =============================================================================
# END OF EDITABLE BLOCK -- everything below is a documented default, safe to leave alone
# =============================================================================

# ---- reproducibility -------------------------------------------------------
SEED = 42

# ---- episode-level split ---------------------------------------------------
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

# ---- training (Section 22) -------------------------------------------------
BATCH_SIZE        = 256
NUM_EPOCHS_FULL   = 60
NUM_EPOCHS_DRY    = 2
LEARNING_RATE     = 3e-4
LEARNING_RATE_ACTOR = 1e-3     # actor is driven by imitation; benefits from a larger step
GAMMA             = 0.99
TAU               = 0.005
POLICY_NOISE      = 0.1
NOISE_CLIP        = 0.3
POLICY_DELAY      = 2
GRAD_CLIP_NORM    = 5.0
CQL_ALPHA         = 1.0        # conservative critic penalty (Section 17)
WEIGHT_DECAY      = 1e-5

# ---- actor objective (Section 15/17) ---------------------------------------
# The actor's PRIMARY signal is advantage-selected best-action imitation: for each physical
# operating point the single highest-reward action observed in the TRAINING episodes becomes the
# regression target. BEST_ACTION_GROUP_COLS defines "operating point" (turbulence seed is
# deliberately marginalised over, so the target is the best action for the physical conditions).
BEST_ACTION_GROUP_COLS = ["tower", "wind_speed_id", "wave_hs_id", "wave_tp_id"]
IMITATION_HUBER_DELTA  = 0.05   # 0 => MSE; >0 => Huber, which reduces mode-averaging

# Weight on the policy-improvement gradient through the critics. DEFAULT 0.0, and that is an
# EMPIRICAL FINDING on this dataset rather than an oversight: the critic's measured RMSE is larger
# than the entire decision-relevant reward range, so dQ/da is not informative enough to improve the
# policy and empirically drives it toward heavy feathering. Section 25 measures and prints this, so
# the default is auditable rather than asserted. Twin + conservative critics are still built,
# trained, logged and used for diagnostics regardless of this value.
Q_IMPROVEMENT_COEF = 0.0

# Actor output scale before clamping to [-1, 1]. >1 makes the CORNERS of the action box exactly
# attainable; with plain tanh, "pitch = 0" and "IPC in {0,1}" are asymptotes the policy can never
# reach, yet the physics-optimal action sits on those corners in most operating conditions.
ACTOR_OUTPUT_SCALE = 1.15
LATENT_DIM         = 64

# ---- N3 multi-objective reward weights (Section 14) ------------------------
# REWARD_WEIGHT_PRESET picks between two documented, self-consistent choices:
#   "dataset_consistent" (DEFAULT) -- reproduces the dataset's OWN `reward` column exactly, so every
#       number in this notebook is directly comparable with the dataset and with any baseline scored
#       against it. Verified by reconstruction check in Section 14.
#   "spec_default" -- the illustrative weights from the project brief. These do NOT reproduce the
#       dataset's native reward, so the reconstruction check is reported as a warning rather than
#       enforced, and cross-comparability with the baselines is lost.
REWARD_WEIGHT_PRESET = "dataset_consistent"
_REWARD_PRESETS = {
    "dataset_consistent": dict(fatigue=2.0, power=1.0, actuation=0.05, smoothness=0.02),
    "spec_default":       dict(fatigue=1.0, power=0.5, actuation=0.25, smoothness=0.10),
}
LAMBDA_FATIGUE    = _REWARD_PRESETS[REWARD_WEIGHT_PRESET]["fatigue"]
LAMBDA_POWER      = _REWARD_PRESETS[REWARD_WEIGHT_PRESET]["power"]
LAMBDA_ACTUATION  = _REWARD_PRESETS[REWARD_WEIGHT_PRESET]["actuation"]
LAMBDA_SMOOTHNESS = _REWARD_PRESETS[REWARD_WEIGHT_PRESET]["smoothness"]

# ---- N4 IoT degradation (Section 13) --------------------------------------
IOT_NOISE_STD           = 0.05   # gaussian noise, in units of each channel's TRAIN std
IOT_DROPOUT_PROB        = 0.10   # per-step packet loss (hold-last within trajectory)
IOT_BIAS_MAGNITUDE      = 0.05   # per-trajectory constant bias, in units of channel TRAIN std
IOT_STALE_PROB          = 0.10   # per-step probability of reporting the previous reading
TRAIN_DEGRADED_FRACTION = 0.5    # fraction of training batches drawn under degradation
LAMBDA_ROBUST           = 0.1    # weight on the clean-vs-degraded consistency loss (Section 21)

# ---- N2 adaptive control gate (Section 16) --------------------------------
# Auxiliary supervision weight for the gate's "is acting beneficial here?" head. The gate is
# learned end-to-end from the action loss; this small auxiliary term additionally grounds it
# against whether the TRAIN best-action target at that operating point is non-neutral, which
# keeps the gate semantically interpretable instead of drifting into an arbitrary scaling.
LAMBDA_GATE_AUX = 0.1

# ---- early stopping / LR plateau (Sections 22, 23, 26) --------------------
EARLY_STOPPING_PATIENCE = 10
MIN_DELTA               = 1e-4
LR_PLATEAU_FACTOR       = 0.5
LR_PLATEAU_PATIENCE     = 4
MIN_LR                  = 1e-6
CHECKPOINT_EVERY        = 5

# ---- fatigue model (Section 09) -------------------------------------------
# Wohler exponent. None => derive empirically from the TRAIN split and cross-check against the
# dataset's own documented value. Never silently hard-coded.
WOHLER_EXPONENT_M = None

# ---- evaluation ----------------------------------------------------------
NO_ACTION_TOLERANCE = 0.05   # |a - neutral| <= this (normalised, all dims) counts as "no action"

# ---- SHAP (Sections 42) ---------------------------------------------------
SHAP_BACKGROUND_SIZE  = 100   # drawn from TRAIN only
SHAP_TEST_SAMPLE_SIZE = 200   # drawn from TEST only
SHAP_NSAMPLES         = 100

# ---- plotting (Sections 38-41) -------------------------------------------
FONT_SIZE = 20
DPI       = 300

NUM_EPOCHS = NUM_EPOCHS_DRY if DRY_RUN else NUM_EPOCHS_FULL

print("Configuration loaded.")
print(f"  DATASET_PATH           = {DATASET_PATH}")
print(f"  ACTION_SWEEP_PATH      = {ACTION_SWEEP_PATH}")
print(f"  OUTPUT_ROOT            = {OUTPUT_ROOT}")
print(f"  RESUME_FROM_CHECKPOINT = {RESUME_FROM_CHECKPOINT!r}")
print(f"  BASELINE_DIR           = {BASELINE_DIR!r}")
print(f"  DRY_RUN                = {DRY_RUN}   -> NUM_EPOCHS = {NUM_EPOCHS}")
print(f"  SEED                   = {SEED}")
print(f"  REWARD_WEIGHT_PRESET   = {REWARD_WEIGHT_PRESET} -> "
      f"fat={LAMBDA_FATIGUE} pow={LAMBDA_POWER} act={LAMBDA_ACTUATION} smooth={LAMBDA_SMOOTHNESS}")

## 03. Imports and Reproducibility

Every random source that affects the experiment is seeded from `SEED`. The environment (Python /
PyTorch / NumPy / pandas / CUDA versions) is recorded to `common/environment.json` so a run can be
reproduced or a discrepancy diagnosed later. Device selection is automatic: CUDA when available,
CPU otherwise, and the notebook is functionally identical either way.

In [ ]:
import os, sys, json, math, time, random, warnings, gc, platform
from pathlib import Path
from dataclasses import dataclass, field, asdict
from copy import deepcopy
from typing import Optional, Sequence

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")          # headless-safe; every figure is saved to disk
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import shap
    _SHAP_AVAILABLE = True
except Exception as _e:
    _SHAP_AVAILABLE = False
    warnings.warn(f"[FOWT-ARISE] shap unavailable ({_e}); Section 42 will be skipped explicitly.")

try:
    import openpyxl  # noqa: F401   (pandas .to_excel backend)
    _OPENPYXL_AVAILABLE = True
except Exception:
    _OPENPYXL_AVAILABLE = False
    warnings.warn("[FOWT-ARISE] openpyxl unavailable; .xlsx exports skipped (.csv still written).")

plt.rcParams["font.size"] = FONT_SIZE


def set_global_seed(seed: int) -> None:
    """Seed every source of randomness that can affect this experiment."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"GPU:    {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU run)'}")
print(f"CUDA:   {torch.version.cuda if torch.cuda.is_available() else 'not available'}")
print()
print(f"python  {platform.python_version()}")
print(f"torch   {torch.__version__}")
print(f"numpy   {np.__version__}")
print(f"pandas  {pd.__version__}")
print(f"shap    {'available' if _SHAP_AVAILABLE else 'NOT AVAILABLE'}")
print(f"\nGlobal seed set to {SEED}.")

### 03b. Output tree

Every directory the notebook writes to is created here, before anything is written, so no later
cell can fail on a missing path. One directory per experiment keeps proposed-model artefacts and
ablation artefacts strictly separate.

In [ ]:
EXPERIMENTS = ["FOWT_ARISE", "ABLATION_N1", "ABLATION_N2", "ABLATION_N3", "ABLATION_N4"]

OUTPUT_ROOT_ = Path(OUTPUT_ROOT)
COMMON_DIR   = OUTPUT_ROOT_ / "common"
COMPARE_DIR  = OUTPUT_ROOT_ / "comparison"


def experiment_dirs(name: str) -> dict:
    """Canonical directory layout for one experiment; created on first request."""
    base = OUTPUT_ROOT_ / name
    d = {
        "base": base,
        "checkpoints": base / "checkpoints",
        "plots": base / "plots",
        "shap": base / "shap",
    }
    for p in d.values():
        p.mkdir(parents=True, exist_ok=True)
    return d


for _p in (COMMON_DIR, COMPARE_DIR, COMPARE_DIR / "comparison_plots"):
    _p.mkdir(parents=True, exist_ok=True)
EXP_DIRS = {name: experiment_dirs(name) for name in EXPERIMENTS}

ENVIRONMENT_INFO = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cuda": torch.version.cuda if torch.cuda.is_available() else None,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "device": str(DEVICE),
    "shap_available": _SHAP_AVAILABLE,
    "openpyxl_available": _OPENPYXL_AVAILABLE,
    "seed": SEED,
    "dry_run": DRY_RUN,
}
with open(COMMON_DIR / "environment.json", "w") as f:
    json.dump(ENVIRONMENT_INFO, f, indent=2)

print(f"Output root: {OUTPUT_ROOT_}")
for _n in EXPERIMENTS:
    print(f"  {_n:12s} -> {EXP_DIRS[_n]['base']}")
print(f"  {'comparison':12s} -> {COMPARE_DIR}")
print(f"  {'common':12s} -> {COMMON_DIR}")
print(f"\nSaved {COMMON_DIR / 'environment.json'}")

## 04. Dataset Loading

Both datasets are discovered by glob rather than by assuming a fixed set of tower names, so the
notebook adapts to whatever tower variants are actually present. It does insist that at least one
of each file type exists and that every tower appearing in the transitions also has an action-sweep
file — counterfactual evaluation is impossible otherwise, and silently dropping a tower would
quietly change what the reported metrics mean.

The sweep is ~1.45 M rows. It is read exactly once, and from Section 27 onward it is accessed
through a **pre-indexed, per-condition cache** rather than by re-filtering the dataframe, so no cell
ever copies the full sweep.

In [ ]:
DATASET_PATH_      = Path(DATASET_PATH)
ACTION_SWEEP_PATH_ = Path(ACTION_SWEEP_PATH)

for _label, _p in (("DATASET_PATH", DATASET_PATH_), ("ACTION_SWEEP_PATH", ACTION_SWEEP_PATH_)):
    if not _p.exists():
        raise RuntimeError(
            f"[FOWT-ARISE] {_label} does not exist: {_p}\n"
            f"Set {_label} in the configuration cell (Section 02)."
        )

transition_files = sorted(DATASET_PATH_.glob("transitions_*.parquet"))
sweep_files      = sorted(ACTION_SWEEP_PATH_.glob("action_sweep_*.parquet"))
if not transition_files:
    raise RuntimeError(f"[FOWT-ARISE] no 'transitions_*.parquet' under {DATASET_PATH_}")
if not sweep_files:
    raise RuntimeError(f"[FOWT-ARISE] no 'action_sweep_*.parquet' under {ACTION_SWEEP_PATH_}")

print(f"transitions files ({len(transition_files)}):")
for _p in transition_files:
    print(f"    {_p.name}")
print(f"action-sweep files ({len(sweep_files)}):")
for _p in sweep_files:
    print(f"    {_p.name}")


def _read_with_tower(path: Path) -> pd.DataFrame:
    """Read one parquet and require an explicit `tower` column.

    The tower name is guessable from the filename, but we refuse to guess: `tower` participates in
    the trajectory key and in sweep matching, so inventing it would silently corrupt both.
    """
    frame = pd.read_parquet(path)
    if "tower" not in frame.columns:
        raise RuntimeError(
            f"[FOWT-ARISE] {path} has no 'tower' column. It is required: `tower` is part of the "
            f"trajectory key (Section 07) and of counterfactual matching (Section 27)."
        )
    return frame


transitions   = pd.concat([_read_with_tower(p) for p in transition_files], ignore_index=True)
action_sweep  = pd.concat([_read_with_tower(p) for p in sweep_files], ignore_index=True)

towers_trans = set(transitions["tower"].unique())
towers_sweep = set(action_sweep["tower"].unique())
_missing = towers_trans - towers_sweep
if _missing:
    raise RuntimeError(
        f"[FOWT-ARISE] tower(s) {sorted(_missing)} appear in transitions but have no action-sweep "
        f"file. Counterfactual evaluation requires a sweep for every tower present."
    )

print(f"\ntransitions  : {transitions.shape[0]:,} rows x {transitions.shape[1]} cols")
print(f"action sweep : {action_sweep.shape[0]:,} rows x {action_sweep.shape[1]} cols")
print(f"towers       : {sorted(towers_trans)}")

## 05. Schema Validation

Column groups are resolved by **exact name matching against ordered candidate lists**, never by
substring or prefix heuristics (which is how `step` and `step_fraction`, or `damage_ratio` and
`damage_ratio_max`, get silently confused).

The resolver deliberately accepts **more than one naming family per semantic role**, because the
two datasets in this project use different names for the same physics:

| role | transitions family | action-sweep family |
|---|---|---|
| governing-section damage, no control | `damage_baseline` | `damage_max` |
| load-path, base / top section | `damage_baseline_base_section` / `..._top_section` | `damage_base_section` / `damage_top_section` |
| controllable load share | `controllable_share_mean`, `controllable_share_max_section` | `controllable_share_max` |
| wind / wave | `true_*` and `meas_*` triples | `mean_wind_speed`, `std_wind_speed`, `wave_hs`, `wave_tp` |
| yaw action | `action_yaw_setpoint_deg` | `action_yaw_error_deg` |

Whichever name is actually present is used, and **which one was chosen is recorded** in
`common/schema_mapping.json`. A required role with no resolvable candidate raises immediately with
the candidates it looked for and a sample of what the file actually contains — it is never quietly
dropped, and no column is ever invented.

### Action-outcome leakage classification

This section also splits the structural columns into two classes, which matters more than it may
appear:

* **condition-only** — determined by the operating condition alone (`damage_baseline*`,
  `controllable_share_*`, `damage_weight`, aerodynamic gains). Safe as policy input.
* **action-outcome** — a *consequence of the action that was taken* (`damage_controlled*`,
  `damage_ratio`, `del_ratio`, `ct_ratio`, `cp_ratio`). Feeding these to the policy would be
  action-outcome leakage: the observation would already encode how well the chosen action worked.
  They are used **only** in reward reconstruction and evaluation.

In [ ]:
# ---------------------------------------------------------------------------
# Semantic role -> ordered candidate names. First present candidate wins.
# `required=True` roles raise if nothing resolves; `required=False` roles are optional enrichments.
# ---------------------------------------------------------------------------
TRANSITION_ROLES = {
    # --- bookkeeping ---
    "episode_id":      dict(cands=["episode_id"],      required=True),
    "step":            dict(cands=["step"],            required=True),
    "done":            dict(cands=["done"],            required=True),
    "tower":           dict(cands=["tower"],           required=True),
    "reward":          dict(cands=["reward"],          required=True),
    "behaviour":       dict(cands=["behaviour_policy", "behavior_policy"], required=False),
    # --- operating-condition identity (used for sweep matching + best-action grouping) ---
    "sim_id":          dict(cands=["sim_id"],          required=True),
    "wind_speed_id":   dict(cands=["wind_speed_id"],   required=True),
    "wave_hs_id":      dict(cands=["wave_hs_id"],      required=True),
    "wave_tp_id":      dict(cands=["wave_tp_id"],      required=True),
    "wind_seed_id":    dict(cands=["wind_seed_id"],    required=False),
    # --- actions (transition side) ---
    "act_pitch":       dict(cands=["action_pitch_offset_deg"],                          required=True),
    "act_yaw":         dict(cands=["action_yaw_setpoint_deg", "action_yaw_error_deg"],   required=True),
    "act_ipc":         dict(cands=["action_ipc_level"],                                 required=True),
    # --- reward decomposition (dataset's own terms; never recomputed from scratch) ---
    "rw_fatigue":      dict(cands=["reward_fatigue_relief"],      required=True),
    "rw_severity":     dict(cands=["reward_severity"],            required=True),
    "rw_power":        dict(cands=["reward_power_loss_fraction"], required=True),
    "rw_duty":         dict(cands=["reward_duty_total"],          required=True),
    # --- N1 structural, CONDITION-ONLY (safe as policy input) ---
    "gov_damage_base_free":  dict(cands=["damage_baseline", "damage_max"],                             required=True),
    "lp_base_section":       dict(cands=["damage_baseline_base_section", "damage_base_section"],       required=True),
    "lp_top_section":        dict(cands=["damage_baseline_top_section", "damage_top_section"],         required=True),
    "ctrl_share_mean":       dict(cands=["controllable_share_mean", "controllable_share_max"],         required=True),
    "ctrl_share_gov":        dict(cands=["controllable_share_max_section", "controllable_share_max"],  required=True),
    "damage_weight":         dict(cands=["damage_weight"],                                             required=True),
    "gain_thrust":           dict(cands=["gain_thrust_turbulence"],                                    required=False),
    "gain_cyclic":           dict(cands=["gain_rotor_cyclic"],                                         required=False),
    "power_baseline":        dict(cands=["power_baseline_w"],                                           required=True),
    "thrust_baseline":       dict(cands=["thrust_baseline_n"],                                          required=False),
    # --- ACTION-OUTCOME (evaluation only; never policy input) ---
    "damage_controlled":     dict(cands=["damage_controlled"],   required=False),
    "damage_ratio":          dict(cands=["damage_ratio", "damage_ratio_max"], required=True),
    "del_ratio":             dict(cands=["del_ratio"],            required=True),
    "ct_ratio":              dict(cands=["ct_ratio"],             required=False),
    "cp_ratio":              dict(cands=["cp_ratio"],             required=False),
}

SWEEP_ROLES = {
    "tower":            dict(cands=["tower"],                    required=True),
    "sim_id":           dict(cands=["sim_id"],                    required=True),
    "wind_speed_id":    dict(cands=["wind_speed_id"],             required=True),
    "wave_hs_id":       dict(cands=["wave_hs_id"],                required=True),
    "wave_tp_id":       dict(cands=["wave_tp_id"],                required=True),
    "wind_seed_id":     dict(cands=["wind_seed_id"],              required=False),
    "act_pitch":        dict(cands=["action_pitch_offset_deg"],   required=True),
    "act_yaw":          dict(cands=["action_yaw_error_deg", "action_yaw_setpoint_deg"], required=True),
    "act_ipc":          dict(cands=["action_ipc_level"],          required=True),
    "damage_weight":    dict(cands=["damage_weight"],             required=True),
    "damage_max":       dict(cands=["damage_max"],                required=True),
    "damage_ratio_max": dict(cands=["damage_ratio_max"],          required=True),
    "ctrl_share_max":   dict(cands=["controllable_share_max"],    required=True),
    "power_w":          dict(cands=["power_w"],                   required=True),
    "power_baseline":   dict(cands=["power_baseline_w"],          required=True),
    "thrust_n":         dict(cands=["thrust_n"],                  required=False),
    "thrust_baseline":  dict(cands=["thrust_baseline_n"],         required=False),
    "ct_ratio":         dict(cands=["ct_ratio"],                  required=False),
    "cp_ratio":         dict(cands=["cp_ratio"],                  required=False),
}

# Measured IoT channels + their validity flags. These are the ONLY columns the N4 degradation engine
# is permitted to touch, and the only environmental channels the policy observes: a real controller
# never sees `true_*`.
IOT_MEASURED_CANDIDATES = [
    "meas_wind_speed", "meas_turbulence_std", "meas_wind_direction", "meas_wave_hs",
    "meas_wave_tp", "meas_thrust", "meas_power", "meas_tower_damage_rate",
]
IOT_VALIDITY_CANDIDATES = [
    "valid_wind_speed", "valid_turbulence_std", "valid_wind_direction", "valid_wave_hs",
    "valid_wave_tp", "valid_thrust", "valid_power", "valid_tower_damage_rate",
]
IOT_HEALTH_CANDIDATES = ["sensor_health"]
# Fallback environmental family, used only if no `meas_*` family exists at all.
ENV_FALLBACK_CANDIDATES = [
    "mean_wind_speed", "std_wind_speed", "wave_hs", "wave_tp",
    "true_wind_speed", "true_turbulence_std", "true_wave_hs", "true_wave_tp",
]
PROPRIOCEPTIVE_CANDIDATES = [
    "prev_pitch_offset_deg", "prev_yaw_setpoint_deg", "prev_ipc_level",
    "nacelle_yaw_deg", "yaw_error_deg", "cumulative_damage_fraction", "step_fraction",
]

# Structural columns that are a CONSEQUENCE of the action taken. Never a policy input.
ACTION_OUTCOME_COLUMNS = [
    "damage_controlled", "damage_controlled_base_section", "damage_controlled_top_section",
    "damage_controlled_max_section_id", "damage_ratio", "damage_ratio_max", "del_ratio",
    "ct_ratio", "cp_ratio", "reward", "reward_fatigue_relief", "reward_severity",
    "reward_fatigue_term", "reward_power_loss_fraction", "reward_duty_total",
]


def resolve_roles(role_spec: dict, columns, frame_label: str) -> dict:
    """Resolve every semantic role to a real column name, or raise with a precise diagnostic."""
    colset, resolved, missing = set(columns), {}, []
    for role, spec in role_spec.items():
        hit = next((c for c in spec["cands"] if c in colset), None)
        if hit is None:
            (missing if spec["required"] else []).append(role) if spec["required"] else None
            if spec["required"]:
                continue
        resolved[role] = hit           # None for an absent optional role
    if missing:
        lines = [f"[FOWT-ARISE] {frame_label}: could not resolve required role(s)."]
        for role in missing:
            lines.append(f"    role '{role}' -- looked for any of {role_spec[role]['cands']}")
        lines.append(f"    {frame_label} actually has {len(colset)} columns, e.g. "
                     f"{sorted(colset)[:15]} ...")
        lines.append("    No column will be invented and no substitute will be guessed. Fix the "
                     "input or extend the candidate list in Section 05.")
        raise RuntimeError("\n".join(lines))
    return resolved


def present_subset(cands, columns) -> list:
    colset = set(columns)
    return [c for c in cands if c in colset]


TR = resolve_roles(TRANSITION_ROLES, transitions.columns, "transitions")
SW = resolve_roles(SWEEP_ROLES, action_sweep.columns, "action_sweep")

IOT_MEASURED_COLS = present_subset(IOT_MEASURED_CANDIDATES, transitions.columns)
IOT_VALIDITY_COLS = present_subset(IOT_VALIDITY_CANDIDATES, transitions.columns)
IOT_HEALTH_COLS   = present_subset(IOT_HEALTH_CANDIDATES,   transitions.columns)
PROPRIO_COLS      = present_subset(PROPRIOCEPTIVE_CANDIDATES, transitions.columns)
if not IOT_MEASURED_COLS:
    IOT_MEASURED_COLS = present_subset(ENV_FALLBACK_CANDIDATES, transitions.columns)
    warnings.warn(
        "[FOWT-ARISE] no `meas_*` IoT channels found; falling back to the plain environmental "
        f"family {IOT_MEASURED_COLS}. N4 degradation will be applied to these instead, and the "
        "clean/degraded distinction therefore compares against ground truth rather than against "
        "the dataset's own sensor model. This is reported, not hidden."
    )
if not IOT_MEASURED_COLS:
    raise RuntimeError(
        "[FOWT-ARISE] no environmental observation channels could be resolved at all. Looked for "
        f"{IOT_MEASURED_CANDIDATES} then {ENV_FALLBACK_CANDIDATES}."
    )

ACTION_COLS      = [TR["act_pitch"], TR["act_yaw"], TR["act_ipc"]]
ACTION_DIM       = len(ACTION_COLS)
SWEEP_ACTION_COLS = [SW["act_pitch"], SW["act_yaw"], SW["act_ipc"]]
if ACTION_DIM != 3:
    raise RuntimeError(f"[FOWT-ARISE] expected exactly 3 actuator columns, resolved {ACTION_COLS}")

# Condition identifiers usable for counterfactual matching: must exist in BOTH frames.
CONDITION_COLS = [
    TR[r] for r in ("sim_id", "wind_speed_id", "wave_hs_id", "wave_tp_id", "wind_seed_id")
    if TR.get(r) is not None and SW.get(r) is not None
]
# `sim_id` alone already identifies the physical condition in this dataset; the rest are carried for
# reporting and for the best-action grouping.
MATCH_KEY = TR["sim_id"]

print("Resolved TRANSITION roles (role -> actual column):")
for _r, _c in TR.items():
    print(f"    {_r:22s} -> {_c}")
print("\nResolved SWEEP roles:")
for _r, _c in SW.items():
    print(f"    {_r:22s} -> {_c}")
print(f"\nIoT measured channels ({len(IOT_MEASURED_COLS)}): {IOT_MEASURED_COLS}")
print(f"IoT validity flags   ({len(IOT_VALIDITY_COLS)}): {IOT_VALIDITY_COLS}")
print(f"IoT health           ({len(IOT_HEALTH_COLS)}): {IOT_HEALTH_COLS}")
print(f"Proprioceptive       ({len(PROPRIO_COLS)}): {PROPRIO_COLS}")
print(f"\nACTION_COLS       (transitions) = {ACTION_COLS}")
print(f"SWEEP_ACTION_COLS (sweep)       = {SWEEP_ACTION_COLS}")
print("Positional action mapping transitions -> sweep:")
for _a, _b in zip(ACTION_COLS, SWEEP_ACTION_COLS):
    print(f"    {_a:28s} -> {_b}")
print(f"\nCONDITION_COLS = {CONDITION_COLS}")
print(f"MATCH_KEY      = {MATCH_KEY}")

In [ ]:
# ---- report any divergence between the documented schema and the real files ----
# Section 56 of the project brief requires discrepancies to be PRINTED, and requires that a column
# never be invented to paper over one. This block is that report.
_DOCUMENTED_TRANSITION_STRUCTURAL = [
    "damage_max", "damage_base_section", "damage_top_section", "damage_mean_section",
    "damage_max_section_id", "damage_ratio_max", "controllable_share_max",
]
_DOCUMENTED_TRANSITION_ENV = ["mean_wind_speed", "std_wind_speed", "wave_hs", "wave_tp"]

_tcols = set(transitions.columns)
_scols = set(action_sweep.columns)
_elsewhere = [c for c in _DOCUMENTED_TRANSITION_STRUCTURAL + _DOCUMENTED_TRANSITION_ENV
              if c not in _tcols and c in _scols]

print("=" * 79)
print("SCHEMA DISCREPANCY REPORT")
print("=" * 79)
if _elsewhere:
    print("The project brief documents these as TRANSITION columns, but they exist only in the")
    print("ACTION-SWEEP file. They have been resolved to their real transition-side equivalents")
    print("via the candidate lists above; nothing was invented:")
    for _c in _elsewhere:
        _role = next((r for r, s in TRANSITION_ROLES.items() if _c in s["cands"]), None)
        _actual = TR.get(_role) if _role else None
        print(f"    {_c:32s} sweep-only -> transition equivalent: {_actual}")
else:
    print("No discrepancy: every documented column name was found where the brief said it would be.")

print(f"\nRow/column counts (actual): transitions {transitions.shape}, sweep {action_sweep.shape}")
_per_tower_eps = transitions.groupby(TR['tower'])[TR['episode_id']].nunique().to_dict()
print(f"Unique episode_id per tower: {_per_tower_eps}")
print(f"=> {sum(_per_tower_eps.values())} distinct trajectories once keyed by (tower, episode_id).")
print("   A global episode_id count would undercount by the number of towers -- see Section 07.")

# ---- persist the resolved mapping ----
SCHEMA = {
    "transition_roles": TR,
    "sweep_roles": SW,
    "iot_measured_cols": IOT_MEASURED_COLS,
    "iot_validity_cols": IOT_VALIDITY_COLS,
    "iot_health_cols": IOT_HEALTH_COLS,
    "proprioceptive_cols": PROPRIO_COLS,
    "action_cols": ACTION_COLS,
    "sweep_action_cols": SWEEP_ACTION_COLS,
    "condition_cols": CONDITION_COLS,
    "match_key": MATCH_KEY,
    "action_outcome_columns_never_observed": sorted(
        c for c in ACTION_OUTCOME_COLUMNS if c in _tcols),
    "documented_but_sweep_only": _elsewhere,
    "transitions_shape": list(transitions.shape),
    "sweep_shape": list(action_sweep.shape),
}
with open(COMMON_DIR / "schema_mapping.json", "w") as f:
    json.dump(SCHEMA, f, indent=2)
print(f"\nSaved {COMMON_DIR / 'schema_mapping.json'}")
print("\nACTION-OUTCOME columns (evaluation only, NEVER a policy input):")
print(f"    {SCHEMA['action_outcome_columns_never_observed']}")
print("SCHEMA VALIDATION PASSED (exact-match resolution only; no substring heuristics).")

## 06. Dataset Integrity Audit

Loud, specific checks before any modelling. Anything that would silently distort a downstream metric
raises here instead of surfacing later as an inexplicable number.

* dtypes, NaN and ±Inf counts on both frames
* exact duplicate rows
* action values inside their physical envelope
* per-tower trajectory structure and step monotonicity
* the sweep's per-condition block structure (uniform action grid), which Section 27's vectorised
  matcher depends on

Missing physical quantities are never imputed with zero: a zero damage or zero power is a
*meaningful physical claim*, so a NaN that reached this point is a data problem to be surfaced, not
smoothed over.

In [ ]:
def audit_frame(frame: pd.DataFrame, label: str) -> dict:
    """NaN / Inf / duplicate audit for one dataframe. Returns a summary dict."""
    num = frame.select_dtypes(include=[np.number])
    nan_by_col = num.isna().sum()
    n_nan = int(nan_by_col.sum())
    arr = num.to_numpy(dtype=np.float64, copy=False)
    inf_mask = np.isinf(arr)
    n_inf = int(inf_mask.sum())
    n_dup = int(frame.duplicated().sum())

    print(f"--- {label} ---")
    print(f"    shape            : {frame.shape}")
    print(f"    numeric columns  : {num.shape[1]}")
    print(f"    total NaN        : {n_nan}")
    print(f"    total +/-Inf     : {n_inf}")
    print(f"    duplicate rows   : {n_dup}")
    if n_nan:
        print("    NaN by column (non-zero only):")
        for c, v in nan_by_col[nan_by_col > 0].items():
            print(f"        {c}: {int(v)}")
    if n_inf:
        bad = [num.columns[j] for j in sorted(set(np.where(inf_mask)[1]))]
        print(f"    Inf in columns   : {bad}")
    return {"shape": list(frame.shape), "n_nan": n_nan, "n_inf": n_inf, "n_duplicate_rows": n_dup}


print("=" * 79)
print("DATASET INTEGRITY AUDIT")
print("=" * 79)
audit_trans = audit_frame(transitions, "transitions")
audit_sweep = audit_frame(action_sweep, "action_sweep")

if audit_trans["n_nan"] or audit_trans["n_inf"] or audit_sweep["n_nan"] or audit_sweep["n_inf"]:
    raise RuntimeError(
        "[FOWT-ARISE] NaN and/or Inf present in the raw data (details above). Refusing to continue: "
        "imputing a missing damage or power value with 0 would assert a physical fact that the data "
        "does not support, and every downstream fatigue/power metric would inherit that fiction."
    )
if audit_trans["n_duplicate_rows"]:
    warnings.warn(f"[FOWT-ARISE] transitions contains {audit_trans['n_duplicate_rows']} exact "
                  f"duplicate rows; they are retained (dropping them would change episode lengths) "
                  f"but flagged here.")

# ---- action envelope ----
print("\n--- action envelope (transitions) ---")
_documented_envelope = {  # plausibility reference only; authoritative limits come from TRAIN split
    TR["act_pitch"]: (0.0, 8.0),
    TR["act_yaw"]:   (-30.0, 30.0),
    TR["act_ipc"]:   (0.0, 1.0),
}
for _c in ACTION_COLS:
    _lo, _hi = float(transitions[_c].min()), float(transitions[_c].max())
    _ref = _documented_envelope.get(_c)
    _note = ""
    if _ref is not None and not (_ref[0] - 0.5 <= _lo and _hi <= _ref[1] + 0.5):
        _note = f"   <-- OUTSIDE documented envelope {_ref}"
    print(f"    {_c:28s} [{_lo:+9.4f}, {_hi:+9.4f}]{_note}")
    if _note:
        warnings.warn(f"[FOWT-ARISE] '{_c}' exceeds its documented physical envelope {_ref}. "
                      f"Train-derived limits (Section 11) remain authoritative, but review this.")

# ---- sweep block structure (Section 27 depends on it) ----
print("\n--- action-sweep block structure ---")
_blocks = action_sweep.groupby([SW["tower"], SW["sim_id"]]).size()
_uniform = _blocks.nunique() == 1
print(f"    conditions (tower, sim_id) : {len(_blocks):,}")
print(f"    actions per condition      : {sorted(_blocks.unique())[:5]}"
      f"{' (uniform)' if _uniform else ' (NON-UNIFORM)'}")
if not _uniform:
    raise RuntimeError(
        "[FOWT-ARISE] the action sweep does not have a uniform number of actions per condition "
        f"(counts seen: {sorted(_blocks.unique())[:10]}). Section 27's vectorised nearest-action "
        "matcher reshapes each condition into a fixed-size block and would mis-align otherwise."
    )
N_SWEEP_ACTIONS = int(_blocks.iloc[0])

DATA_SUMMARY = {
    "transitions": audit_trans,
    "action_sweep": audit_sweep,
    "towers": sorted(towers_trans),
    "n_sweep_conditions": int(len(_blocks)),
    "n_sweep_actions_per_condition": N_SWEEP_ACTIONS,
    "unique_episode_id_per_tower": {str(k): int(v) for k, v in _per_tower_eps.items()},
}
with open(COMMON_DIR / "data_summary.json", "w") as f:
    json.dump(DATA_SUMMARY, f, indent=2)
print(f"\nSaved {COMMON_DIR / 'data_summary.json'}")
print("INTEGRITY AUDIT PASSED.")

## 07. Episode / Trajectory Construction

**The trap this section exists to avoid.** `episode_id` restarts from the same range for every
tower, so three physically distinct trajectories share each `episode_id` value. A global assertion
like `transitions.groupby("episode_id")["done"].sum() == 1` therefore *fails on correct data*,
because it sees three terminal rows where there are three separate trajectories each with one.
Worse, splitting on `episode_id` alone would place the *same* metocean trajectory into train and
test under different towers — genuine leakage that would silently inflate every reported metric.

The trajectory key is therefore `(tower, episode_id)`, materialised as `traj_id`. Validation is done
**per trajectory**:

* exactly one terminal (`done == 1`) row, and it is the last row in step order
* `step` values contiguous from their minimum, strictly increasing
* consistent trajectory length distribution, reported rather than assumed

In [ ]:
_ep, _tw, _st, _dn = TR["episode_id"], TR["tower"], TR["step"], TR["done"]

transitions["traj_id"] = (
    transitions[_tw].astype(str) + "__" + transitions[_ep].astype(str)
)
transitions = transitions.sort_values([_tw, _ep, _st], kind="stable").reset_index(drop=True)

n_traj        = transitions["traj_id"].nunique()
n_episode_ids = transitions[_ep].nunique()
n_towers      = transitions[_tw].nunique()

print("=" * 79)
print("TRAJECTORY STRUCTURE")
print("=" * 79)
print(f"    towers                       : {n_towers}")
print(f"    distinct episode_id values   : {n_episode_ids}")
print(f"    distinct (tower, episode_id) : {n_traj}   <-- the true trajectory count")
print(f"    transitions                  : {len(transitions):,}")
if n_traj != n_episode_ids:
    print(f"\n    NOTE: {n_episode_ids} episode_id values x {n_towers} towers = {n_traj} trajectories.")
    print( "    Any per-episode_id assertion would treat co-numbered trajectories from different")
    print( "    towers as one episode. All splitting and validation below uses traj_id.")

# ---- per-trajectory validation ----
_g = transitions.groupby("traj_id", sort=False)
_done_sum = _g[_dn].sum()
_bad_done = _done_sum[_done_sum != 1]
if len(_bad_done):
    raise RuntimeError(
        f"[FOWT-ARISE] {len(_bad_done)} trajectory/ies do not have exactly one terminal row "
        f"(done == 1). Examples: {_bad_done.head(5).to_dict()}. Offline next-state construction "
        f"(Section 18) relies on this invariant; continuing would corrupt every bootstrap target."
    )
print("\n    [OK] every trajectory has exactly one done == 1 row")

_last_is_done = _g.apply(lambda d: bool(d[_dn].to_numpy()[-1] == 1), include_groups=False)
if not _last_is_done.all():
    _bad = _last_is_done[~_last_is_done].index.tolist()[:5]
    raise RuntimeError(
        f"[FOWT-ARISE] the terminal row is not the last row in step order for trajectory/ies {_bad}. "
        f"Rows are sorted by (tower, episode_id, step), so this indicates a genuine ordering problem."
    )
print("    [OK] the terminal row is the last row in step order for every trajectory")

def _steps_contiguous(d) -> bool:
    s = d[_st].to_numpy()
    return bool(np.all(np.diff(s) == 1))

_contig = _g.apply(_steps_contiguous, include_groups=False)
if not _contig.all():
    _bad = _contig[~_contig].index.tolist()[:5]
    warnings.warn(
        f"[FOWT-ARISE] {int((~_contig).sum())} trajectory/ies have non-contiguous `step` values "
        f"(e.g. {_bad}). Next-state construction uses the successor ROW within the trajectory, so "
        f"this is tolerated, but any gap means the successor is further ahead in time than one step."
    )
else:
    print("    [OK] step values are contiguous and strictly increasing within every trajectory")

_lens = _g.size()
print(f"\n    trajectory length: min={_lens.min()} max={_lens.max()} "
      f"mean={_lens.mean():.1f} (unique lengths: {sorted(_lens.unique())[:8]})")
print(f"    transitions per tower: {transitions[_tw].value_counts().to_dict()}")

## 08. Episode-Level Train / Validation / Test Split

The split is over **whole trajectories**, never individual transitions. Transitions inside one
trajectory share a slowly evolving metocean realisation, so a transition-level split would leak
almost-identical neighbouring rows across the boundary and every reported metric would be optimistic.

Fractions are 0.70 / 0.15 / 0.15 and the assignment is deterministic given `SEED`. Disjointness is
asserted at **both** the trajectory level and the row level, and the test set is then frozen: a
guard function re-checks on every later use that it has not been mutated.

In [ ]:
def make_trajectory_split(traj_ids, train_frac, val_frac, seed):
    """Deterministic, disjoint trajectory-level split. Returns (train, val, test) id arrays."""
    if not (0 < train_frac < 1) or not (0 < val_frac < 1):
        raise RuntimeError("[FOWT-ARISE] TRAIN_FRAC and VAL_FRAC must lie in (0, 1).")
    if train_frac + val_frac >= 1.0:
        raise RuntimeError("[FOWT-ARISE] TRAIN_FRAC + VAL_FRAC must be < 1 or TEST would be empty.")
    ids = np.sort(np.asarray(traj_ids))              # sort first => permutation is reproducible
    perm = np.random.default_rng(seed).permutation(len(ids))
    shuffled = ids[perm]
    n = len(shuffled)
    n_tr, n_va = int(round(n * train_frac)), int(round(n * val_frac))
    tr, va, te = shuffled[:n_tr], shuffled[n_tr:n_tr + n_va], shuffled[n_tr + n_va:]
    assert set(tr).isdisjoint(va) and set(tr).isdisjoint(te) and set(va).isdisjoint(te)
    assert len(tr) + len(va) + len(te) == n
    return tr, va, te


TRAIN_TRAJ, VAL_TRAJ, TEST_TRAJ = make_trajectory_split(
    transitions["traj_id"].unique(), TRAIN_FRAC, VAL_FRAC, SEED)
TRAIN_SET, VAL_SET, TEST_SET = set(TRAIN_TRAJ), set(VAL_TRAJ), set(TEST_TRAJ)

train_mask = transitions["traj_id"].isin(TRAIN_SET).to_numpy()
val_mask   = transitions["traj_id"].isin(VAL_SET).to_numpy()
test_mask  = transitions["traj_id"].isin(TEST_SET).to_numpy()

assert train_mask.sum() + val_mask.sum() + test_mask.sum() == len(transitions)
assert not np.any(train_mask & val_mask) and not np.any(train_mask & test_mask)
assert not np.any(val_mask & test_mask)

# No trajectory may straddle splits, and no (tower, episode_id) pair may appear in two splits.
_split_of = {}
for _s, _ids in (("train", TRAIN_SET), ("val", VAL_SET), ("test", TEST_SET)):
    for _i in _ids:
        _split_of[_i] = _s
_leak = [t for t, sub in transitions.groupby("traj_id", sort=False)
         if len({_split_of[t]}) != 1]
assert not _leak, f"[FOWT-ARISE] trajectory straddles splits: {_leak[:5]}"

print("=" * 79)
print("EPISODE-LEVEL SPLIT")
print("=" * 79)
print(f"    towers                : {n_towers}")
print(f"    trajectories total    : {n_traj}")
print(f"    transitions total     : {len(transitions):,}")
print(f"    train trajectories    : {len(TRAIN_SET):5d}   transitions: {int(train_mask.sum()):,}")
print(f"    val   trajectories    : {len(VAL_SET):5d}   transitions: {int(val_mask.sum()):,}")
print(f"    test  trajectories    : {len(TEST_SET):5d}   transitions: {int(test_mask.sum()):,}")
print(f"\n    fractions realised    : train={len(TRAIN_SET)/n_traj:.4f} "
      f"val={len(VAL_SET)/n_traj:.4f} test={len(TEST_SET)/n_traj:.4f}")
print("    [OK] disjoint at trajectory level and at row level; no trajectory straddles a split")

# tower balance across splits -- an accidental tower-skewed split would confound every comparison
_bal = pd.DataFrame({
    "train": transitions.loc[train_mask, _tw].value_counts(),
    "val":   transitions.loc[val_mask,   _tw].value_counts(),
    "test":  transitions.loc[test_mask,  _tw].value_counts(),
}).fillna(0).astype(int)
print("\n    transitions per tower per split:")
print(_bal.to_string().replace("\n", "\n    "))

# ---- freeze the test set ----
_FROZEN_TEST = frozenset(TEST_SET)

def assert_no_test_leakage(context: str) -> None:
    """Defensive guard: the frozen test trajectory set must never change."""
    if TEST_SET != set(_FROZEN_TEST):
        raise RuntimeError(f"[FOWT-ARISE] TEST_SET was mutated! context={context}")

assert_no_test_leakage("post-split")

SPLIT_SUMMARY = {
    "seed": SEED,
    "fractions_requested": {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": TEST_FRAC},
    "n_towers": int(n_towers),
    "n_trajectories": int(n_traj),
    "n_episode_id_values": int(n_episode_ids),
    "n_transitions": int(len(transitions)),
    "n_train_trajectories": len(TRAIN_SET),
    "n_val_trajectories": len(VAL_SET),
    "n_test_trajectories": len(TEST_SET),
    "n_train_transitions": int(train_mask.sum()),
    "n_val_transitions": int(val_mask.sum()),
    "n_test_transitions": int(test_mask.sum()),
    "trajectory_key": "tower + '__' + episode_id",
}
with open(COMMON_DIR / "split_summary.json", "w") as f:
    json.dump(SPLIT_SUMMARY, f, indent=2)
for _name, _ids in (("train", TRAIN_TRAJ), ("val", VAL_TRAJ), ("test", TEST_TRAJ)):
    pd.DataFrame({"traj_id": sorted(_ids)}).to_csv(COMMON_DIR / f"{_name}_trajectories.csv",
                                                   index=False)
print(f"\n    Saved {COMMON_DIR / 'split_summary.json'} and per-split trajectory id lists")

### 08b. Fatigue-model calibration (Wöhler exponent)

Damage and damage-equivalent load are related through the Wöhler exponent $m$:

$$\text{DEL ratio} \;=\; \left(\text{damage ratio}\right)^{1/m}
\qquad\Longleftrightarrow\qquad
\log(\text{DEL ratio}) \;=\; \tfrac{1}{m}\,\log(\text{damage ratio})$$

Rather than hard-coding $m$, it is recovered by least squares on the **training split only**, from
the dataset's own `damage_ratio` and `del_ratio` columns, and then cross-checked against the value
the dataset documents. Points with $|\log(\text{damage ratio})| \le 0.02$ are excluded from the fit
because near a ratio of 1 the quotient is dominated by floating-point noise rather than by $m$.

In [ ]:
_dr_col, _del_col = TR["damage_ratio"], TR["del_ratio"]
_fit = train_mask & (transitions[_dr_col].to_numpy(dtype=np.float64) > 1e-6)
_ldr = np.log(transitions.loc[_fit, _dr_col].to_numpy(dtype=np.float64))
_lde = np.log(np.maximum(transitions.loc[_fit, _del_col].to_numpy(dtype=np.float64), 1e-12))
_keep = np.abs(_ldr) > 0.02
if _keep.sum() < 100:
    raise RuntimeError(
        f"[FOWT-ARISE] only {int(_keep.sum())} training rows have a damage ratio far enough from 1 "
        f"to identify the Wohler exponent. Refusing to fit m from noise; set WOHLER_EXPONENT_M "
        f"explicitly in Section 02 if you know the correct value for this dataset."
    )
_slope = float(np.sum(_ldr[_keep] * _lde[_keep]) / np.sum(_ldr[_keep] ** 2))
_m_fitted = 1.0 / _slope
_resid = _lde[_keep] - _slope * _ldr[_keep]
_r2 = 1.0 - float(np.sum(_resid ** 2) / np.sum((_lde[_keep] - _lde[_keep].mean()) ** 2))

WOHLER_M = float(WOHLER_EXPONENT_M) if WOHLER_EXPONENT_M is not None else _m_fitted
print("=" * 79)
print("FATIGUE MODEL CALIBRATION")
print("=" * 79)
print(f"    fitted on {int(_keep.sum()):,} TRAIN rows (|log damage_ratio| > 0.02)")
print(f"    empirical Wohler exponent m = {_m_fitted:.6f}   (fit R^2 = {_r2:.8f})")
print(f"    configured WOHLER_EXPONENT_M = {WOHLER_EXPONENT_M}")
print(f"    -> using m = {WOHLER_M:.6f}")
if abs(_m_fitted - 3.0) > 0.05:
    warnings.warn(f"[FOWT-ARISE] fitted m={_m_fitted:.4f} differs notably from the dataset's "
                  f"documented value of 3.0. Using m={WOHLER_M:.4f}; review the fatigue model.")
else:
    print("    consistent with the dataset's documented value of 3.0")


def damage_ratio_to_del_ratio(damage_ratio) -> np.ndarray:
    """DEL ratio = damage_ratio ** (1/m). Single definition, reused everywhere."""
    return np.power(np.maximum(np.asarray(damage_ratio, dtype=np.float64), 0.0), 1.0 / WOHLER_M)


def del_ratio_to_fatigue_relief(del_ratio) -> np.ndarray:
    """Fatigue relief (fraction) = 1 - DEL ratio. Multiply by 100 for the reported percentage."""
    return 1.0 - np.asarray(del_ratio, dtype=np.float64)


_check = damage_ratio_to_del_ratio(transitions.loc[_fit, _dr_col].to_numpy())
_err = float(np.max(np.abs(_check - transitions.loc[_fit, _del_col].to_numpy(dtype=np.float64))))
print(f"    reconstruction of the dataset's own del_ratio: max abs error = {_err:.3e}")
if _err > 1e-4:
    warnings.warn(f"[FOWT-ARISE] del_ratio reconstruction error {_err:.3e} is larger than expected; "
                  f"the DEL/damage relationship may not be a pure power law on this dataset.")

## 09. State Feature Construction  &  10. Physics-Informed N1 Features

Two state vectors are built, differing **only** by the N1 block, so that Ablation N1 is a clean
single-variable removal.

**`CORE_STATE_COLS` — the conventional observation (used by Ablation N1)**

| group | contents | why |
|---|---|---|
| IoT measured | `meas_*` wind / wave / thrust / power / damage-rate channels | what the sensor network actually delivers |
| IoT validity | `valid_*` flags + `sensor_health` | the controller knows which readings it can trust |
| proprioceptive | previous pitch / yaw / IPC command, nacelle yaw, yaw error, cumulative damage fraction, step fraction | exactly known to the controller: its own last command and encoders |

**`N1_EXTRA_COLS` — the physics-informed block (N1)**

Directly available, condition-only:

* `damage_baseline` — governing-section damage with no control applied
* `damage_baseline_base_section`, `damage_baseline_top_section` — the load-path decomposition
* `controllable_share_mean`, `controllable_share_max_section` — controllable-load share
* `damage_weight` — lifetime weighting of this condition
* `gain_thrust_turbulence`, `gain_rotor_cyclic` — aerodynamic gains: how strongly thrust and cyclic
  loading respond to control, i.e. *control authority*

Derived here and documented (§13 permits unambiguous derivation):

$$
\text{base share} = \frac{D_{\text{base}}}{D_{\text{base}} + D_{\text{top}}}
\qquad
\text{governing section} = \mathbb{1}\!\left[D_{\text{base}} \ge D_{\text{top}}\right]
$$
$$
\text{governing-section lifetime DEL} = \left(D_{\text{gov}}\cdot w\right)^{1/m}
\qquad
\text{controllable damage} = D_{\text{gov}} \cdot s_{\text{gov}}
$$

Every one of these depends on the **operating condition only**, never on the action taken. The
action-outcome quantities (`damage_ratio`, `del_ratio`, `damage_controlled*`, `cp_ratio`,
`ct_ratio`) are excluded from both state vectors by an explicit assertion — including them would
tell the policy how well its own action worked before it chose it.

Every final feature is recorded in `state_feature_manifest.json` with its source column, derivation
and normalisation.

In [ ]:
# ---------------------------------------------------------------------------
# Derived N1 features. Prefix `n1_` so provenance is unmistakable downstream.
# ---------------------------------------------------------------------------
_d_gov  = transitions[TR["gov_damage_base_free"]].to_numpy(dtype=np.float64)
_d_base = transitions[TR["lp_base_section"]].to_numpy(dtype=np.float64)
_d_top  = transitions[TR["lp_top_section"]].to_numpy(dtype=np.float64)
_w_dmg  = transitions[TR["damage_weight"]].to_numpy(dtype=np.float64)
_s_gov  = transitions[TR["ctrl_share_gov"]].to_numpy(dtype=np.float64)

_lp_total = _d_base + _d_top
transitions["n1_load_path_base_share"] = np.where(_lp_total > 0.0, _d_base / np.maximum(_lp_total, 1e-30), 0.5)
transitions["n1_governing_is_base"]    = (_d_base >= _d_top).astype(np.float64)
transitions["n1_gov_lifetime_del"]     = np.power(np.maximum(_d_gov * _w_dmg, 0.0), 1.0 / WOHLER_M)
transitions["n1_controllable_damage"]  = _d_gov * np.clip(_s_gov, 0.0, 1.0)

DERIVED_N1 = {
    "n1_load_path_base_share": dict(
        derivation="damage_baseline_base_section / (base + top); 0.5 where the total is 0",
        sources=[TR["lp_base_section"], TR["lp_top_section"]],
        meaning="load-path decomposition expressed as the base section's share of tower damage"),
    "n1_governing_is_base": dict(
        derivation="1 if damage_baseline_base_section >= damage_baseline_top_section else 0",
        sources=[TR["lp_base_section"], TR["lp_top_section"]],
        meaning="identity of the governing (worst) section"),
    "n1_gov_lifetime_del": dict(
        derivation=f"(damage_baseline * damage_weight) ** (1/m), m={WOHLER_M:.6f}",
        sources=[TR["gov_damage_base_free"], TR["damage_weight"]],
        meaning="governing-section fatigue in lifetime damage-equivalent-load units"),
    "n1_controllable_damage": dict(
        derivation="damage_baseline * clip(controllable_share_max_section, 0, 1)",
        sources=[TR["gov_damage_base_free"], TR["ctrl_share_gov"]],
        meaning="absolute magnitude of governing-section damage that rotor control can influence"),
}

# ---------------------------------------------------------------------------
# Assemble the two state vectors
# ---------------------------------------------------------------------------
CORE_STATE_COLS = list(IOT_MEASURED_COLS) + list(IOT_VALIDITY_COLS) + list(IOT_HEALTH_COLS) \
                  + list(PROPRIO_COLS)

_n1_direct = [TR["gov_damage_base_free"], TR["lp_base_section"], TR["lp_top_section"],
              TR["ctrl_share_mean"], TR["ctrl_share_gov"], TR["damage_weight"]]
for _r in ("gain_thrust", "gain_cyclic"):
    if TR.get(_r) is not None:
        _n1_direct.append(TR[_r])
# de-duplicate while preserving order (e.g. if two roles resolved to the same column)
_seen = set()
N1_DIRECT_COLS = [c for c in _n1_direct if not (c in _seen or _seen.add(c))]
N1_DERIVED_COLS = list(DERIVED_N1.keys())
N1_EXTRA_COLS = N1_DIRECT_COLS + N1_DERIVED_COLS

FULL_STATE_COLS = CORE_STATE_COLS + N1_EXTRA_COLS
CORE_STATE_DIM, FULL_STATE_DIM = len(CORE_STATE_COLS), len(FULL_STATE_COLS)

# ---- leakage assertion: no action-outcome column may appear in ANY state vector ----
_leaky = sorted(set(FULL_STATE_COLS) & set(ACTION_OUTCOME_COLUMNS))
if _leaky:
    raise RuntimeError(
        f"[FOWT-ARISE] action-outcome column(s) {_leaky} reached the state vector. These are "
        f"consequences of the action that was taken, so observing them would tell the policy how "
        f"well its own action worked before it chose it. Remove them from the N1 block."
    )
_dupes = [c for c in set(FULL_STATE_COLS) if FULL_STATE_COLS.count(c) > 1]
if _dupes:
    raise RuntimeError(f"[FOWT-ARISE] duplicated state feature(s): {sorted(_dupes)}")
_absent = [c for c in FULL_STATE_COLS if c not in transitions.columns]
if _absent:
    raise RuntimeError(f"[FOWT-ARISE] state feature(s) not present in the dataframe: {_absent}")

# Group boundaries, computed from the lists rather than hard-coded indices, so the encoder in
# Section 15 can never slice the wrong block if a group changes size.
G_IOT   = len(IOT_MEASURED_COLS)
G_VALID = len(IOT_VALIDITY_COLS) + len(IOT_HEALTH_COLS)
G_PROP  = len(PROPRIO_COLS)
G_N1    = len(N1_EXTRA_COLS)
assert G_IOT + G_VALID + G_PROP == CORE_STATE_DIM
assert G_IOT + G_VALID + G_PROP + G_N1 == FULL_STATE_DIM
# The N4 degradation engine may only touch the measured + validity block.
IOT_DEGRADABLE_SLICE = slice(0, G_IOT + G_VALID)
assert FULL_STATE_COLS[IOT_DEGRADABLE_SLICE] == \
    list(IOT_MEASURED_COLS) + list(IOT_VALIDITY_COLS) + list(IOT_HEALTH_COLS)

print("=" * 79)
print("STATE CONSTRUCTION")
print("=" * 79)
print(f"    CORE_STATE_DIM (Ablation N1 uses this) = {CORE_STATE_DIM}")
print(f"      IoT measured   ({G_IOT:2d}): {IOT_MEASURED_COLS}")
print(f"      IoT validity   ({G_VALID:2d}): {list(IOT_VALIDITY_COLS) + list(IOT_HEALTH_COLS)}")
print(f"      proprioceptive ({G_PROP:2d}): {PROPRIO_COLS}")
print(f"\n    N1 physics block ({G_N1}):")
print(f"      direct  : {N1_DIRECT_COLS}")
print(f"      derived : {N1_DERIVED_COLS}")
print(f"\n    FULL_STATE_DIM (FOWT-ARISE, N2/N3/N4) = {FULL_STATE_DIM}")
print(f"    IoT-degradable slice = columns [0:{G_IOT + G_VALID}]")
print(f"\n    [OK] no action-outcome column reached the state (checked against "
      f"{len(ACTION_OUTCOME_COLUMNS)} known outcome columns)")

## 11. Action Space Construction

The action is **joint and three-dimensional**: `[pitch_offset_deg, yaw_setpoint_deg, ipc_level]`.
There is one policy emitting all three; there is no configuration in this notebook that trains three
independent agents.

Bounds are taken from the **training split only** — never from a hard-coded physical constant, and
never from the full dataset (which would let test-set extremes influence the normalisation the policy
is trained under). The dataset's documented actuator envelope is used purely as a plausibility
cross-check and a warning is raised on divergence; the train-derived numbers stay authoritative.

The neutral / "do nothing" action is $[0, 0, 0]$: no pitch offset, no yaw offset relative to tracking
the inflow, no IPC activation. **No-Action Rate** is the fraction of steps where all three normalised
components lie within `NO_ACTION_TOLERANCE` of neutral.

In [ ]:
_train_actions = transitions.loc[train_mask, ACTION_COLS]
ACTION_LO    = _train_actions.min(axis=0).to_numpy(dtype=np.float64)
ACTION_HI    = _train_actions.max(axis=0).to_numpy(dtype=np.float64)
ACTION_RANGE = np.maximum(ACTION_HI - ACTION_LO, 1e-9)
ACTION_NEUTRAL = np.zeros(ACTION_DIM, dtype=np.float64)

for _i, _c in enumerate(ACTION_COLS):
    if not (ACTION_LO[_i] - 1e-6 <= ACTION_NEUTRAL[_i] <= ACTION_HI[_i] + 1e-6):
        raise RuntimeError(
            f"[FOWT-ARISE] the neutral action 0.0 for '{_c}' lies outside the train-derived range "
            f"[{ACTION_LO[_i]}, {ACTION_HI[_i]}], so 'do nothing' is not representable. The "
            f"No-Action Rate metric would be meaningless."
        )


def action_to_norm(a) -> np.ndarray:
    """Physical action units -> [-1, 1], using TRAIN-derived bounds."""
    return 2.0 * (np.asarray(a, dtype=np.float64) - ACTION_LO) / ACTION_RANGE - 1.0


def norm_to_action(a) -> np.ndarray:
    """[-1, 1] -> physical units, then clip defensively into the valid envelope."""
    phys = (np.asarray(a, dtype=np.float64) + 1.0) / 2.0 * ACTION_RANGE + ACTION_LO
    return np.clip(phys, ACTION_LO, ACTION_HI)


ACTION_NEUTRAL_NORM = action_to_norm(ACTION_NEUTRAL)


def no_action_indicator(a_norm) -> np.ndarray:
    """True where ALL normalised action dims are within NO_ACTION_TOLERANCE of neutral."""
    dev = np.abs(np.asarray(a_norm, dtype=np.float64) - ACTION_NEUTRAL_NORM[None, :])
    return np.all(dev <= NO_ACTION_TOLERANCE, axis=-1)


print("=" * 79)
print("ACTION SPACE (bounds derived from the TRAIN split only)")
print("=" * 79)
for _i, _c in enumerate(ACTION_COLS):
    print(f"    {_c:28s} lo={ACTION_LO[_i]:+9.4f}  hi={ACTION_HI[_i]:+9.4f}  "
          f"range={ACTION_RANGE[_i]:8.4f}  neutral={ACTION_NEUTRAL[_i]:+.2f}")
print(f"\n    neutral in normalised space : {ACTION_NEUTRAL_NORM}")
print(f"    ACTION_DIM                  : {ACTION_DIM}  (one joint policy, never 3 agents)")
print(f"    NO_ACTION_TOLERANCE         : {NO_ACTION_TOLERANCE} (normalised, all dims)")

_rt = action_to_norm(_train_actions.to_numpy(dtype=np.float64))
assert _rt.min() >= -1 - 1e-9 and _rt.max() <= 1 + 1e-9, "train actions must normalise into [-1,1]"
_rt_back = norm_to_action(_rt)
_roundtrip = float(np.max(np.abs(_rt_back - _train_actions.to_numpy(dtype=np.float64))))
print(f"    normalise/denormalise round-trip max abs error: {_roundtrip:.3e}")
assert _roundtrip < 1e-6

ACTION_SPACE_INFO = {
    "action_cols": ACTION_COLS,
    "action_lo": ACTION_LO.tolist(),
    "action_hi": ACTION_HI.tolist(),
    "action_range": ACTION_RANGE.tolist(),
    "action_neutral": ACTION_NEUTRAL.tolist(),
    "derived_from": "TRAIN split only",
    "no_action_tolerance": NO_ACTION_TOLERANCE,
}

## 12. Feature Normalisation

A standard-score scaler is fitted on the **training split only** and reused unchanged for validation,
test, and every ablation. Fitting on all rows would let test statistics leak into the input
transformation the policy was trained under.

Two details that matter on this data:

* validity flags are near-binary, so a channel can have (near-)zero variance; those get a unit scale
  rather than dividing by ~0 and manufacturing enormous inputs
* scaling is applied **after** N4 degradation, in the same order at train and eval time, so degraded
  observations are not accidentally re-centred by statistics computed on clean data

The scaler is serialised into every checkpoint so a resumed or reloaded model normalises inputs
exactly as it did during training.

In [ ]:
@dataclass
class FeatureScaler:
    """Standard-score scaler fitted on TRAIN only. Serialisable into checkpoints."""
    mean: np.ndarray
    std: np.ndarray
    columns: list

    @classmethod
    def fit(cls, values: np.ndarray, columns: list) -> "FeatureScaler":
        mean = values.mean(axis=0).astype(np.float64)
        std = values.std(axis=0).astype(np.float64)
        n_flat = int(np.sum(std < 1e-8))
        std = np.where(std < 1e-8, 1.0, std)          # constant channel -> unit scale, not 1/0
        if n_flat:
            print(f"    note: {n_flat} state channel(s) are constant on TRAIN; unit scale applied "
                  f"instead of dividing by ~0")
        return cls(mean=mean, std=std, columns=list(columns))

    def transform(self, values: np.ndarray) -> np.ndarray:
        return (np.asarray(values, dtype=np.float64) - self.mean) / self.std

    def inverse_transform(self, values: np.ndarray) -> np.ndarray:
        return np.asarray(values, dtype=np.float64) * self.std + self.mean

    def to_dict(self) -> dict:
        return {"mean": self.mean.tolist(), "std": self.std.tolist(), "columns": list(self.columns)}

    @classmethod
    def from_dict(cls, d: dict) -> "FeatureScaler":
        return cls(mean=np.asarray(d["mean"], dtype=np.float64),
                   std=np.asarray(d["std"], dtype=np.float64),
                   columns=list(d["columns"]))


print("=" * 79)
print("FEATURE NORMALISATION (fitted on TRAIN only)")
print("=" * 79)
_train_state_full = transitions.loc[train_mask, FULL_STATE_COLS].to_numpy(dtype=np.float64)
SCALER_FULL = FeatureScaler.fit(_train_state_full, FULL_STATE_COLS)
_train_state_core = transitions.loc[train_mask, CORE_STATE_COLS].to_numpy(dtype=np.float64)
SCALER_CORE = FeatureScaler.fit(_train_state_core, CORE_STATE_COLS)
print(f"    SCALER_FULL: {len(SCALER_FULL.columns)} features (FOWT-ARISE, N2/N3/N4)")
print(f"    SCALER_CORE: {len(SCALER_CORE.columns)} features (Ablation N1)")

_z = SCALER_FULL.transform(_train_state_full)
print(f"    TRAIN after scaling: mean|.|={np.abs(_z.mean(axis=0)).max():.2e} "
      f"std in [{_z.std(axis=0).min():.3f}, {_z.std(axis=0).max():.3f}]")
assert np.isfinite(_z).all(), "scaled TRAIN state contains non-finite values"

# ---- state feature manifest: source, derivation, normalisation, per feature ----
def build_state_manifest() -> dict:
    def _group_of(col):
        if col in IOT_MEASURED_COLS:                     return "iot_measured"
        if col in IOT_VALIDITY_COLS or col in IOT_HEALTH_COLS: return "iot_validity"
        if col in PROPRIO_COLS:                          return "proprioceptive"
        if col in N1_DERIVED_COLS:                       return "n1_physics_derived"
        if col in N1_DIRECT_COLS:                        return "n1_physics_direct"
        return "unknown"

    feats = []
    for i, col in enumerate(FULL_STATE_COLS):
        grp = _group_of(col)
        if col in DERIVED_N1:
            src, der = DERIVED_N1[col]["sources"], DERIVED_N1[col]["derivation"]
            meaning = DERIVED_N1[col]["meaning"]
        else:
            src, der, meaning = [col], "used as-is from the dataset", ""
        feats.append({
            "index": i, "feature": col, "group": grp,
            "source_columns": src, "derivation": der, "meaning": meaning,
            "normalization": "standard score ((x - mean) / std), statistics from TRAIN only",
            "train_mean": float(SCALER_FULL.mean[i]), "train_std": float(SCALER_FULL.std[i]),
            "in_core_state": col in CORE_STATE_COLS,
            "iot_degradable": i < (G_IOT + G_VALID),
            "action_outcome_leak_risk": col in ACTION_OUTCOME_COLUMNS,
        })
    return {
        "core_state_dim": CORE_STATE_DIM,
        "full_state_dim": FULL_STATE_DIM,
        "core_state_columns": CORE_STATE_COLS,
        "n1_extra_columns": N1_EXTRA_COLS,
        "group_sizes": {"iot_measured": G_IOT, "iot_validity": G_VALID,
                        "proprioceptive": G_PROP, "n1_physics": G_N1},
        "wohler_exponent_m": WOHLER_M,
        "action_space": ACTION_SPACE_INFO,
        "excluded_action_outcome_columns": SCHEMA["action_outcome_columns_never_observed"],
        "features": feats,
    }


STATE_MANIFEST = build_state_manifest()
for _n in EXPERIMENTS:
    with open(EXP_DIRS[_n]["base"] / "state_feature_manifest.json", "w") as f:
        json.dump(STATE_MANIFEST, f, indent=2)
with open(COMMON_DIR / "state_feature_manifest.json", "w") as f:
    json.dump(STATE_MANIFEST, f, indent=2)
print(f"\n    Saved state_feature_manifest.json ({len(STATE_MANIFEST['features'])} features) to "
      f"common/ and to every experiment directory")

## 13. IoT Degradation Module — N4

The dataset already ships a native sensor-error layer: the gap between its `true_*` and `meas_*`
columns encodes per-channel noise, bias, drift and dropout. N4 layers an **additional**,
independently configurable degradation on top of the already-measured channels. So in this notebook

* **clean** = the dataset's own sensor model, and
* **degraded** = the dataset's own sensor model *plus* a further degraded network.

That is a harder and more honest stress test than comparing against a hypothetical perfect-sensor
world no deployment ever has.

### The four mechanisms

| | mechanism | parameter | implementation |
|---|---|---|---|
| A | Gaussian noise | `IOT_NOISE_STD` | additive, scaled by each channel's **TRAIN** standard deviation |
| B | Dropout | `IOT_DROPOUT_PROB` | per-step packet loss; hold the last good in-trajectory reading |
| C | Persistent bias | `IOT_BIAS_MAGNITUDE` | one constant offset per trajectory per channel, drawn once |
| D | Stale observation | `IOT_STALE_PROB` | report the previous in-trajectory reading instead of the current one |

Dropout and staleness are **trajectory-aware**: a hold-last never reaches back across a trajectory
boundary, which would fabricate continuity between unrelated metocean realisations. The forward-fill
is fully vectorised (`np.maximum.accumulate` over a masked index) — no Python row loop — because it
runs on every minibatch.

### The invariant that matters

The engine receives a **copy of the observation matrix** and returns a new array. It never sees, and
therefore cannot touch, reward, action labels, `true_*` ground truth, damage columns, trajectory ids
or split ids. Degrading observations must change *what the policy sees*, never *what the world did*.
A unit test below asserts this byte-for-byte.

Reproducibility: every configuration carries its own seed derived from `SEED`, so a given
(config, batch) pair produces the same degradation on every run.

In [ ]:
@dataclass
class IoTConfig:
    """One named degradation setting. `seed` makes the corruption reproducible."""
    noise_std: float = 0.0
    dropout_prob: float = 0.0
    bias_magnitude: float = 0.0
    stale_prob: float = 0.0
    seed: int = 0
    name: str = "custom"


def _hold_last_vectorised(values: np.ndarray, drop: np.ndarray, traj: np.ndarray) -> np.ndarray:
    """Forward-fill `values` where `drop`, resetting at every trajectory boundary.

    The first row of a trajectory is never held (there is nothing valid to hold), so a dropout there
    simply keeps the current reading rather than importing a value from the previous trajectory.
    """
    n = values.size
    idx = np.arange(n)
    start = np.empty(n, dtype=bool)
    start[0] = True
    start[1:] = traj[1:] != traj[:-1]
    effective = drop & (~start)
    good = np.where(effective, -1, idx)
    good[start] = idx[start]                     # force a reset at each trajectory start
    return values[np.maximum.accumulate(good)]


class IoTDegradationEngine:
    """Applies noise / dropout / bias / staleness to an observation matrix copy."""

    def __init__(self, config: IoTConfig):
        self.config = config

    def apply(self, obs: np.ndarray, traj_ids: np.ndarray, channel_scale: np.ndarray,
              batch_offset: int = 0) -> np.ndarray:
        cfg = self.config
        rng = np.random.default_rng(cfg.seed + int(batch_offset))
        out = np.asarray(obs, dtype=np.float64).copy()
        n, d = out.shape

        # C. persistent per-trajectory bias (drawn once per trajectory, applied to every step)
        if cfg.bias_magnitude > 0:
            uniq, inv = np.unique(traj_ids, return_inverse=True)
            bias = rng.normal(0.0, cfg.bias_magnitude, size=(len(uniq), d)) * channel_scale[None, :]
            out += bias[inv]

        # A. gaussian noise, in units of each channel's TRAIN std
        if cfg.noise_std > 0:
            out += rng.normal(0.0, cfg.noise_std, size=out.shape) * channel_scale[None, :]

        # B. dropout with trajectory-aware hold-last, per channel
        if cfg.dropout_prob > 0:
            drop = rng.random(out.shape) < cfg.dropout_prob
            for c in range(d):
                out[:, c] = _hold_last_vectorised(out[:, c], drop[:, c], traj_ids)

        # D. staleness: report the previous in-trajectory reading
        if cfg.stale_prob > 0:
            stale = rng.random(n) < cfg.stale_prob
            shifted = np.roll(out, 1, axis=0)
            same_traj = np.roll(traj_ids, 1) == traj_ids
            m = stale & same_traj
            out[m] = shifted[m]

        return out


IOT_CLEAN = IoTConfig(seed=SEED, name="clean")
IOT_DEGRADED = IoTConfig(noise_std=IOT_NOISE_STD, dropout_prob=IOT_DROPOUT_PROB,
                         bias_magnitude=IOT_BIAS_MAGNITUDE, stale_prob=IOT_STALE_PROB,
                         seed=SEED + 1, name="combined")
IOT_NOISE_ONLY   = IoTConfig(noise_std=IOT_NOISE_STD,          seed=SEED + 2, name="noise")
IOT_DROPOUT_ONLY = IoTConfig(dropout_prob=IOT_DROPOUT_PROB,    seed=SEED + 3, name="dropout")
IOT_BIAS_ONLY    = IoTConfig(bias_magnitude=IOT_BIAS_MAGNITUDE, seed=SEED + 4, name="bias")
IOT_STALE_ONLY   = IoTConfig(stale_prob=IOT_STALE_PROB,        seed=SEED + 5, name="stale")
IOT_MODES = {c.name: c for c in (IOT_NOISE_ONLY, IOT_DROPOUT_ONLY, IOT_BIAS_ONLY,
                                 IOT_STALE_ONLY, IOT_DEGRADED)}

# Per-channel scale for the degradable block, from TRAIN only.
IOT_CHANNEL_SCALE = _train_state_full[:, IOT_DEGRADABLE_SLICE].std(axis=0)
IOT_CHANNEL_SCALE = np.where(IOT_CHANNEL_SCALE < 1e-8, 1.0, IOT_CHANNEL_SCALE)

print("=" * 79)
print("IoT DEGRADATION (N4)")
print("=" * 79)
for _n, _c in [("clean", IOT_CLEAN)] + list(IOT_MODES.items()):
    print(f"    {_n:9s} noise={_c.noise_std:.3f} dropout={_c.dropout_prob:.3f} "
          f"bias={_c.bias_magnitude:.3f} stale={_c.stale_prob:.3f} seed={_c.seed}")
print(f"\n    degradable channels ({G_IOT + G_VALID}): {FULL_STATE_COLS[IOT_DEGRADABLE_SLICE]}")
print(f"    channel scale (TRAIN std) range: [{IOT_CHANNEL_SCALE.min():.4g}, "
      f"{IOT_CHANNEL_SCALE.max():.4g}]")

In [ ]:
def test_degradation_never_touches_ground_truth() -> None:
    """The engine must not mutate reward / ground truth / ids / actions. Asserted byte-for-byte."""
    probe = transitions.iloc[:3000].copy()
    protected = [c for c in (
        [TR["reward"], TR["damage_ratio"], TR["del_ratio"], TR["act_pitch"], TR["act_yaw"],
         TR["act_ipc"], TR["episode_id"], TR["step"], TR["done"], "traj_id"]
        + [c for c in transitions.columns if c.startswith("true_") or c.startswith("next_true_")]
        + [c for c in ACTION_OUTCOME_COLUMNS if c in transitions.columns]
    ) if c in probe.columns]
    before = probe[protected].copy()

    obs = probe[FULL_STATE_COLS].to_numpy(dtype=np.float64)[:, IOT_DEGRADABLE_SLICE]
    out = IoTDegradationEngine(IOT_DEGRADED).apply(obs, probe["traj_id"].to_numpy(),
                                                  IOT_CHANNEL_SCALE)
    if not before.equals(probe[protected]):
        raise RuntimeError("[FOWT-ARISE] CRITICAL: degradation mutated protected columns.")
    if out.shape != obs.shape:
        raise RuntimeError("[FOWT-ARISE] degradation changed the observation shape.")
    if np.allclose(out, obs):
        raise RuntimeError("[FOWT-ARISE] degradation had NO effect; N4 would be vacuous. Check the "
                           "IOT_* parameters in Section 02.")
    if not np.isfinite(out).all():
        raise RuntimeError("[FOWT-ARISE] degradation produced non-finite observations.")

    # reproducibility: same config + same offset => identical corruption
    out2 = IoTDegradationEngine(IOT_DEGRADED).apply(obs, probe["traj_id"].to_numpy(),
                                                   IOT_CHANNEL_SCALE)
    if not np.allclose(out, out2):
        raise RuntimeError("[FOWT-ARISE] degradation is not reproducible for a fixed seed/offset.")

    _shift = float(np.mean(np.abs(out - obs) / IOT_CHANNEL_SCALE[None, :]))
    print(f"    [PASS] protected columns unchanged; degradation active "
          f"(mean |delta| = {_shift:.4f} channel-sigma); reproducible for a fixed seed")


test_degradation_never_touches_ground_truth()

## 14. Reward Construction — N3

### Definition

$$
R \;=\;
\underbrace{\lambda_{\text{fat}}\; f\cdot s}_{\text{fatigue benefit}}
\;-\;\underbrace{\lambda_{\text{pow}}\; p}_{\text{power penalty}}
\;-\;\underbrace{\lambda_{\text{act}}\; u}_{\text{actuator duty}}
\;-\;\underbrace{\lambda_{\text{sm}}\; \sigma}_{\text{smoothness}}
$$

with every term dimensionless:

| symbol | quantity | source |
|---|---|---|
| $f$ | fatigue relief $= 1 - \text{DEL ratio}$ | dataset column `reward_fatigue_relief` |
| $s$ | condition severity, $\mathrm{clip}\big((D_{\text{base}}w)^{1/m}/p_{90},\,0,\,2\big)$ | dataset column `reward_severity` |
| $p$ | power loss fraction $=(P_{\text{baseline}}-P)/P_{\text{rated}}$ | dataset column `reward_power_loss_fraction` |
| $u$ | actuator duty (pitch travel + hold + yaw engagement + IPC) | dataset column `reward_duty_total` |
| $\sigma$ | $\frac{1}{3}\sum_j\big|a_j^{(t)}-a_j^{(t-1)}\big|$ in normalised action units, 0 at the first step of a trajectory | **new in this work** |

Three details are easy to get wrong and are worth stating explicitly, because getting any of them
wrong silently changes what "reward" means:

1. **Power loss is normalised by *rated* power, not by baseline power.** `P_rated` is derived from
   the data as $\max(P_{\text{baseline}})$, never hard-coded.
2. **Severity multiplies the fatigue term** and reaches 2.0. Dropping it does not merely rescale the
   reward, it reweights *which conditions matter*.
3. **The smoothness term is genuinely new.** The dataset's duty term already penalises actuator
   usage, but it does not isolate *action change between consecutive steps*. This is the one additive
   extension N3 contributes beyond the dataset's own composition.

### Why the default weights reproduce the dataset's own reward

With `REWARD_WEIGHT_PRESET = "dataset_consistent"` the first three terms reconstruct the dataset's
native `reward` column to floating-point precision, which is verified below and **enforced**. That
matters for two reasons: any baseline scored against the dataset's reward remains directly
comparable, and it proves the notebook has not quietly invented its own objective. Selecting
`"spec_default"` uses the brief's illustrative weights instead; the check then downgrades to a
warning and cross-comparability with the baselines is explicitly forfeited.

### Ablation N3

Ablation N3 replaces $R$ with the **load-relief term alone**, $\lambda_{\text{fat}} f s$ — no power,
duty or smoothness penalty. This is where N3 actually bites: the reward is also what *selects the
imitation target* at each operating point, so the ablation will happily choose aggressive feathering
and pay for it in power.

In [ ]:
RATED_POWER_W = float(max(transitions[TR["power_baseline"]].max(),
                          action_sweep[SW["power_baseline"]].max()))

# actuator-duty constants, matching the dataset's own actuator model (steady-state / action-held form)
_PITCH_SPAN = 8.0
_DUTY_NORMALISER = 1.0 + 0.1 + 1.0 + 1.0     # pitch travel + pitch hold + yaw engagement + IPC


def action_smoothness(a_norm: np.ndarray, traj_ids: np.ndarray) -> np.ndarray:
    """Mean |delta a| vs the previous step of the SAME trajectory; exactly 0 at a trajectory start."""
    a = np.asarray(a_norm, dtype=np.float64)
    same = np.zeros(len(a), dtype=bool)
    same[1:] = traj_ids[1:] == traj_ids[:-1]
    delta = np.zeros_like(a)
    delta[1:][same[1:]] = a[1:][same[1:]] - a[:-1][same[1:]]
    return np.mean(np.abs(delta), axis=-1)


def steady_state_duty(pitch_deg, ipc_level) -> np.ndarray:
    """`actuator_duty` with the action held: no pitch travel, no yaw slew. Used for static sweep rows."""
    return (0.1 * np.abs(np.asarray(pitch_deg, dtype=np.float64)) / _PITCH_SPAN
            + 1.0 * np.clip(np.asarray(ipc_level, dtype=np.float64), 0.0, 1.0)) / _DUTY_NORMALISER


def reward_components(frame: pd.DataFrame) -> dict:
    """Every N3 component for `frame`, from validated dataset columns + the new smoothness term."""
    f = frame[TR["rw_fatigue"]].to_numpy(dtype=np.float64)
    s = frame[TR["rw_severity"]].to_numpy(dtype=np.float64)
    p = frame[TR["rw_power"]].to_numpy(dtype=np.float64)
    u = frame[TR["rw_duty"]].to_numpy(dtype=np.float64)
    native = frame[TR["reward"]].to_numpy(dtype=np.float64)

    a_norm = action_to_norm(frame[ACTION_COLS].to_numpy(dtype=np.float64))
    sm = action_smoothness(a_norm, frame["traj_id"].to_numpy())

    fatigue_term = LAMBDA_FATIGUE * f * s
    power_pen    = LAMBDA_POWER * p
    duty_pen     = LAMBDA_ACTUATION * u
    smooth_pen   = LAMBDA_SMOOTHNESS * sm

    base = fatigue_term - power_pen - duty_pen
    return {
        "fatigue_term": fatigue_term, "power_penalty": power_pen,
        "actuation_penalty": duty_pen, "smoothness_penalty": smooth_pen,
        "reward_n3": base - smooth_pen,
        "reward_native": native,
        "reconstruction_error": float(np.max(np.abs(base - native))),
    }


def reward_ablation_n3(frame: pd.DataFrame) -> np.ndarray:
    """Ablation N3: load-relief only. No power, duty or smoothness penalty."""
    f = frame[TR["rw_fatigue"]].to_numpy(dtype=np.float64)
    s = frame[TR["rw_severity"]].to_numpy(dtype=np.float64)
    return LAMBDA_FATIGUE * f * s


print("=" * 79)
print("REWARD CONSTRUCTION (N3)")
print("=" * 79)
print(f"    preset          : {REWARD_WEIGHT_PRESET}")
print(f"    lambda_fatigue  : {LAMBDA_FATIGUE}")
print(f"    lambda_power    : {LAMBDA_POWER}")
print(f"    lambda_actuation: {LAMBDA_ACTUATION}")
print(f"    lambda_smooth   : {LAMBDA_SMOOTHNESS}")
print(f"    P_rated (derived from data, never hard-coded) = {RATED_POWER_W:,.0f} W")

_rc_train = reward_components(transitions.loc[train_mask])
_err = _rc_train["reconstruction_error"]
print(f"\n    reconstruction vs the dataset's own `reward` column: max abs error = {_err:.3e}")
if REWARD_WEIGHT_PRESET == "dataset_consistent":
    if _err > 1e-3:
        raise RuntimeError(
            f"[FOWT-ARISE] the 'dataset_consistent' preset does NOT reproduce the dataset's own "
            f"reward column (max abs error {_err:.6f}). Either the dataset's RewardConfig differs "
            f"from (2.0, 1.0, 0.05), or a reward term column has changed meaning. Refusing to "
            f"continue with an objective that silently disagrees with the data it is scored on."
        )
    print("    [PASS] the multi-objective reward reproduces the dataset's native reward exactly,")
    print("           so every number here stays comparable with the dataset and its baselines.")
else:
    warnings.warn(
        f"[FOWT-ARISE] preset '{REWARD_WEIGHT_PRESET}' does not reproduce the dataset's native "
        f"reward (max abs error {_err:.6f}). This is a legitimate choice, but comparability with "
        f"baselines scored on the dataset's own reward is forfeited. Reported, not hidden.")

# ---- component statistics: confirm no term dominates purely through units ----
print("\n    component statistics on TRAIN (already dimensionless; weights applied):")
_stats_rows = []
for _k in ("fatigue_term", "power_penalty", "actuation_penalty", "smoothness_penalty",
           "reward_n3", "reward_native"):
    _v = _rc_train[_k]
    _row = dict(component=_k, mean=float(np.mean(_v)), std=float(np.std(_v)),
                min=float(np.min(_v)), max=float(np.max(_v)),
                mean_abs=float(np.mean(np.abs(_v))))
    _stats_rows.append(_row)
    print(f"        {_k:20s} mean={_row['mean']:+9.5f} std={_row['std']:8.5f} "
          f"range=[{_row['min']:+8.4f}, {_row['max']:+8.4f}] mean|.|={_row['mean_abs']:.5f}")

_scales = {r["component"]: r["mean_abs"] for r in _stats_rows
           if r["component"].endswith(("term", "penalty"))}
_mx, _mn = max(_scales.values()), min(v for v in _scales.values() if v > 0)
print(f"\n    largest / smallest mean|component| ratio = {_mx / _mn:.1f}")
if _mx / _mn > 1e3:
    warnings.warn(f"[FOWT-ARISE] reward components span {_mx/_mn:.0f}x in magnitude; one term may "
                  f"numerically dominate. Review the lambdas in Section 02.")

REWARD_COMPONENT_STATS = _stats_rows
pd.DataFrame(_stats_rows).to_csv(COMMON_DIR / "reward_component_stats.csv", index=False)
print(f"    Saved {COMMON_DIR / 'reward_component_stats.csv'}")

## 15. FOWT-ARISE Architecture &nbsp;·&nbsp; 16. Adaptive Control Gate &nbsp;·&nbsp; 17. Critic

```
                physics-informed state  (FULL_STATE_DIM)
                              |
        +---------------------------------------------+
        |  PhysicsInformedEncoder            (N1)     |
        |    IoT measured   --> MLP(32) --+           |
        |    IoT validity   --> MLP(32) --+-> concat  |
        |    proprioceptive --> MLP(32) --+   -> Linear -> LayerNorm -> latent(64)
        |    N1 physics     --> MLP(16) --+           |
        +---------------------------------------------+
                              |
                 +------------+-------------+
                 v                          v
      +---------------------+     +--------------------------+
      | JointActuatorActor  |     | ConservativeTwinCritic   |
      |         (N2)        |     |   own encoder (not the   |
      |  trunk MLP(64)      |     |   actor's drifting one)  |
      |   |                 |     |   Q1(s,a), Q2(s,a)       |
      |   +-> head(3)       |     |   + CQL logsumexp penalty|
      |   +-> AdaptiveGate  |     +--------------------------+
      |        benefit(1)   |
      |        gate(3)      |
      |  a = clamp(scale * tanh(head * gate))
      +---------------------+
```

**N1 — grouped encoder.** Each physically distinct group is projected through its own small MLP
before fusion, rather than concatenating 30-odd heterogeneous features into one linear layer.
Near-binary validity flags and continuous physical measurements have very different statistics; a
grouped encoder lets each learn a representation suited to its own scale first. With
`use_n1=False` (Ablation N1) the encoder collapses to a single flat MLP over `CORE_STATE_COLS` and
the physics block is not merely zeroed but genuinely absent from the input.

**N2 — the adaptive control gate.** The gate is a learned, fully differentiable module, **not** a
threshold:

$$
b = \mathrm{MLP}_b(z) \in \mathbb{R},\qquad
g = \sigma\!\big(\mathrm{MLP}_g([z, b])\big) \in (0,1)^3,\qquad
a = \mathrm{clamp}\big(\kappa\cdot\tanh(h \odot g),\,-1,\,1\big)
$$

$b$ is an explicit *expected-net-benefit-of-acting* scalar. It is learned end-to-end from the action
loss and additionally grounded by a small auxiliary term (`LAMBDA_GATE_AUX`) against whether the
TRAIN best-action target at that operating point is non-neutral. Because `controllable_share_*` and
the aerodynamic gains flow into $z$, the gate can learn to suppress action magnitude toward neutral
exactly where control authority is low — above rated wind speed, where wave loading dominates and
the physically correct move is to do nothing. Ablation N2 removes the gate and the benefit head
entirely while keeping the head **joint** over all three actuators: N2 ablates the *adaptive gating*,
not the single-policy design, which is a fixed architectural commitment.

**Scaled tanh.** $\kappa =$ `ACTOR_OUTPUT_SCALE` $> 1$ makes the **corners** of the action box exactly
attainable. With plain `tanh`, "pitch $=0$" and "IPC $\in\{0,1\}$" are asymptotes the policy can only
approach — yet the physics-optimal action sits on those corners in most operating conditions, so
plain `tanh` would cap achievable performance for a purely parameterisation reason.

**Critic.** Twin Q heads over **their own encoder**. Sharing the actor's encoder and detaching the
latent is tempting but wrong here: $Q$ would then be defined over a representation that drifts as the
actor trains, so $\partial Q/\partial a$ compares values computed in two different feature spaces.
The conservative (CQL) penalty pushes down $\log\sum_{a'}\exp Q(s,a')$ over sampled actions while
pulling up $Q(s,a_{\text{data}})$, which suppresses the out-of-support optimism that makes naive
offline Q-learning select actions the dataset never demonstrates.

In [ ]:
class PhysicsInformedEncoder(nn.Module):
    """N1 grouped state encoder. `use_n1=False` collapses to a flat MLP over the core state."""

    def __init__(self, use_n1: bool = True, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.use_n1 = use_n1
        self.in_dim = FULL_STATE_DIM if use_n1 else CORE_STATE_DIM
        if use_n1:
            self.iot_proj    = nn.Sequential(nn.Linear(G_IOT, 32), nn.ReLU())
            self.valid_proj  = nn.Sequential(nn.Linear(G_VALID, 32), nn.ReLU())
            self.prop_proj   = nn.Sequential(nn.Linear(G_PROP, 32), nn.ReLU())
            self.n1_proj     = nn.Sequential(nn.Linear(G_N1, 16), nn.ReLU())
            self.fuse = nn.Linear(32 * 3 + 16, latent_dim)
        else:
            self.flat = nn.Sequential(nn.Linear(CORE_STATE_DIM, latent_dim), nn.ReLU())
        self.norm = nn.LayerNorm(latent_dim)

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        if not self.use_n1:
            return self.norm(self.flat(state[:, :CORE_STATE_DIM]))
        a = 0
        iot = state[:, a:a + G_IOT];                 a += G_IOT
        val = state[:, a:a + G_VALID];               a += G_VALID
        prp = state[:, a:a + G_PROP];                a += G_PROP
        n1  = state[:, a:a + G_N1]
        fused = torch.cat([self.iot_proj(iot), self.valid_proj(val),
                           self.prop_proj(prp), self.n1_proj(n1)], dim=-1)
        return self.norm(self.fuse(fused))


class AdaptiveControlGate(nn.Module):
    """N2: learned, differentiable control-authority gate + expected-net-benefit head."""

    def __init__(self, latent_dim: int = LATENT_DIM, action_dim: int = ACTION_DIM):
        super().__init__()
        self.benefit = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 1))
        self.gate    = nn.Sequential(nn.Linear(latent_dim + 1, 32), nn.ReLU(),
                                     nn.Linear(32, action_dim))

    def forward(self, latent: torch.Tensor):
        b = self.benefit(latent)                                   # (B, 1) expected net benefit
        g = torch.sigmoid(self.gate(torch.cat([latent, b], dim=-1)))  # (B, action_dim) in (0,1)
        return g, b


class JointActuatorActor(nn.Module):
    """One joint head for [pitch, yaw, IPC], optionally gated. Never three separate policies."""

    def __init__(self, latent_dim: int = LATENT_DIM, action_dim: int = ACTION_DIM,
                 use_gate: bool = True, output_scale: float = ACTOR_OUTPUT_SCALE):
        super().__init__()
        self.use_gate = use_gate
        self.output_scale = output_scale
        self.trunk = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU())
        self.head  = nn.Linear(64, action_dim)
        self.gate  = AdaptiveControlGate(latent_dim, action_dim) if use_gate else None

    def forward(self, latent: torch.Tensor):
        raw = self.head(self.trunk(latent))
        if self.use_gate:
            g, b = self.gate(latent)
            raw = raw * g
        else:
            g = torch.ones_like(raw)
            b = torch.zeros(raw.shape[0], 1, device=raw.device, dtype=raw.dtype)
        action = torch.clamp(self.output_scale * torch.tanh(raw), -1.0, 1.0)
        return action, g, b


class FowtAriseActor(nn.Module):
    """Encoder (N1) + joint gated actor (N2). Ablations are expressed purely through the two flags."""

    def __init__(self, use_n1: bool = True, use_gate: bool = True):
        super().__init__()
        self.encoder = PhysicsInformedEncoder(use_n1=use_n1)
        self.policy  = JointActuatorActor(use_gate=use_gate)
        self.use_n1, self.use_gate = use_n1, use_gate

    def forward(self, state: torch.Tensor):
        z = self.encoder(state)
        action, gate, benefit = self.policy(z)
        return action, z, gate, benefit


class ConservativeTwinCritic(nn.Module):
    """Twin Q heads over their OWN encoder, plus a CQL conservative penalty."""

    def __init__(self, use_n1: bool = True, latent_dim: int = LATENT_DIM,
                 action_dim: int = ACTION_DIM):
        super().__init__()
        self.encoder = PhysicsInformedEncoder(use_n1=use_n1)

        def q_head():
            return nn.Sequential(nn.Linear(latent_dim + action_dim, 64), nn.ReLU(),
                                 nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))

        self.q1, self.q2 = q_head(), q_head()

    def forward(self, state: torch.Tensor, action: torch.Tensor):
        z = self.encoder(state)
        x = torch.cat([z, action], dim=-1)
        return self.q1(x), self.q2(x)

    def q_min(self, state, action):
        a, b = self.forward(state, action)
        return torch.min(a, b)

    def cql_penalty(self, state: torch.Tensor, action_data: torch.Tensor,
                    action_policy: torch.Tensor, n_random: int = 10) -> torch.Tensor:
        """CQL(H): log-sum-exp over candidate actions minus Q at the dataset action.

        Pushes value DOWN on actions the dataset never demonstrates and UP on the ones it does, so
        the critic cannot become arbitrarily optimistic off-support.
        """
        B = state.shape[0]
        rand = torch.empty(B, n_random, ACTION_DIM, device=state.device).uniform_(-1.0, 1.0)
        cands = torch.cat([rand, action_policy.detach().unsqueeze(1)], dim=1)   # (B, n+1, A)
        n_c = cands.shape[1]
        s_rep = state.unsqueeze(1).expand(-1, n_c, -1).reshape(B * n_c, -1)
        a_rep = cands.reshape(B * n_c, ACTION_DIM)
        q1, q2 = self.forward(s_rep, a_rep)
        q1 = q1.view(B, n_c); q2 = q2.view(B, n_c)
        q1_d, q2_d = self.forward(state, action_data)
        pen1 = torch.logsumexp(q1, dim=1).mean() - q1_d.mean()
        pen2 = torch.logsumexp(q2, dim=1).mean() - q2_d.mean()
        return pen1 + pen2


# ---- per-experiment architecture flags: one dict, so "what an ablation removes" and "what it
# ---- actually builds" cannot drift apart.
ARCH_FLAGS = {
    "FOWT_ARISE":  dict(use_n1=True,  use_gate=True),
    "ABLATION_N1": dict(use_n1=False, use_gate=True),   # physics state removed
    "ABLATION_N2": dict(use_n1=True,  use_gate=False),  # adaptive gate removed
    "ABLATION_N3": dict(use_n1=True,  use_gate=True),   # reward reduced (Section 14)
    "ABLATION_N4": dict(use_n1=True,  use_gate=True),   # trained clean, no consistency loss
}


def count_parameters(module: nn.Module) -> int:
    return int(sum(p.numel() for p in module.parameters() if p.requires_grad))


def build_actor_critic(experiment: str):
    """Fresh actor + critic for `experiment`, from ARCH_FLAGS. No checkpoint is ever shared."""
    if experiment not in ARCH_FLAGS:
        raise RuntimeError(f"[FOWT-ARISE] unknown experiment '{experiment}'. "
                           f"Known: {list(ARCH_FLAGS)}")
    f = ARCH_FLAGS[experiment]
    return FowtAriseActor(**f), ConservativeTwinCritic(use_n1=f["use_n1"])


def state_columns_for(experiment: str) -> list:
    return FULL_STATE_COLS if ARCH_FLAGS[experiment]["use_n1"] else CORE_STATE_COLS


def scaler_for(experiment: str) -> "FeatureScaler":
    return SCALER_FULL if ARCH_FLAGS[experiment]["use_n1"] else SCALER_CORE


print("=" * 79)
print("ARCHITECTURE SMOKE TEST")
print("=" * 79)
PARAM_COUNTS = {}
for _n in EXPERIMENTS:
    _a, _c = build_actor_critic(_n)
    _cols = state_columns_for(_n)
    _x = torch.randn(8, len(_cols))
    _act, _z, _g, _b = _a(_x)
    _q1, _q2 = _c(_x, _act)
    assert _act.shape == (8, ACTION_DIM), f"{_n}: action shape {_act.shape}"
    assert _g.shape == (8, ACTION_DIM) and _b.shape == (8, 1)
    assert _q1.shape == (8, 1) and _q2.shape == (8, 1)
    assert float(_act.min()) >= -1.0 - 1e-6 and float(_act.max()) <= 1.0 + 1e-6
    _pen = _c.cql_penalty(_x, _act.detach(), _act.detach())
    assert _pen.ndim == 0 and torch.isfinite(_pen)
    PARAM_COUNTS[_n] = {"actor": count_parameters(_a), "critic": count_parameters(_c),
                        "total": count_parameters(_a) + count_parameters(_c)}
    print(f"    {_n:12s} state_dim={len(_cols):3d} use_n1={str(_a.use_n1):5s} "
          f"use_gate={str(_a.use_gate):5s} actor={PARAM_COUNTS[_n]['actor']:6,d} "
          f"critic={PARAM_COUNTS[_n]['critic']:6,d} total={PARAM_COUNTS[_n]['total']:6,d}")
    del _a, _c
print(f"\n    FOWT-ARISE trainable parameters: {PARAM_COUNTS['FOWT_ARISE']['total']:,}")
print("    [OK] every configuration builds, runs forward, respects action bounds, and the CQL")
print("         penalty is finite")

## 18. Training Utilities

### 18a. Counterfactual scorer — defined here, on purpose

Section 27 is where the counterfactual evaluation is *reported*, but the machinery has to exist
**before** training, because model selection uses the **validation counterfactual objective** as its
monitored metric. Defining it later would create exactly the forward dependency that §49 forbids, so
it is built here and merely *applied* in Section 27.

### Deriving the sweep objective (§7)

The action-sweep file has **no native `sweep_objective` column**, and none is fabricated. The
objective is derived from the verified sweep quantities using the *same* reward definition as
Section 14:

$$
r_{\text{sweep}} \;=\; \lambda_{\text{fat}}\,f\,s \;-\; \lambda_{\text{pow}}\,p \;-\; \lambda_{\text{act}}\,u
$$

$$
f = 1 - \big(\text{damage\_ratio\_max}\big)^{1/m},\qquad
p = \mathrm{clip}\!\left(\frac{P_{\text{baseline}} - P}{P_{\text{rated}}},\,0,\,1\right),\qquad
u = \text{steady-state actuator duty}
$$

$$
s = \mathrm{clip}\!\left(\frac{\big(D_{\text{max}}^{(a=0)}\cdot w\big)^{1/m}}{p_{90}},\,0,\,2\right)
$$

Severity is anchored on each condition's **zero-action** row — the no-control baseline damage — and
$p_{90}$ is the 90th percentile of lifetime DEL across conditions, matching the dataset's own
severity normalisation. Duty uses the *action-held* (steady-state) form because a static sweep row
has no previous action from which pitch travel or yaw slew could be computed.

The resulting number is reported throughout under one name only: **Mean Matched Sweep Objective**.
It is a *counterfactual estimate* of what the policy's action would have produced under the same
physical condition — **never** presented as a measured transition reward.

### Efficiency

The sweep is ~1.45 M rows and is touched once. Each tower is pre-indexed into contiguous
`(n_conditions, n_actions, ...)` blocks so matching is a single vectorised distance computation per
tower, with no repeated dataframe filtering or copying.

In [ ]:
def build_sweep_index(tower: str) -> dict:
    """Pre-index one tower's sweep into per-condition blocks, scored with the Section 14 reward."""
    if "action_sweep" not in globals():
        raise RuntimeError(
            "[FOWT-ARISE] the raw action-sweep frame has already been released to save memory, so "
            "the index cannot be rebuilt in-place. Re-run Section 04 (Dataset Loading) first.")
    sw = action_sweep[action_sweep[SW["tower"]] == tower]

    zero = sw[(sw[SW["act_pitch"]] == 0) & (sw[SW["act_yaw"]] == 0) & (sw[SW["act_ipc"]] == 0)]
    if len(zero) == 0:
        raise RuntimeError(
            f"[FOWT-ARISE] tower '{tower}' has no zero-action sweep row. Severity is anchored on the "
            f"no-control baseline damage at each condition and cannot be derived without it.")
    zero = zero.set_index(SW["sim_id"])

    sim = sw[SW["sim_id"]].to_numpy()
    d_base_free = sw[SW["sim_id"]].map(zero[SW["damage_max"]]).to_numpy(dtype=np.float64)
    w = sw[SW["damage_weight"]].to_numpy(dtype=np.float64)

    lifetime_del = np.power(np.maximum(d_base_free * w, 0.0), 1.0 / WOHLER_M)
    per_condition = np.power(
        np.maximum(zero[SW["damage_max"]].to_numpy(dtype=np.float64)
                   * zero[SW["damage_weight"]].to_numpy(dtype=np.float64), 0.0), 1.0 / WOHLER_M)
    p90 = float(np.percentile(per_condition, 90.0)) or 1.0
    severity = np.clip(lifetime_del / p90, 0.0, 2.0)

    del_ratio = damage_ratio_to_del_ratio(sw[SW["damage_ratio_max"]].to_numpy(dtype=np.float64))
    fatigue_relief = del_ratio_to_fatigue_relief(del_ratio)
    power_loss = np.clip((sw[SW["power_baseline"]].to_numpy(dtype=np.float64)
                          - sw[SW["power_w"]].to_numpy(dtype=np.float64)) / RATED_POWER_W, 0.0, 1.0)
    duty = steady_state_duty(sw[SW["act_pitch"]].to_numpy(), sw[SW["act_ipc"]].to_numpy())

    reward = (LAMBDA_FATIGUE * fatigue_relief * severity
              - LAMBDA_POWER * power_loss - LAMBDA_ACTUATION * duty)

    order = np.argsort(sim, kind="stable")
    sims, counts = np.unique(sim[order], return_counts=True)
    n_act = int(counts[0])
    if not np.all(counts == n_act):
        raise RuntimeError(f"[FOWT-ARISE] tower '{tower}' sweep blocks are non-uniform.")

    actions_norm = action_to_norm(
        sw[SWEEP_ACTION_COLS].to_numpy(dtype=np.float64)[order]).reshape(len(sims), n_act, ACTION_DIM)

    packed = {}
    for k, arr in {
        "reward": reward, "fatigue_relief": fatigue_relief, "power_loss_fraction": power_loss,
        "del_ratio": del_ratio, "severity": severity, "duty": duty,
        "damage_ratio_max": sw[SW["damage_ratio_max"]].to_numpy(dtype=np.float64),
        "controllable_share_max": sw[SW["ctrl_share_max"]].to_numpy(dtype=np.float64),
    }.items():
        packed[k] = arr[order].reshape(len(sims), n_act)

    return {"sim_to_block": {int(s): j for j, s in enumerate(sims)},
            "actions_norm": actions_norm, "n_actions": n_act, **packed}


SWEEP_OUTCOME_KEYS = ["reward", "fatigue_relief", "power_loss_fraction", "del_ratio",
                      "severity", "duty", "damage_ratio_max", "controllable_share_max"]
ACTION_SWEEP_OUTCOME_COL = "reward"   # the derived per-action objective inside the index

SWEEP_INDEX = {t: build_sweep_index(t) for t in sorted(towers_trans)}
print("=" * 79)
print("ACTION-SWEEP INDEX (built once; no further copies of the 1.45M-row frame)")
print("=" * 79)
for _t, _ix in SWEEP_INDEX.items():
    print(f"    tower '{_t}': {len(_ix['sim_to_block']):,} conditions x {_ix['n_actions']} actions "
          f"| derived objective range [{_ix['reward'].min():+.4f}, {_ix['reward'].max():+.4f}]")
print(f"    m = {WOHLER_M:.6f}   P_rated = {RATED_POWER_W:,.0f} W")
print(f"    objective name reported everywhere: 'Mean Matched Sweep Objective' (counterfactual)")

# The full sweep frame (~1.45M x 27) is superseded by the index; release it so no later cell can
# accidentally copy it. Section 04 must be re-run before this cell if you want to rebuild the index.
if "action_sweep" in globals():
    _sweep_mb = float(action_sweep.memory_usage(deep=True).sum()) / 1e6
    del action_sweep
    gc.collect()
    print(f"    released the raw action-sweep dataframe (~{_sweep_mb:,.0f} MB); index retained")

In [ ]:
def match_actions_to_sweep(actions_phys: np.ndarray, sim_ids: np.ndarray, towers: np.ndarray):
    """Nearest-action counterfactual match, per tower, in the TRAIN-derived normalised action space.

    Returns (outcomes, covered, distance). Matching happens ONLY within an identical physical
    operating condition (same tower + sim_id); rows whose condition is absent from the sweep are
    left NaN and reported as uncovered rather than matched to something unrelated.
    """
    n = len(actions_phys)
    out = {k: np.full(n, np.nan) for k in SWEEP_OUTCOME_KEYS}
    covered = np.zeros(n, dtype=bool)
    dist_out = np.full(n, np.nan)
    query = action_to_norm(actions_phys)

    for tower in np.unique(towers):
        ix = SWEEP_INDEX.get(tower)
        if ix is None:
            continue
        tmask = towers == tower
        blocks = np.array([ix["sim_to_block"].get(int(s), -1) for s in sim_ids[tmask]])
        ok = blocks >= 0
        rows = np.flatnonzero(tmask)[ok]
        blocks = blocks[ok]
        if len(rows) == 0:
            continue
        d = np.linalg.norm(ix["actions_norm"][blocks] - query[rows][:, None, :], axis=2)
        best = np.argmin(d, axis=1)
        dist_out[rows] = d[np.arange(len(rows)), best]
        covered[rows] = True
        for k in SWEEP_OUTCOME_KEYS:
            out[k][rows] = ix[k][blocks, best]
    return out, covered, dist_out


def score_actions(actions_phys: np.ndarray, frame: pd.DataFrame) -> dict:
    """Episode-level counterfactual scoring for a set of actions on `frame`.

    Aggregation is per TRAJECTORY first, then across trajectories. Treating all ~19k transitions as
    independent samples would be pseudoreplication: transitions inside one trajectory share a
    metocean realisation, so their errors are strongly correlated.
    """
    outcomes, covered, dist = match_actions_to_sweep(
        actions_phys, frame[MATCH_KEY].to_numpy(), frame[TR["tower"]].to_numpy())
    per_row = pd.DataFrame({
        "traj_id": frame["traj_id"].to_numpy()[covered],
        "reward": outcomes["reward"][covered],
        "fatigue_relief": outcomes["fatigue_relief"][covered],
        "power_loss_fraction": outcomes["power_loss_fraction"][covered],
        "del_ratio": outcomes["del_ratio"][covered],
        "duty": outcomes["duty"][covered],
        "controllable_share_max": outcomes["controllable_share_max"][covered],
    })
    per_traj = per_row.groupby("traj_id").mean()
    a_norm = action_to_norm(actions_phys)
    # Two DISTINCT quantities, and they must not collapse into each other:
    #   mean_action_magnitude -- how far the command sits from the NEUTRAL action, in normalised
    #       units. Measured against neutral, not against 0: pitch=0 maps to -1 in normalised space,
    #       so mean|a_norm| would score a do-nothing policy as maximally active.
    #   actuator_duty_proxy   -- actuator USAGE COST, using the dataset's own actuator_duty model
    #       (pitch hold + IPC activation) applied to the policy's physical action. Reusing the
    #       dataset's definition rather than inventing a second one keeps this comparable.
    mag = float(np.mean(np.abs(a_norm - ACTION_NEUTRAL_NORM[None, :])))
    duty = float(np.mean(steady_state_duty(actions_phys[:, 0], actions_phys[:, 2])))
    return {
        "matched_sweep_objective": float(per_traj["reward"].mean()),
        "mean_del_ratio": float(per_traj["del_ratio"].mean()),
        "median_del_ratio": float(np.median(outcomes["del_ratio"][covered])) if covered.any() else float("nan"),
        "fatigue_relief_pct": float(100.0 * per_traj["fatigue_relief"].mean()),
        "power_loss_pct": float(100.0 * per_traj["power_loss_fraction"].mean()),
        "actuator_duty_proxy": duty,
        "mean_action_magnitude": mag,
        "mean_abs_yaw_action": float(np.mean(np.abs(actions_phys[:, 1]))),
        "no_action_rate": float(np.mean(no_action_indicator(a_norm))),
        "coverage_pct": float(100.0 * covered.mean()),
        "mean_match_distance": float(np.nanmean(dist)) if covered.any() else float("nan"),
        "n_trajectories": int(per_traj.shape[0]),
        "_per_traj": per_traj,
        "_outcomes": outcomes, "_covered": covered, "_distance": dist,
    }


print("    match_actions_to_sweep / score_actions defined (episode-level aggregation).")

### 18b. Reference policies

An absolute objective value is uninterpretable on its own. These constant and oracle policies are
scored with **exactly the same code path** as the learned policies, which pins the achievable band:

* `do_nothing` — the neutral action everywhere; the floor a controller must beat to be worth deploying
* `ipc_half`, `ipc_only` — constant IPC activation, cheap non-adaptive load relief
* `feather2_ipc1` — aggressive constant feathering; shows what ignoring the power cost buys
* `behaviour_logged` — the mixture policy that actually collected the data
* `ORACLE_best_of_sweep` — per-condition best of the sweep grid; the ceiling reachable by *any*
  policy restricted to this action grid, and the denominator for "% of oracle"

In [ ]:
def reference_policy_scores(mask: np.ndarray) -> dict:
    """Score every reference policy on the rows selected by `mask`, via score_actions."""
    sub = transitions.loc[mask]
    n = len(sub)
    res = {}
    for name, act in {"do_nothing": (0.0, 0.0, 0.0), "ipc_half": (0.0, 0.0, 0.5),
                      "ipc_only": (0.0, 0.0, 1.0), "feather2_ipc1": (2.0, 0.0, 1.0)}.items():
        res[name] = score_actions(np.tile(np.array(act, dtype=np.float64), (n, 1)), sub)
    res["behaviour_logged"] = score_actions(sub[ACTION_COLS].to_numpy(dtype=np.float64), sub)

    oracle = np.zeros((n, ACTION_DIM))
    towers_arr, sims_arr = sub[TR["tower"]].to_numpy(), sub[MATCH_KEY].to_numpy()
    for tower in np.unique(towers_arr):
        ix = SWEEP_INDEX.get(tower)
        if ix is None:
            continue
        tmask = towers_arr == tower
        blocks = np.array([ix["sim_to_block"].get(int(s), -1) for s in sims_arr[tmask]])
        ok = blocks >= 0
        rows = np.flatnonzero(tmask)[ok]
        blocks = blocks[ok]
        best = np.argmax(ix["reward"][blocks], axis=1)
        oracle[rows] = norm_to_action(ix["actions_norm"][blocks, best])
    res["ORACLE_best_of_sweep"] = score_actions(oracle, sub)
    return res


VAL_REFERENCES = reference_policy_scores(val_mask)
ORACLE_VAL = VAL_REFERENCES["ORACLE_best_of_sweep"]["matched_sweep_objective"]

print("=" * 79)
print("REFERENCE POLICIES on VALIDATION trajectories (same evaluation code as the learned policy)")
print("=" * 79)
for _n, _s in VAL_REFERENCES.items():
    print(f"    {_n:22s} objective={_s['matched_sweep_objective']:+.5f}  "
          f"DEL={_s['mean_del_ratio']:.5f}  fatigue={_s['fatigue_relief_pct']:5.2f}%  "
          f"power_loss={_s['power_loss_pct']:6.2f}%  no_action={_s['no_action_rate']:.3f}")
print(f"\n    achievable band on VALIDATION: {VAL_REFERENCES['do_nothing']['matched_sweep_objective']:+.5f}"
      f" (do nothing)  ->  {ORACLE_VAL:+.5f} (per-condition oracle)")
if not np.isfinite(ORACLE_VAL) or abs(ORACLE_VAL) < 1e-12:
    raise RuntimeError("[FOWT-ARISE] the validation oracle objective is ~0 or non-finite; "
                       "'% of oracle' would be meaningless. Investigate the sweep index.")

### 18c. Best-action imitation targets

The actor's primary learning signal. For each physical operating point
(`BEST_ACTION_GROUP_COLS`, marginalising over turbulence seed) the **single highest-reward action
observed in the training trajectories** becomes the regression target.

Why a single best action rather than a reward-weighted average: this dataset's good actions are
*diverse* across conditions, and averaging them produces a compromise action that is worse than any
of the individual actions it averages. A single target per operating point avoids that mode-averaging
collapse. Huber loss (`IMITATION_HUBER_DELTA`) further limits the pull of outliers.

**This is where N3 bites.** The full model selects targets with the complete multi-objective reward,
so an action is only preferred if its load relief is worth its power and actuator cost. Ablation N3
selects targets with the load-relief term alone and will therefore choose aggressive feathering.

Provenance: targets come from **training trajectories only**. The action sweep is used for
evaluation and validation-based model selection, never as a training target — training on the
evaluation oracle would be circular.

In [ ]:
def selection_reward_for(experiment: str, frame: pd.DataFrame) -> np.ndarray:
    """The reward used to CHOOSE the imitation target. Ablation N3 uses load relief only."""
    if experiment == "ABLATION_N3":
        return reward_ablation_n3(frame)
    return reward_components(frame)["reward_n3"]


def training_reward_for(experiment: str, frame: pd.DataFrame):
    """(reward array, component dict) used for critic targets and epoch-wise logging."""
    if experiment == "ABLATION_N3":
        r = reward_ablation_n3(frame)
        nan = np.full(len(frame), np.nan)
        # NaN, not 0: these terms are absent from this ablation's objective and reporting them as
        # zero would imply the ablation achieved no power loss rather than not accounting for it.
        return r, {"fatigue_term": r.copy(), "power_penalty": nan.copy(),
                   "actuation_penalty": nan.copy(), "smoothness_penalty": nan.copy()}
    rc = reward_components(frame)
    return rc["reward_n3"], {k: rc[k] for k in
                             ("fatigue_term", "power_penalty", "actuation_penalty",
                              "smoothness_penalty")}


def build_best_action_targets(experiment: str) -> np.ndarray:
    """Per-operating-point best action, from TRAIN rows only. Returns (n_rows_total, ACTION_DIM)."""
    group_cols = [c for c in BEST_ACTION_GROUP_COLS if c in transitions.columns]
    missing = [c for c in BEST_ACTION_GROUP_COLS if c not in transitions.columns]
    if missing:
        raise RuntimeError(f"[FOWT-ARISE] BEST_ACTION_GROUP_COLS not in the data: {missing}")

    sel = selection_reward_for(experiment, transitions)
    sub = transitions.loc[train_mask, group_cols + ACTION_COLS].copy()
    sub["_sel"] = sel[train_mask]
    best_idx = sub.groupby(group_cols, sort=False)["_sel"].idxmax()
    best = transitions.loc[best_idx, group_cols + ACTION_COLS]

    lookup = {tuple(k): v for k, v in zip(best[group_cols].to_numpy(),
                                          best[ACTION_COLS].to_numpy(dtype=np.float64))}
    keys = list(map(tuple, transitions[group_cols].to_numpy()))
    # Every row's operating point is present because the grid recurs across splits by dataset
    # construction; a genuinely unseen point falls back to neutral rather than crashing, and is
    # counted so the fallback can never be silent.
    n_fallback = 0
    targets = np.empty((len(transitions), ACTION_DIM), dtype=np.float64)
    for i, k in enumerate(keys):
        v = lookup.get(k)
        if v is None:
            targets[i] = ACTION_NEUTRAL
            n_fallback += 1
        else:
            targets[i] = v
    print(f"      best-action targets [{experiment}]: {len(lookup):,} operating points, "
          f"{train_mask.sum() / max(len(lookup), 1):.1f} train rows/point, "
          f"mean selected reward={float(sub.loc[best_idx.values, '_sel'].mean()):+.5f}"
          + (f", NEUTRAL FALLBACK for {n_fallback:,} rows" if n_fallback else ""))
    if n_fallback:
        warnings.warn(f"[FOWT-ARISE] {n_fallback} rows had no TRAIN operating point and fell back to "
                      f"the neutral target. They are counted, not hidden.")
    return targets


print("    selection_reward_for / training_reward_for / build_best_action_targets defined.")
print(f"    BEST_ACTION_GROUP_COLS = {BEST_ACTION_GROUP_COLS}")

### 18d. Offline replay buffer and batch preparation

The FLOATBench transitions are a fixed offline dataset, so the "replay buffer" is a CPU-resident
tensor store built once per experiment. Only the current minibatch is moved to the device, so GPU
memory stays flat and small regardless of dataset size.

**Next-state convention.** The successor is the next row in the same trajectory; a terminal row
self-loops. The successor's **measured** columns are used — never `next_true_*`. A real controller
observes its sensors at $t+1$, not ground truth, and using the latter would leak information no
deployment has.

**Where N4 enters.** `prepare_batch` is the single choke point: degradation is applied to the
measured/validity block of the *raw, unscaled* state and only then is the scaler applied, in exactly
the same order at training and evaluation time.

In [ ]:
class OfflineBuffer:
    """Fixed offline buffer over one experiment's TRAIN (or VAL) rows. Raw, unscaled, CPU-resident."""

    def __init__(self, raw_state, raw_next_state, action, reward, done, traj_ids,
                 target_action=None, components=None, seed: int = SEED):
        n = raw_state.shape[0]
        for nm, arr in (("next_state", raw_next_state), ("action", action),
                        ("reward", reward), ("done", done), ("traj_ids", traj_ids)):
            if len(arr) != n:
                raise RuntimeError(f"[FOWT-ARISE] buffer length mismatch: {nm}={len(arr)} vs {n}")
        self.raw_state = np.ascontiguousarray(raw_state, dtype=np.float32)
        self.raw_next_state = np.ascontiguousarray(raw_next_state, dtype=np.float32)
        self.action = np.ascontiguousarray(action, dtype=np.float32)
        self.reward = np.ascontiguousarray(reward, dtype=np.float32)
        self.done = np.ascontiguousarray(done, dtype=np.float32)
        self.traj_ids = np.asarray(traj_ids)
        self.target_action = (None if target_action is None
                              else np.ascontiguousarray(target_action, dtype=np.float32))
        self.components = components or {}
        self.n = n
        self._rng = np.random.default_rng(seed)

    def sample_indices(self, batch_size: int) -> np.ndarray:
        return self._rng.integers(0, self.n, size=batch_size)

    def __len__(self):
        return self.n


def build_buffer(experiment: str, mask: np.ndarray, reward_arr: np.ndarray,
                 components: dict, targets_full=None, seed: int = SEED) -> OfflineBuffer:
    """Assemble an OfflineBuffer for `mask`, using the successor-row next-state convention."""
    cols = state_columns_for(experiment)
    sub = transitions.loc[mask]
    pos = np.flatnonzero(mask)                       # positions into the (already sorted) frame

    done = sub[TR["done"]].to_numpy(dtype=np.int8)
    n = len(sub)
    succ = np.arange(n) + 1
    succ[-1] = n - 1
    is_last = done.astype(bool)
    succ[is_last] = np.flatnonzero(is_last)          # terminal rows self-loop

    # Guard: the successor must belong to the same trajectory, or the bootstrap crosses trajectories.
    traj = sub["traj_id"].to_numpy()
    cross = (traj[succ] != traj) & (~is_last)
    if cross.any():
        raise RuntimeError(
            f"[FOWT-ARISE] {int(cross.sum())} non-terminal rows would bootstrap across a trajectory "
            f"boundary. Rows must be sorted by (tower, episode_id, step) -- see Section 07.")

    state = sub[cols].to_numpy(dtype=np.float64)
    return OfflineBuffer(
        raw_state=state,
        raw_next_state=state[succ],
        action=action_to_norm(sub[ACTION_COLS].to_numpy(dtype=np.float64)),
        reward=reward_arr[pos],
        done=done.astype(np.float32),
        traj_ids=traj,
        target_action=None if targets_full is None else action_to_norm(targets_full[pos]),
        components={k: v[pos] for k, v in components.items()},
        seed=seed,
    )


def prepare_batch(buf: OfflineBuffer, idx: np.ndarray, scaler: FeatureScaler,
                  degrade: Optional[IoTDegradationEngine] = None, batch_offset: int = 0) -> dict:
    """Gather a minibatch; optionally degrade the observation block BEFORE scaling; move to DEVICE.

    Returns both the clean and (when degrading) the degraded state, so Section 21's consistency loss
    can compare the policy's own output under the two views of the same underlying transition.
    """
    raw_s = buf.raw_state[idx].astype(np.float64)
    raw_ns = buf.raw_next_state[idx].astype(np.float64)

    s_clean = scaler.transform(raw_s)
    ns_clean = scaler.transform(raw_ns)
    out = {
        "state": torch.from_numpy(s_clean.astype(np.float32)).to(DEVICE),
        "next_state": torch.from_numpy(ns_clean.astype(np.float32)).to(DEVICE),
        "action": torch.from_numpy(buf.action[idx]).to(DEVICE),
        "reward": torch.from_numpy(buf.reward[idx]).unsqueeze(-1).to(DEVICE),
        "done": torch.from_numpy(buf.done[idx]).unsqueeze(-1).to(DEVICE),
        "state_degraded": None,
    }
    out["target_action"] = (None if buf.target_action is None
                            else torch.from_numpy(buf.target_action[idx]).to(DEVICE))

    if degrade is not None:
        traj = buf.traj_ids[idx]
        d_s, d_ns = raw_s.copy(), raw_ns.copy()
        d_s[:, IOT_DEGRADABLE_SLICE] = degrade.apply(
            raw_s[:, IOT_DEGRADABLE_SLICE], traj, IOT_CHANNEL_SCALE, batch_offset)
        d_ns[:, IOT_DEGRADABLE_SLICE] = degrade.apply(
            raw_ns[:, IOT_DEGRADABLE_SLICE], traj, IOT_CHANNEL_SCALE, batch_offset + 1)
        out["state_degraded"] = torch.from_numpy(
            scaler.transform(d_s).astype(np.float32)).to(DEVICE)
        out["next_state_degraded"] = torch.from_numpy(
            scaler.transform(d_ns).astype(np.float32)).to(DEVICE)
    return out


class RunningMean:
    """Epoch-level scalar accumulator that ignores NaN (TD3's delayed update leaves gaps)."""

    def __init__(self):
        self._v = {}

    def add(self, d: dict) -> None:
        for k, v in d.items():
            if v is None:
                continue
            try:
                fv = float(v)
            except (TypeError, ValueError):
                continue
            if math.isnan(fv):
                continue
            self._v.setdefault(k, []).append(fv)

    def means(self) -> dict:
        return {k: float(np.mean(v)) if v else float("nan") for k, v in self._v.items()}


print("    OfflineBuffer / build_buffer / prepare_batch / RunningMean defined.")

### 18e. The agent — base offline actor–critic with the N4 consistency term

One update step comprises:

**Critic.** Standard clipped double-Q target with target-policy smoothing,
$y = r + \gamma(1-d)\min_i Q_i^{\text{targ}}(s', \tilde a')$, plus the conservative CQL penalty
weighted by `CQL_ALPHA`.

**Actor** (every `POLICY_DELAY` steps):

$$
\mathcal{L}_{\text{actor}}
= \underbrace{\mathrm{Huber}\big(\pi(s),\,a^{\star}(s)\big)}_{\text{best-action imitation}}
\;-\;\underbrace{\lambda_Q\,Q_1(s,\pi(s))}_{\text{optional, default off}}
\;+\;\underbrace{\lambda_{\text{rob}}\big\|\pi(s)-\pi(\tilde s)\big\|^2}_{\text{N4 consistency}}
\;+\;\underbrace{\lambda_{\text{gate}}\,\mathrm{BCE}\big(b,\,\mathbb{1}[a^\star \neq 0]\big)}_{\text{gate grounding}}
$$

**The consistency term is N4's training-time mechanism (§21).** $\tilde s$ is the *same* transition
seen through a degraded sensor network, so the term asks the policy to commit to the same action
under both views. It is a penalty on *disagreement*, not on the reward, so it cannot inflate any
reported objective — and it is only active on batches drawn under degradation, which is
`TRAIN_DEGRADED_FRACTION` of them.

**On $\lambda_Q$ defaulting to 0.** The critic is fully built, trained, conservatively regularised
and logged, and Section 25 *measures* its accuracy. On this dataset the measured critic error is
larger than the entire decision-relevant reward range, so $\partial Q/\partial a$ is not informative
enough to improve the policy and empirically pushes it toward heavy feathering. Reporting that
measurement and defaulting the term off is the honest handling; raising `Q_IMPROVEMENT_COEF` re-enables
it for anyone who wants to verify the finding.

In [ ]:
@dataclass
class AgentConfig:
    gamma: float = GAMMA
    tau: float = TAU
    policy_delay: int = POLICY_DELAY
    policy_noise: float = POLICY_NOISE
    noise_clip: float = NOISE_CLIP
    grad_clip: float = GRAD_CLIP_NORM
    cql_alpha: float = CQL_ALPHA
    lambda_robust: float = LAMBDA_ROBUST
    lambda_gate_aux: float = LAMBDA_GATE_AUX
    q_improvement_coef: float = Q_IMPROVEMENT_COEF


class FowtAriseAgent:
    """Offline actor-critic. Knows nothing about which novelty is ablated -- only about the modules
    it was handed and the flags in its AgentConfig."""

    def __init__(self, actor: nn.Module, critic: nn.Module, cfg: AgentConfig, device: torch.device):
        self.cfg, self.device = cfg, device
        self.actor = actor.to(device)
        self.critic = critic.to(device)
        self.actor_target = deepcopy(self.actor).to(device)
        self.critic_target = deepcopy(self.critic).to(device)
        for p in list(self.actor_target.parameters()) + list(self.critic_target.parameters()):
            p.requires_grad_(False)

        self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=LEARNING_RATE_ACTOR,
                                          weight_decay=WEIGHT_DECAY)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=LEARNING_RATE)
        self.actor_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.actor_opt, mode="max", factor=LR_PLATEAU_FACTOR,
            patience=LR_PLATEAU_PATIENCE, min_lr=MIN_LR)
        self.critic_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.critic_opt, mode="max", factor=LR_PLATEAU_FACTOR,
            patience=LR_PLATEAU_PATIENCE, min_lr=MIN_LR)
        self.updates = 0

    def _soft_update(self, target, source):
        with torch.no_grad():
            for tp, sp in zip(target.parameters(), source.parameters()):
                tp.data.mul_(1.0 - self.cfg.tau).add_(sp.data, alpha=self.cfg.tau)

    def update(self, batch: dict) -> dict:
        cfg = self.cfg
        self.updates += 1
        s, a, r, ns, d = (batch["state"], batch["action"], batch["reward"],
                          batch["next_state"], batch["done"])
        s_deg = batch.get("state_degraded")
        tgt = batch.get("target_action")

        # ---------------- critic ----------------
        with torch.no_grad():
            na, _, _, _ = self.actor_target(ns)
            noise = (torch.randn_like(na) * cfg.policy_noise).clamp(-cfg.noise_clip, cfg.noise_clip)
            na = (na + noise).clamp(-1.0, 1.0)
            y = r + cfg.gamma * (1.0 - d) * self.critic_target.q_min(ns, na)

        q1, q2 = self.critic(s, a)
        td_loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)
        with torch.no_grad():
            pa_for_cql, _, _, _ = self.actor(s)
        cql = self.critic.cql_penalty(s, a, pa_for_cql)
        critic_loss = td_loss + cfg.cql_alpha * cql

        self.critic_opt.zero_grad(set_to_none=True)
        critic_loss.backward()
        cgn = torch.nn.utils.clip_grad_norm_(self.critic.parameters(), cfg.grad_clip)
        self.critic_opt.step()

        diag = {
            "critic_loss": float(critic_loss.item()), "td_loss": float(td_loss.item()),
            "cql_penalty": float(cql.item()), "critic_grad_norm": float(cgn),
            "q1_mean": float(q1.mean().item()), "q2_mean": float(q2.mean().item()),
            "target_q_mean": float(y.mean().item()),
            "actor_loss": float("nan"), "imitation_loss": float("nan"),
            "robustness_loss": float("nan"), "gate_aux_loss": float("nan"),
            "actor_grad_norm": float("nan"), "gate_mean": float("nan"),
            "benefit_mean": float("nan"),
        }

        # ---------------- actor (delayed) ----------------
        if self.updates % cfg.policy_delay == 0:
            pa, _, gate, benefit = self.actor(s)

            if tgt is not None:
                imitation = (F.huber_loss(pa, tgt, delta=IMITATION_HUBER_DELTA)
                             if IMITATION_HUBER_DELTA > 0 else F.mse_loss(pa, tgt))
            else:
                imitation = F.mse_loss(pa, a)      # behaviour-support fallback
            actor_loss = imitation

            if cfg.q_improvement_coef > 0.0:
                q_pi, _ = self.critic(s, pa)
                lam = cfg.q_improvement_coef / (q_pi.abs().mean().detach() + 1e-6)
                actor_loss = actor_loss - lam * q_pi.mean()

            # ---- N4 consistency (Section 21): same action under clean and degraded views ----
            if s_deg is not None and cfg.lambda_robust > 0.0:
                pa_deg, _, _, _ = self.actor(s_deg)
                rob = F.mse_loss(pa_deg, pa.detach()) + F.mse_loss(pa, pa_deg.detach())
                actor_loss = actor_loss + cfg.lambda_robust * rob
                diag["robustness_loss"] = float(rob.item())

            # ---- gate grounding: is acting beneficial at this operating point? ----
            if self.actor.use_gate and cfg.lambda_gate_aux > 0.0 and tgt is not None:
                acts = (tgt - torch.as_tensor(ACTION_NEUTRAL_NORM, dtype=tgt.dtype,
                                              device=tgt.device)).abs().amax(dim=1, keepdim=True)
                label = (acts > NO_ACTION_TOLERANCE).float()
                gate_aux = F.binary_cross_entropy_with_logits(benefit, label)
                actor_loss = actor_loss + cfg.lambda_gate_aux * gate_aux
                diag["gate_aux_loss"] = float(gate_aux.item())

            self.actor_opt.zero_grad(set_to_none=True)
            actor_loss.backward()
            agn = torch.nn.utils.clip_grad_norm_(self.actor.parameters(), cfg.grad_clip)
            self.actor_opt.step()

            self._soft_update(self.actor_target, self.actor)
            self._soft_update(self.critic_target, self.critic)

            diag.update(actor_loss=float(actor_loss.item()),
                        imitation_loss=float(imitation.item()),
                        actor_grad_norm=float(agn),
                        gate_mean=float(gate.mean().item()),
                        benefit_mean=float(torch.sigmoid(benefit).mean().item()))
        return diag

    @torch.no_grad()
    def act(self, state: torch.Tensor) -> np.ndarray:
        """Deterministic normalised action for evaluation. Restores train mode on exit."""
        was = self.actor.training
        self.actor.eval()
        a, _, _, _ = self.actor(state)
        if was:
            self.actor.train()
        return a.detach().cpu().numpy()

    def state_dict(self) -> dict:
        return {
            "actor": self.actor.state_dict(), "critic": self.critic.state_dict(),
            "actor_target": self.actor_target.state_dict(),
            "critic_target": self.critic_target.state_dict(),
            "actor_opt": self.actor_opt.state_dict(), "critic_opt": self.critic_opt.state_dict(),
            "actor_sched": self.actor_sched.state_dict(),
            "critic_sched": self.critic_sched.state_dict(),
            "updates": self.updates,
        }

    def load_state_dict(self, p: dict) -> None:
        self.actor.load_state_dict(p["actor"]);          self.critic.load_state_dict(p["critic"])
        self.actor_target.load_state_dict(p["actor_target"])
        self.critic_target.load_state_dict(p["critic_target"])
        self.actor_opt.load_state_dict(p["actor_opt"]);  self.critic_opt.load_state_dict(p["critic_opt"])
        self.actor_sched.load_state_dict(p["actor_sched"])
        self.critic_sched.load_state_dict(p["critic_sched"])
        self.updates = p["updates"]


print("    FowtAriseAgent defined (twin conservative critic + N4 consistency + gate grounding).")

## 19. Validation Utilities

`evaluate_actions_on_split` turns a trained policy into the full metric set on any split, under any
observation regime, through the single counterfactual code path from Section 18a. It is the only
place policy metrics are computed, so validation, test, ablation and robustness numbers are
guaranteed to share one definition — §8's requirement that metric definitions not be quietly
re-specified per experiment.

The monitored metric for model selection and early stopping is the **validation Mean Matched Sweep
Objective**.

Not the critic's own $Q$: $Q$ is the critic's *opinion*, so selecting the epoch with the highest $Q$
selects the most over-optimistic critic rather than the best policy. Not the training loss either,
which says nothing about achieved physical objective. The validation counterfactual objective is an
actual estimate of policy performance on held-out trajectories.

In [ ]:
@torch.no_grad()
def policy_actions_for_split(agent: FowtAriseAgent, experiment: str, mask: np.ndarray,
                             iot: Optional[IoTConfig] = None):
    """Deterministic physical actions for every row of `mask`, optionally under degraded sensing.

    Returns (actions_phys, gate, benefit, subframe).
    """
    cols = state_columns_for(experiment)
    scaler = scaler_for(experiment)
    sub = transitions.loc[mask]
    raw = sub[cols].to_numpy(dtype=np.float64)

    if iot is not None and iot.name != "clean":
        raw = raw.copy()
        raw[:, IOT_DEGRADABLE_SLICE] = IoTDegradationEngine(iot).apply(
            raw[:, IOT_DEGRADABLE_SLICE], sub["traj_id"].to_numpy(), IOT_CHANNEL_SCALE)

    x = torch.from_numpy(scaler.transform(raw).astype(np.float32)).to(DEVICE)
    agent.actor.eval()
    outs, gates, bens = [], [], []
    CH = 8192                                     # chunked so a large split cannot exhaust GPU memory
    for i in range(0, len(x), CH):
        a, _, g, b = agent.actor(x[i:i + CH])
        outs.append(a.detach().cpu().numpy())
        gates.append(g.detach().cpu().numpy())
        bens.append(torch.sigmoid(b).detach().cpu().numpy())
    a_norm = np.concatenate(outs, axis=0)
    return (norm_to_action(a_norm), np.concatenate(gates, axis=0),
            np.concatenate(bens, axis=0), sub)


def evaluate_actions_on_split(agent: FowtAriseAgent, experiment: str, mask: np.ndarray,
                              iot: Optional[IoTConfig] = None, split_name: str = "") -> dict:
    """Full metric set for one policy on one split under one observation regime."""
    acts, gate, benefit, sub = policy_actions_for_split(agent, experiment, mask, iot)
    m = score_actions(acts, sub)
    a_norm = action_to_norm(acts)
    m.update({
        "experiment": experiment,
        "split": split_name,
        "iot_mode": "clean" if iot is None else iot.name,
        "n_transitions": int(len(acts)),
        "mean_pitch_action": float(np.mean(acts[:, 0])),
        "mean_yaw_action": float(np.mean(acts[:, 1])),
        "mean_ipc_action": float(np.mean(acts[:, 2])),
        "mean_abs_pitch_action": float(np.mean(np.abs(acts[:, 0]))),
        "mean_abs_ipc_action": float(np.mean(np.abs(acts[:, 2]))),
        "std_pitch_action": float(np.std(acts[:, 0])),
        "std_yaw_action": float(np.std(acts[:, 1])),
        "std_ipc_action": float(np.std(acts[:, 2])),
        "action_smoothness_mean": float(np.mean(action_smoothness(a_norm, sub["traj_id"].to_numpy()))),
        "gate_mean": float(np.mean(gate)),
        "benefit_mean": float(np.mean(benefit)),
        "pct_of_oracle": float("nan"),
    })
    m["_actions_phys"] = acts
    m["_gate"] = gate
    m["_benefit"] = benefit
    m["_subframe_index"] = sub.index.to_numpy()
    return m


def action_saturation_rates(acts: np.ndarray, tol: float = 0.02) -> dict:
    """Fraction of predictions within `tol` of either actuator bound (pushed to its physical limit)."""
    near_lo = acts <= (ACTION_LO[None, :] + tol * ACTION_RANGE[None, :])
    near_hi = acts >= (ACTION_HI[None, :] - tol * ACTION_RANGE[None, :])
    sat = (near_lo | near_hi).mean(axis=0)
    return {"pitch_saturation_rate": float(sat[0]), "yaw_saturation_rate": float(sat[1]),
            "ipc_saturation_rate": float(sat[2])}


MONITORED_METRIC = "matched_sweep_objective"
MONITORED_METRIC_LABEL = "validation Mean Matched Sweep Objective (counterfactual, higher is better)"
print(f"    Monitored metric for model selection / early stopping / LR plateau:\n      {MONITORED_METRIC_LABEL}")

## 20. Checkpointing &nbsp;·&nbsp; 21. Resume Support

A checkpoint carries everything needed for a bit-faithful continuation: actor, critic, both target
networks, both optimisers, both schedulers, the epoch, the best validation metric, the early-stopping
counter, the full history so far, the feature scaler, the architecture flags, the configuration, and
the Python / NumPy / Torch RNG states.

**One non-obvious failure this handles.** `torch.load(..., map_location=...)` remaps *every* tensor in
the payload — including the saved RNG-state buffers. But `torch.set_rng_state` requires a **CPU
`ByteTensor`** specifically (an API requirement, not a device preference), so on a GPU runtime the
loaded RNG tensor arrives as a CUDA tensor and the call raises
`TypeError: RNG state must be a torch.ByteTensor`. The loader coerces those two buffers back to
CPU/uint8 unconditionally.

Checkpoint filenames are deterministic (`latest.pt`, `best.pt`) so re-running is idempotent rather
than accumulating timestamped copies. A checkpoint whose architecture does not match the experiment
raises rather than silently loading a mismatched policy.

In [ ]:
def save_checkpoint(path: Path, *, epoch: int, agent: FowtAriseAgent, scaler: FeatureScaler,
                    best_metric: float, best_epoch: int, patience: int, history: list,
                    experiment: str, config: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "format_version": 2,
        "experiment": experiment,
        "epoch": epoch,
        "agent": agent.state_dict(),
        "scaler": scaler.to_dict(),
        "best_metric": float(best_metric),
        "best_epoch": int(best_epoch),
        "patience": int(patience),
        "history": history,
        "config": config,
        "arch_flags": ARCH_FLAGS[experiment],
        "monitored_metric": MONITORED_METRIC,
        "param_counts": PARAM_COUNTS.get(experiment),
        "rng_python": random.getstate(),
        "rng_numpy": np.random.get_state(),
        "rng_torch": torch.get_rng_state(),
        "rng_torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    tmp.replace(path)            # atomic: a crash mid-write cannot leave a truncated checkpoint


def _as_cpu_bytetensor(t):
    """torch.set_rng_state* require a CPU uint8 tensor; map_location may have moved it to CUDA."""
    if isinstance(t, torch.Tensor) and (t.device.type != "cpu" or t.dtype != torch.uint8):
        return t.to(device="cpu", dtype=torch.uint8)
    return t


def load_checkpoint(path: Path, agent: FowtAriseAgent, experiment: str,
                    restore_rng: bool = True) -> dict:
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f"[FOWT-ARISE] checkpoint not found: {path}")
    payload = torch.load(path, map_location=DEVICE, weights_only=False)

    ckpt_exp = payload.get("experiment")
    if ckpt_exp is not None and ckpt_exp != experiment:
        raise RuntimeError(
            f"[FOWT-ARISE] checkpoint at {path} was written for experiment '{ckpt_exp}' but is being "
            f"loaded into '{experiment}'. Refusing: this would silently mix experiments.")
    ckpt_flags = payload.get("arch_flags")
    if ckpt_flags is not None and ckpt_flags != ARCH_FLAGS[experiment]:
        raise RuntimeError(
            f"[FOWT-ARISE] checkpoint architecture flags {ckpt_flags} do not match "
            f"{ARCH_FLAGS[experiment]} for '{experiment}'.")
    try:
        agent.load_state_dict(payload["agent"])
    except Exception as e:
        raise RuntimeError(
            f"[FOWT-ARISE] checkpoint at {path} is incompatible with the current architecture. This "
            f"usually means the model code changed since it was written. Original error: {e}")

    if restore_rng:
        try:
            random.setstate(payload["rng_python"])
            np.random.set_state(payload["rng_numpy"])
            torch.set_rng_state(_as_cpu_bytetensor(payload["rng_torch"]))
            if payload.get("rng_torch_cuda") is not None and torch.cuda.is_available():
                torch.cuda.set_rng_state_all([_as_cpu_bytetensor(t)
                                              for t in payload["rng_torch_cuda"]])
        except Exception as e:
            warnings.warn(f"[FOWT-ARISE] could not restore RNG state from {path} ({e}); training "
                          f"continues but is not bit-identical to an uninterrupted run.")
    return payload


def resolve_resume_path(experiment: str) -> Optional[Path]:
    """RESUME_FROM_CHECKPOINT applies to FOWT_ARISE; every experiment auto-resumes its own latest.pt.

    An explicitly supplied path that does not exist is a hard error -- silently starting a fresh run
    would quietly discard the experiment the user meant to continue.
    """
    if experiment == "FOWT_ARISE" and RESUME_FROM_CHECKPOINT:
        p = Path(RESUME_FROM_CHECKPOINT)
        if not p.exists():
            raise RuntimeError(
                f"[FOWT-ARISE] RESUME_FROM_CHECKPOINT='{RESUME_FROM_CHECKPOINT}' does not exist. "
                f"Set it to a real checkpoint or to \"\" for a fresh run. Refusing to silently "
                f"start a new experiment under the same output directory.")
        return p
    latest = EXP_DIRS[experiment]["checkpoints"] / "latest.pt"
    return latest if latest.exists() else None


print("    save_checkpoint / load_checkpoint / resolve_resume_path defined.")
print(f"    CHECKPOINT_EVERY = {CHECKPOINT_EVERY} epochs (best.pt written whenever validation improves)")

## 22. Early Stopping &nbsp;·&nbsp; 23. Learning-Rate on Plateau

Both watch the **same** monitored metric — the validation Mean Matched Sweep Objective — so the
learning rate and the stopping decision never disagree about what "better" means.

* **Early stopping**: stop after `EARLY_STOPPING_PATIENCE` epochs without an improvement exceeding
  `MIN_DELTA`; print the reason, then restore `best.pt` so the reported model is the selected one
  rather than whatever the last epoch happened to leave behind.
* **LR plateau**: halve the LR after `LR_PLATEAU_PATIENCE` stagnant epochs, floored at `MIN_LR`.
  Every reduction is logged, and the LR is recorded every epoch in `history.csv`.

In [ ]:
class EarlyStopping:
    """max-mode early stopping on the monitored metric."""

    def __init__(self, patience: int = EARLY_STOPPING_PATIENCE, min_delta: float = MIN_DELTA):
        self.patience, self.min_delta = patience, min_delta
        self.best = -float("inf")
        self.best_epoch = -1
        self.counter = 0
        self.should_stop = False
        self.reason = ""

    def step(self, value: float, epoch: int) -> bool:
        """Returns True if this epoch is a new best."""
        if not np.isfinite(value):
            self.counter += 1
            warnings.warn(f"[FOWT-ARISE] epoch {epoch}: monitored metric is {value}; counted as "
                          f"no-improvement rather than accepted as a best score.")
        elif value > self.best + self.min_delta:
            self.best, self.best_epoch, self.counter = float(value), int(epoch), 0
            return True
        else:
            self.counter += 1
        if self.counter >= self.patience:
            self.should_stop = True
            self.reason = (f"no improvement > {self.min_delta:g} in the monitored metric for "
                           f"{self.counter} consecutive epochs (patience={self.patience}); "
                           f"best={self.best:+.6f} at epoch {self.best_epoch}")
        return False

    def load_state(self, best: float, best_epoch: int, counter: int) -> None:
        self.best, self.best_epoch, self.counter = float(best), int(best_epoch), int(counter)


print("    EarlyStopping defined.")
print(f"    patience={EARLY_STOPPING_PATIENCE}  min_delta={MIN_DELTA:g}")
print(f"    LR plateau: factor={LR_PLATEAU_FACTOR} patience={LR_PLATEAU_PATIENCE} min_lr={MIN_LR:g}")
print(f"    both driven by: {MONITORED_METRIC_LABEL}")

## 24. FOWT-ARISE Training

One generic `run_experiment(name)` trains the proposed model **and every ablation**. Nothing about it
branches on "is this the proposed model"; the only things that vary are read from
`ARCH_FLAGS[name]` (which novelty is architecturally removed), `training_reward_for` (which reward),
and `iot_training_mode_for` (whether N4 is active). That is what makes §31's fair-ablation
requirement structural rather than a promise: dataset, split, seed, batch size, optimiser, budget,
action limits, normalisation, matcher and metrics are literally the same code.

### Epoch-wise logging (§27)

Every epoch appends one row to `history.csv` with the full metric set. Two of those need their
definition stated rather than assumed:

* `train_loss` — `actor_loss + critic_loss`, the total optimised objective. It is a *training*
  quantity and is deliberately **not** used for model selection.
* `validation_loss` — imitation loss of the policy against the validation best-action targets. A
  held-out fit measure, still not the selection metric.

`IoT_clean_performance` / `IoT_degraded_performance` / `IoT_performance_gap` are recomputed on the
**validation** split every epoch, so N4's effect is visible during training and not only at the end.

Where a metric is not mathematically meaningful for a configuration it is written as `NaN` and
labelled, never as `0`. Ablation N3's power / duty / smoothness components are the clear case: they
are absent from that ablation's objective, and reporting `0` would assert it achieved zero power loss
rather than that it did not account for power at all.

On resume, history is **loaded and appended to** — never truncated or overwritten.

In [ ]:
def iot_training_mode_for(experiment: str) -> str:
    """N4 removed => train on clean observations only. Every other experiment trains mixed."""
    return "clean" if experiment == "ABLATION_N4" else "mixed"


def agent_config_for(experiment: str) -> AgentConfig:
    """Ablation N4 additionally switches off the clean-vs-degraded consistency term."""
    cfg = AgentConfig()
    if experiment == "ABLATION_N4":
        cfg.lambda_robust = 0.0
    if not ARCH_FLAGS[experiment]["use_gate"]:
        cfg.lambda_gate_aux = 0.0        # no gate to ground
    return cfg


def experiment_config_dict(experiment: str) -> dict:
    """The exact configuration this experiment ran under; written to its config.json."""
    ac = agent_config_for(experiment)
    return {
        "experiment": experiment,
        "seed": SEED,
        "dry_run": DRY_RUN,
        "paths": {"dataset": str(DATASET_PATH_), "action_sweep": str(ACTION_SWEEP_PATH_),
                  "output_root": str(OUTPUT_ROOT_), "baseline_dir": BASELINE_DIR,
                  "resume_from_checkpoint": RESUME_FROM_CHECKPOINT},
        "split": {"train_frac": TRAIN_FRAC, "val_frac": VAL_FRAC, "test_frac": TEST_FRAC,
                  "trajectory_key": "tower + '__' + episode_id"},
        "architecture": {**ARCH_FLAGS[experiment], "latent_dim": LATENT_DIM,
                         "actor_output_scale": ACTOR_OUTPUT_SCALE,
                         "state_dim": len(state_columns_for(experiment)),
                         "action_dim": ACTION_DIM},
        "hyperparameters": {
            "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS,
            "learning_rate_critic": LEARNING_RATE, "learning_rate_actor": LEARNING_RATE_ACTOR,
            "gamma": ac.gamma, "tau": ac.tau, "policy_noise": ac.policy_noise,
            "noise_clip": ac.noise_clip, "policy_delay": ac.policy_delay,
            "grad_clip_norm": ac.grad_clip, "cql_alpha": ac.cql_alpha,
            "weight_decay": WEIGHT_DECAY,
            "imitation_huber_delta": IMITATION_HUBER_DELTA,
            "q_improvement_coef": ac.q_improvement_coef,
            "best_action_group_cols": BEST_ACTION_GROUP_COLS,
        },
        "reward": {"preset": REWARD_WEIGHT_PRESET, "lambda_fatigue": LAMBDA_FATIGUE,
                   "lambda_power": LAMBDA_POWER, "lambda_actuation": LAMBDA_ACTUATION,
                   "lambda_smoothness": LAMBDA_SMOOTHNESS,
                   "reduced_to_load_relief_only": experiment == "ABLATION_N3",
                   "rated_power_w": RATED_POWER_W, "wohler_m": WOHLER_M},
        "iot": {"training_mode": iot_training_mode_for(experiment),
                "noise_std": IOT_NOISE_STD, "dropout_prob": IOT_DROPOUT_PROB,
                "bias_magnitude": IOT_BIAS_MAGNITUDE, "stale_prob": IOT_STALE_PROB,
                "train_degraded_fraction": (0.0 if experiment == "ABLATION_N4"
                                            else TRAIN_DEGRADED_FRACTION),
                "lambda_robust": ac.lambda_robust},
        "gate": {"lambda_gate_aux": ac.lambda_gate_aux},
        "early_stopping": {"patience": EARLY_STOPPING_PATIENCE, "min_delta": MIN_DELTA,
                           "monitored_metric": MONITORED_METRIC},
        "lr_scheduler": {"factor": LR_PLATEAU_FACTOR, "patience": LR_PLATEAU_PATIENCE,
                         "min_lr": MIN_LR},
        "checkpointing": {"every": CHECKPOINT_EVERY},
        "shap": {"background_size": SHAP_BACKGROUND_SIZE, "test_sample_size": SHAP_TEST_SAMPLE_SIZE,
                 "nsamples": SHAP_NSAMPLES},
        "environment": ENVIRONMENT_INFO,
    }


HISTORY_COLUMNS = [
    "epoch", "train_loss", "validation_loss", "actor_loss", "critic_loss", "td_loss",
    "cql_penalty", "robustness_loss", "gate_aux_loss",
    "fatigue_reward", "power_penalty", "actuation_penalty", "smoothness_penalty", "mean_reward",
    "mean_action_magnitude", "mean_yaw_action", "no_action_rate",
    "mean_DEL_ratio", "fatigue_relief_percent", "power_loss_percent", "actuator_duty",
    "matched_sweep_objective", "pct_of_oracle", "coverage_pct",
    "IoT_clean_performance", "IoT_degraded_performance", "IoT_performance_gap",
    "gate_mean", "benefit_mean", "q1_mean", "q2_mean", "target_q_mean",
    "actor_grad_norm", "critic_grad_norm", "learning_rate", "learning_rate_critic",
    "iot_degraded_batch_fraction", "epoch_seconds",
]
print(f"    history.csv columns ({len(HISTORY_COLUMNS)}): {HISTORY_COLUMNS}")

In [ ]:
def run_experiment(experiment: str) -> dict:
    """Train one experiment end to end. Generic: used unchanged for FOWT-ARISE and all ablations."""
    print("=" * 79)
    print(f"TRAINING: {experiment}")
    print("=" * 79)
    t_start = time.time()
    set_global_seed(SEED)                       # identical starting conditions for every experiment

    dirs = EXP_DIRS[experiment]
    cfg_dict = experiment_config_dict(experiment)
    with open(dirs["base"] / "config.json", "w") as f:
        json.dump(cfg_dict, f, indent=2, default=str)

    iot_mode = iot_training_mode_for(experiment)
    acfg = agent_config_for(experiment)
    print(f"    architecture      : {ARCH_FLAGS[experiment]}")
    print(f"    reward            : "
          f"{'load-relief only (N3 ablated)' if experiment == 'ABLATION_N3' else 'N3 multi-objective'}")
    print(f"    IoT training mode : {iot_mode} "
          f"(degraded fraction {cfg_dict['iot']['train_degraded_fraction']}, "
          f"lambda_robust {acfg.lambda_robust})")
    print(f"    state dim         : {len(state_columns_for(experiment))}")

    # ---- data ----
    reward_arr, components = training_reward_for(experiment, transitions)
    targets_full = build_best_action_targets(experiment)
    train_buf = build_buffer(experiment, train_mask, reward_arr, components, targets_full,
                             seed=SEED + 100)
    val_targets = targets_full
    val_buf = build_buffer(experiment, val_mask, reward_arr, components, val_targets, seed=SEED + 200)
    print(f"    train buffer      : {len(train_buf):,} transitions")
    print(f"    val buffer        : {len(val_buf):,} transitions")

    for nm, arr in (("state", train_buf.raw_state), ("action", train_buf.action),
                    ("reward", train_buf.reward)):
        if not np.isfinite(arr).all():
            raise RuntimeError(f"[FOWT-ARISE] non-finite {nm} in the {experiment} train buffer.")
    if not (train_buf.action.min() >= -1 - 1e-4 and train_buf.action.max() <= 1 + 1e-4):
        raise RuntimeError("[FOWT-ARISE] normalised train actions escaped [-1, 1].")

    scaler = scaler_for(experiment)
    actor, critic = build_actor_critic(experiment)
    agent = FowtAriseAgent(actor, critic, acfg, DEVICE)
    print(f"    trainable params  : actor={count_parameters(agent.actor):,} "
          f"critic={count_parameters(agent.critic):,} "
          f"total={count_parameters(agent.actor) + count_parameters(agent.critic):,}")

    degrade_engine = None if iot_mode == "clean" else IoTDegradationEngine(IOT_DEGRADED)
    degraded_fraction = 0.0 if iot_mode == "clean" else TRAIN_DEGRADED_FRACTION

    # ---- resume ----
    history: list = []
    start_epoch = 1
    stopper = EarlyStopping()
    best_ckpt = dirs["checkpoints"] / "best.pt"
    latest_ckpt = dirs["checkpoints"] / "latest.pt"
    resume_path = resolve_resume_path(experiment)
    if resume_path is not None:
        print(f"\n[RESUME] Resuming from checkpoint: {resume_path}")
        payload = load_checkpoint(resume_path, agent, experiment)
        start_epoch = int(payload["epoch"]) + 1
        history = list(payload.get("history", []))
        stopper.load_state(payload.get("best_metric", -float("inf")),
                           payload.get("best_epoch", -1), payload.get("patience", 0))
        scaler = FeatureScaler.from_dict(payload["scaler"])
        print(f"[RESUME] Starting epoch: {start_epoch}")
        print(f"[RESUME] best {MONITORED_METRIC} so far = {stopper.best:+.6f} "
              f"(epoch {stopper.best_epoch}), patience counter = {stopper.counter}, "
              f"{len(history)} history rows preserved")
    else:
        print("\n    no checkpoint found; starting from epoch 1")

    n_batches = max(1, len(train_buf) // BATCH_SIZE)
    stopped_early = False

    if start_epoch > NUM_EPOCHS:
        print(f"    start_epoch ({start_epoch}) > NUM_EPOCHS ({NUM_EPOCHS}); nothing left to train.")
    else:
        for epoch in range(start_epoch, NUM_EPOCHS + 1):
            t0 = time.time()
            agent.actor.train(); agent.critic.train()
            run = RunningMean()
            n_degraded = 0
            sampled = []
            rng_epoch = np.random.default_rng(SEED + 10_000 + epoch)

            for b in range(n_batches):
                idx = train_buf.sample_indices(BATCH_SIZE)
                sampled.append(idx)
                use_deg = degrade_engine is not None and rng_epoch.random() < degraded_fraction
                n_degraded += int(use_deg)
                batch = prepare_batch(train_buf, idx, scaler,
                                      degrade_engine if use_deg else None,
                                      batch_offset=epoch * 10_000 + b)
                run.add(agent.update(batch))

            tm = run.means()
            all_idx = np.concatenate(sampled)

            def _comp(name: str) -> float:
                v = train_buf.components.get(name)
                if v is None:
                    return float("nan")
                s = v[all_idx]
                return float("nan") if np.all(np.isnan(s)) else float(np.nanmean(s))

            # ---- validation: clean and degraded, every epoch ----
            agent.actor.eval(); agent.critic.eval()
            val_clean = evaluate_actions_on_split(agent, experiment, val_mask, None, "val")
            val_deg = evaluate_actions_on_split(agent, experiment, val_mask, IOT_DEGRADED, "val")
            monitored = val_clean[MONITORED_METRIC]

            with torch.no_grad():
                vb = prepare_batch(val_buf, np.arange(len(val_buf)), scaler, None)
                vpa, _, _, _ = agent.actor(vb["state"])
                val_loss = float(F.mse_loss(vpa, vb["target_action"]).item()
                                 if vb["target_action"] is not None
                                 else F.mse_loss(vpa, vb["action"]).item())
                del vb, vpa

            lr_a = agent.actor_opt.param_groups[0]["lr"]
            lr_c = agent.critic_opt.param_groups[0]["lr"]
            row = {
                "epoch": epoch,
                "train_loss": float(np.nansum([tm.get("actor_loss", np.nan),
                                               tm.get("critic_loss", np.nan)])),
                "validation_loss": val_loss,
                "actor_loss": tm.get("actor_loss", float("nan")),
                "critic_loss": tm.get("critic_loss", float("nan")),
                "td_loss": tm.get("td_loss", float("nan")),
                "cql_penalty": tm.get("cql_penalty", float("nan")),
                "robustness_loss": tm.get("robustness_loss", float("nan")),
                "gate_aux_loss": tm.get("gate_aux_loss", float("nan")),
                "fatigue_reward": _comp("fatigue_term"),
                "power_penalty": _comp("power_penalty"),
                "actuation_penalty": _comp("actuation_penalty"),
                "smoothness_penalty": _comp("smoothness_penalty"),
                "mean_reward": float(np.mean(train_buf.reward[all_idx])),
                "mean_action_magnitude": val_clean["mean_action_magnitude"],
                "mean_yaw_action": val_clean["mean_abs_yaw_action"],
                "no_action_rate": val_clean["no_action_rate"],
                "mean_DEL_ratio": val_clean["mean_del_ratio"],
                "fatigue_relief_percent": val_clean["fatigue_relief_pct"],
                "power_loss_percent": val_clean["power_loss_pct"],
                "actuator_duty": val_clean["actuator_duty_proxy"],
                "matched_sweep_objective": monitored,
                "pct_of_oracle": 100.0 * monitored / ORACLE_VAL,
                "coverage_pct": val_clean["coverage_pct"],
                "IoT_clean_performance": val_clean[MONITORED_METRIC],
                "IoT_degraded_performance": val_deg[MONITORED_METRIC],
                "IoT_performance_gap": val_clean[MONITORED_METRIC] - val_deg[MONITORED_METRIC],
                "gate_mean": val_clean["gate_mean"],
                "benefit_mean": val_clean["benefit_mean"],
                "q1_mean": tm.get("q1_mean", float("nan")),
                "q2_mean": tm.get("q2_mean", float("nan")),
                "target_q_mean": tm.get("target_q_mean", float("nan")),
                "actor_grad_norm": tm.get("actor_grad_norm", float("nan")),
                "critic_grad_norm": tm.get("critic_grad_norm", float("nan")),
                "learning_rate": lr_a,
                "learning_rate_critic": lr_c,
                "iot_degraded_batch_fraction": n_degraded / n_batches,
                "epoch_seconds": time.time() - t0,
            }
            history.append(row)

            print(f"[{experiment}] ep {epoch:3d}/{NUM_EPOCHS} | "
                  f"VALOBJ={row['matched_sweep_objective']:+.5f} "
                  f"({row['pct_of_oracle']:5.1f}% oracle) | "
                  f"DEL={row['mean_DEL_ratio']:.5f} fat={row['fatigue_relief_percent']:5.2f}% "
                  f"pow={row['power_loss_percent']:5.2f}% | "
                  f"actor_L={row['actor_loss']:.4f} critic_L={row['critic_loss']:.4f} "
                  f"rob_L={row['robustness_loss']:.4f} cql={row['cql_penalty']:+.3f} | "
                  f"|a|={row['mean_action_magnitude']:.3f} noact={row['no_action_rate']:.3f} "
                  f"gate={row['gate_mean']:.3f} | "
                  f"IoTgap={row['IoT_performance_gap']:+.5f} | "
                  f"lr={row['learning_rate']:.2e} | {row['epoch_seconds']:.1f}s")

            # ---- LR plateau (both schedulers watch the monitored metric) ----
            prev_lr = lr_a
            agent.actor_sched.step(monitored)
            agent.critic_sched.step(monitored)
            new_lr = agent.actor_opt.param_groups[0]["lr"]
            if new_lr < prev_lr - 1e-15:
                print(f"    [LR] actor learning rate reduced {prev_lr:.3e} -> {new_lr:.3e} "
                      f"(monitored metric plateaued)")

            # ---- checkpointing / early stopping ----
            improved = stopper.step(monitored, epoch)
            _ck = dict(agent=agent, scaler=scaler, best_metric=stopper.best,
                       best_epoch=stopper.best_epoch, patience=stopper.counter,
                       history=history, experiment=experiment, config=cfg_dict)
            if improved:
                save_checkpoint(best_ckpt, epoch=epoch, **_ck)
                print(f"    [BEST] new best {MONITORED_METRIC} = {monitored:+.6f} -> best.pt")
            if epoch % CHECKPOINT_EVERY == 0 or epoch == NUM_EPOCHS or stopper.should_stop:
                save_checkpoint(latest_ckpt, epoch=epoch, **_ck)

            pd.DataFrame(history)[HISTORY_COLUMNS].to_csv(dirs["base"] / "history.csv", index=False)

            if stopper.should_stop:
                print(f"\n    [EARLY STOPPING] {stopper.reason}")
                stopped_early = True
                break

    # ---- restore the SELECTED model (best validation), not the last epoch ----
    if best_ckpt.exists():
        load_checkpoint(best_ckpt, agent, experiment, restore_rng=False)
        print(f"\n    restored best.pt (epoch {stopper.best_epoch}, "
              f"{MONITORED_METRIC}={stopper.best:+.6f}) for evaluation")
    else:
        warnings.warn(f"[FOWT-ARISE] no best.pt for {experiment}; evaluating the final-epoch weights. "
                      f"This happens only if validation never improved even once.")

    pd.DataFrame(history)[HISTORY_COLUMNS].to_csv(dirs["base"] / "history.csv", index=False)
    summary = {
        "experiment": experiment,
        "trainable_parameters_actor": count_parameters(agent.actor),
        "trainable_parameters_critic": count_parameters(agent.critic),
        "trainable_parameters_total": count_parameters(agent.actor) + count_parameters(agent.critic),
        "epochs_run": len(history),
        "epochs_requested": NUM_EPOCHS,
        "stopped_early": stopped_early,
        "early_stopping_reason": stopper.reason,
        "best_epoch": stopper.best_epoch,
        "best_validation_metric": stopper.best,
        "monitored_metric": MONITORED_METRIC,
        "monitored_metric_label": MONITORED_METRIC_LABEL,
        "wall_clock_seconds": time.time() - t_start,
        "resumed_from": str(resume_path) if resume_path else None,
        "iot_training_mode": iot_mode,
        "dry_run": DRY_RUN,
    }
    with open(dirs["base"] / "training_summary.json", "w") as f:
        json.dump(summary, f, indent=2, default=str)
    print(f"    Saved history.csv ({len(history)} rows), config.json, training_summary.json")
    print(f"    {experiment} trainable parameters: {summary['trainable_parameters_total']:,}")

    del train_buf, val_buf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"agent": agent, "scaler": scaler, "history": history, "summary": summary,
            "stopper": stopper, "dirs": dirs}


print("    run_experiment() defined -- the single training entry point for all five experiments.")

In [ ]:
assert_no_test_leakage("pre-training")
TRAINED = {}
TRAINED["FOWT_ARISE"] = run_experiment("FOWT_ARISE")
assert_no_test_leakage("post-training-FOWT_ARISE")

## 25. FOWT-ARISE Validation

Two things happen here.

**A validation report** on the selected checkpoint, under clean and degraded observations, placed
against the reference band from Section 18b so the number is interpretable rather than free-floating.

**A measurement of whether the critic can rank actions at all.** This is the evidence behind
`Q_IMPROVEMENT_COEF = 0.0`, and it is measured rather than asserted. For a sample of validation
conditions the trained critic scores **all** sweep actions available at that condition, and its
ranking is compared with the sweep's own derived objective for the same actions:

* **Spearman ρ** between $Q(s,a)$ and the true counterfactual objective across the action grid
* **top-1 agreement** — how often $\arg\max_a Q$ is the genuinely best action
* **regret** — objective lost by following $\arg\max_a Q$ instead of the true best action

If ρ is near zero and top-1 agreement is near chance, then $\partial Q/\partial a$ carries no usable
signal about *which* action to take, and driving the actor with it would inject noise rather than
improvement. Reporting these three numbers lets a reader disagree with the default from evidence.
Everything is computed on **validation** trajectories; the test split is untouched.

In [ ]:
def _spearman(a: np.ndarray, b: np.ndarray) -> float:
    """Spearman rho via ranks; scipy when available, otherwise an equivalent manual computation."""
    try:
        from scipy.stats import spearmanr
        r = spearmanr(a, b).statistic
        return float(r) if np.isfinite(r) else float("nan")
    except Exception:
        ra = pd.Series(a).rank().to_numpy()
        rb = pd.Series(b).rank().to_numpy()
        if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
            return float("nan")
        return float(np.corrcoef(ra, rb)[0, 1])


@torch.no_grad()
def measure_critic_action_ranking(agent: FowtAriseAgent, experiment: str,
                                  n_conditions: int = 300) -> dict:
    """Can the critic tell WHICH action is better at a fixed operating condition?"""
    cols, scaler = state_columns_for(experiment), scaler_for(experiment)
    sub = transitions.loc[val_mask]
    rng = np.random.default_rng(SEED + 777)
    pick = rng.choice(len(sub), size=min(n_conditions, len(sub)), replace=False)
    rows = sub.iloc[pick]

    agent.critic.eval()
    rhos, top1, regrets, ranges = [], [], [], []
    for _, r in rows.iterrows():
        ix = SWEEP_INDEX.get(r[TR["tower"]])
        if ix is None:
            continue
        blk = ix["sim_to_block"].get(int(r[MATCH_KEY]), -1)
        if blk < 0:
            continue
        a_norm = ix["actions_norm"][blk]                       # (n_actions, 3)
        true_obj = ix["reward"][blk]                           # (n_actions,)
        s = scaler.transform(r[cols].to_numpy(dtype=np.float64)[None, :])
        s_t = torch.from_numpy(np.repeat(s, len(a_norm), axis=0).astype(np.float32)).to(DEVICE)
        a_t = torch.from_numpy(a_norm.astype(np.float32)).to(DEVICE)
        q = agent.critic.q_min(s_t, a_t).squeeze(-1).cpu().numpy()

        rhos.append(_spearman(q, true_obj))
        top1.append(float(int(np.argmax(q)) == int(np.argmax(true_obj))))
        regrets.append(float(true_obj.max() - true_obj[int(np.argmax(q))]))
        ranges.append(float(true_obj.max() - true_obj.min()))

    n = len(rhos)
    if n == 0:
        return {"n_conditions": 0}
    chance = 1.0 / float(N_SWEEP_ACTIONS)
    return {
        "n_conditions": n,
        "mean_spearman_rho": float(np.nanmean(rhos)),
        "top1_agreement": float(np.mean(top1)),
        "top1_chance_level": chance,
        "mean_regret": float(np.mean(regrets)),
        "mean_decision_range": float(np.mean(ranges)),
        "regret_as_fraction_of_range": float(np.mean(regrets) / max(np.mean(ranges), 1e-12)),
    }


print("=" * 79)
print("FOWT-ARISE VALIDATION (selected checkpoint)")
print("=" * 79)
_ag = TRAINED["FOWT_ARISE"]["agent"]
VAL_CLEAN_FA = evaluate_actions_on_split(_ag, "FOWT_ARISE", val_mask, None, "val")
VAL_DEG_FA = evaluate_actions_on_split(_ag, "FOWT_ARISE", val_mask, IOT_DEGRADED, "val")
VAL_CLEAN_FA["pct_of_oracle"] = 100.0 * VAL_CLEAN_FA[MONITORED_METRIC] / ORACLE_VAL

print(f"    Mean Matched Sweep Objective (clean)    : {VAL_CLEAN_FA[MONITORED_METRIC]:+.6f}"
      f"   ({VAL_CLEAN_FA['pct_of_oracle']:.1f}% of oracle)")
print(f"    Mean Matched Sweep Objective (degraded) : {VAL_DEG_FA[MONITORED_METRIC]:+.6f}")
print(f"    IoT performance gap                     : "
      f"{VAL_CLEAN_FA[MONITORED_METRIC] - VAL_DEG_FA[MONITORED_METRIC]:+.6f}")
print(f"    Mean / Median DEL ratio                 : {VAL_CLEAN_FA['mean_del_ratio']:.6f} / "
      f"{VAL_CLEAN_FA['median_del_ratio']:.6f}")
print(f"    Fatigue relief %                        : {VAL_CLEAN_FA['fatigue_relief_pct']:.3f}")
print(f"    Power loss %                            : {VAL_CLEAN_FA['power_loss_pct']:.3f}")
print(f"    No-action rate                          : {VAL_CLEAN_FA['no_action_rate']:.4f}")
print(f"    Action-sweep coverage %                 : {VAL_CLEAN_FA['coverage_pct']:.2f}")
print(f"    Mean gate value / benefit               : {VAL_CLEAN_FA['gate_mean']:.4f} / "
      f"{VAL_CLEAN_FA['benefit_mean']:.4f}")

print("\n    reference band on the same validation trajectories:")
for _n in ("do_nothing", "ipc_half", "ipc_only", "behaviour_logged", "ORACLE_best_of_sweep"):
    _s = VAL_REFERENCES[_n]
    print(f"        {_n:22s} {_s['matched_sweep_objective']:+.6f}  "
          f"({100.0 * _s['matched_sweep_objective'] / ORACLE_VAL:6.1f}% of oracle)")

print("\n" + "-" * 79)
print("CRITIC ACTION-RANKING MEASUREMENT (the evidence behind Q_IMPROVEMENT_COEF)")
print("-" * 79)
CRITIC_RANKING = measure_critic_action_ranking(_ag, "FOWT_ARISE")
if CRITIC_RANKING.get("n_conditions", 0) == 0:
    print("    could not sample any validation condition present in the sweep; no measurement made.")
else:
    print(f"    conditions sampled            : {CRITIC_RANKING['n_conditions']}")
    print(f"    mean Spearman rho(Q, true obj): {CRITIC_RANKING['mean_spearman_rho']:+.4f}")
    print(f"    top-1 agreement               : {CRITIC_RANKING['top1_agreement']:.4f}   "
          f"(chance = {CRITIC_RANKING['top1_chance_level']:.4f})")
    print(f"    mean regret of argmax_a Q     : {CRITIC_RANKING['mean_regret']:.6f}")
    print(f"    mean decision range           : {CRITIC_RANKING['mean_decision_range']:.6f}")
    print(f"    regret / decision range       : "
          f"{CRITIC_RANKING['regret_as_fraction_of_range']:.3f}")
    _rho = CRITIC_RANKING["mean_spearman_rho"]
    if abs(_rho) < 0.2 or CRITIC_RANKING["regret_as_fraction_of_range"] > 0.5:
        print("\n    => The critic does NOT reliably rank actions within a condition. dQ/da therefore")
        print("       carries little information about WHICH action to take, which is why")
        print(f"       Q_IMPROVEMENT_COEF defaults to {Q_IMPROVEMENT_COEF} and the actor is driven by")
        print("       best-action imitation instead. The critics are still trained, conservatively")
        print("       regularised and logged; they are simply not trusted to choose the action.")
    else:
        print("\n    => The critic ranks actions informatively on this run. Raising Q_IMPROVEMENT_COEF")
        print("       above 0 is worth testing on VALIDATION (never on test).")
with open(EXP_DIRS["FOWT_ARISE"]["base"] / "critic_ranking_diagnostic.json", "w") as f:
    json.dump(CRITIC_RANKING, f, indent=2)

## 26. FOWT-ARISE Test Evaluation &nbsp;·&nbsp; 27. Counterfactual Action-Sweep Evaluation

The test split is evaluated **now, once**, on the checkpoint already selected by validation. Nothing
after this point feeds back into training, tuning, early stopping or checkpoint choice.

### What the matching does, and what it does not claim

The transitions record the outcome of the action the *behaviour policy* took, not the action our
policy would take. To estimate the latter, each test transition's predicted action is matched to the
**nearest action in the sweep grid at the identical physical condition** (same tower and `sim_id`),
in the TRAIN-derived normalised action space, and the sweep's physical outcome for that action is
read off.

This is a **counterfactual estimate**, and it is labelled as such everywhere:
**Mean Matched Sweep Objective**. It is never described as a measured transition reward.

Two honesty measures are reported alongside it:

* **Action-Sweep Coverage %** — the fraction of test rows whose condition was found in the sweep.
  Uncovered rows stay `NaN` and are excluded, never matched to an unrelated condition.
* **mean match distance** — how far, in normalised action units, the policy's action sat from the
  nearest grid point. A large distance means the reported outcome belongs to a noticeably different
  action than the one requested, which would weaken the estimate; it is surfaced rather than buried.

In [ ]:
def evaluate_experiment_on_test(experiment: str, agent: FowtAriseAgent) -> dict:
    """Single, final test evaluation: clean + degraded + per-mode robustness. Saves predictions."""
    assert_no_test_leakage(f"test-eval::{experiment}")
    dirs = EXP_DIRS[experiment]

    clean = evaluate_actions_on_split(agent, experiment, test_mask, None, "test")
    degraded = evaluate_actions_on_split(agent, experiment, test_mask, IOT_DEGRADED, "test")

    acts = clean["_actions_phys"]
    a_norm = action_to_norm(acts)
    sub = transitions.loc[test_mask]

    # per-transition predictions + the matched counterfactual outcome
    outcomes, covered, dist = match_actions_to_sweep(
        acts, sub[MATCH_KEY].to_numpy(), sub[TR["tower"]].to_numpy())
    preds = pd.DataFrame({
        "tower": sub[TR["tower"]].to_numpy(),
        "episode_id": sub[TR["episode_id"]].to_numpy(),
        "traj_id": sub["traj_id"].to_numpy(),
        "step": sub[TR["step"]].to_numpy(),
        MATCH_KEY: sub[MATCH_KEY].to_numpy(),
        "pred_pitch_offset_deg": acts[:, 0],
        "pred_yaw_setpoint_deg": acts[:, 1],
        "pred_ipc_level": acts[:, 2],
        "gate_pitch": clean["_gate"][:, 0],
        "gate_yaw": clean["_gate"][:, 1],
        "gate_ipc": clean["_gate"][:, 2],
        "benefit_prob": clean["_benefit"][:, 0],
        "no_action": no_action_indicator(a_norm),
        "behaviour_pitch_offset_deg": sub[TR["act_pitch"]].to_numpy(),
        "behaviour_yaw_setpoint_deg": sub[TR["act_yaw"]].to_numpy(),
        "behaviour_ipc_level": sub[TR["act_ipc"]].to_numpy(),
        "matched": covered,
        "match_distance_normalised": dist,
        "matched_sweep_objective": outcomes["reward"],
        "matched_del_ratio": outcomes["del_ratio"],
        "matched_fatigue_relief": outcomes["fatigue_relief"],
        "matched_power_loss_fraction": outcomes["power_loss_fraction"],
        "matched_damage_ratio_max": outcomes["damage_ratio_max"],
        "matched_controllable_share_max": outcomes["controllable_share_max"],
    })
    preds.to_csv(dirs["base"] / "test_predictions.csv", index=False)

    # structural sanity: 1:1 alignment and in-bounds actions
    if len(preds) != int(test_mask.sum()):
        raise RuntimeError(f"[FOWT-ARISE] {experiment}: {len(preds)} predictions vs "
                           f"{int(test_mask.sum())} test transitions.")
    for j, (c, lo, hi) in enumerate([("pred_pitch_offset_deg", ACTION_LO[0], ACTION_HI[0]),
                                     ("pred_yaw_setpoint_deg", ACTION_LO[1], ACTION_HI[1]),
                                     ("pred_ipc_level", ACTION_LO[2], ACTION_HI[2])]):
        v = preds[c].to_numpy()
        if not (np.all(v >= lo - 1e-3) and np.all(v <= hi + 1e-3)):
            raise RuntimeError(f"[FOWT-ARISE] {experiment}: '{c}' escaped [{lo}, {hi}].")

    # validation predictions too (§54 expects both files)
    v_acts, v_gate, v_ben, v_sub = policy_actions_for_split(agent, experiment, val_mask, None)
    pd.DataFrame({
        "tower": v_sub[TR["tower"]].to_numpy(),
        "traj_id": v_sub["traj_id"].to_numpy(),
        "step": v_sub[TR["step"]].to_numpy(),
        MATCH_KEY: v_sub[MATCH_KEY].to_numpy(),
        "pred_pitch_offset_deg": v_acts[:, 0],
        "pred_yaw_setpoint_deg": v_acts[:, 1],
        "pred_ipc_level": v_acts[:, 2],
        "benefit_prob": v_ben[:, 0],
        "no_action": no_action_indicator(action_to_norm(v_acts)),
    }).to_csv(dirs["base"] / "validation_predictions.csv", index=False)

    clean_perf, deg_perf = clean[MONITORED_METRIC], degraded[MONITORED_METRIC]
    gap = clean_perf - deg_perf
    metrics = {
        "Method": experiment,
        "Mean Matched Sweep Objective": clean_perf,
        "Mean DEL Ratio": clean["mean_del_ratio"],
        "Median DEL Ratio": clean["median_del_ratio"],
        "Fatigue Relief %": clean["fatigue_relief_pct"],
        "Power Loss %": clean["power_loss_pct"],
        "Actuator Duty Proxy": clean["actuator_duty_proxy"],
        "Mean Action Magnitude": clean["mean_action_magnitude"],
        "Mean |Yaw Action|": clean["mean_abs_yaw_action"],
        "No-Action Rate": clean["no_action_rate"],
        "Action-Sweep Coverage %": clean["coverage_pct"],
        "Clean Performance": clean_perf,
        "IoT-Degraded Performance": deg_perf,
        "IoT Performance Gap": gap,
        "Robustness Drop %": (100.0 * gap / abs(clean_perf)) if abs(clean_perf) > 1e-12 else float("nan"),
        "Parameter Count": PARAM_COUNTS[experiment]["total"],
        "Best Validation Epoch": TRAINED[experiment]["summary"]["best_epoch"],
        "pct_of_oracle": 100.0 * clean_perf / ORACLE_TEST if "ORACLE_TEST" in globals() else float("nan"),
        "mean_match_distance": clean["mean_match_distance"],
        "action_smoothness_mean": clean["action_smoothness_mean"],
        "gate_mean": clean["gate_mean"],
        "benefit_mean": clean["benefit_mean"],
        **action_saturation_rates(acts),
    }
    return {"metrics": metrics, "clean": clean, "degraded": degraded, "predictions": preds}


TEST_REFERENCES = reference_policy_scores(test_mask)
ORACLE_TEST = TEST_REFERENCES["ORACLE_best_of_sweep"]["matched_sweep_objective"]
print("=" * 79)
print("REFERENCE POLICIES on TEST trajectories")
print("=" * 79)
for _n, _s in TEST_REFERENCES.items():
    print(f"    {_n:22s} objective={_s['matched_sweep_objective']:+.6f}  "
          f"({100.0 * _s['matched_sweep_objective'] / ORACLE_TEST:6.1f}% of oracle)  "
          f"DEL={_s['mean_del_ratio']:.5f}  no_action={_s['no_action_rate']:.3f}")
print(f"\n    achievable band on TEST: "
      f"{TEST_REFERENCES['do_nothing']['matched_sweep_objective']:+.6f} -> {ORACLE_TEST:+.6f}")

In [ ]:
RESULTS = {}
RESULTS["FOWT_ARISE"] = evaluate_experiment_on_test("FOWT_ARISE", TRAINED["FOWT_ARISE"]["agent"])
_m = RESULTS["FOWT_ARISE"]["metrics"]

print("=" * 79)
print("FOWT-ARISE TEST EVALUATION (single pass, on the validation-selected checkpoint)")
print("=" * 79)
for _k in ("Mean Matched Sweep Objective", "Mean DEL Ratio", "Median DEL Ratio", "Fatigue Relief %",
           "Power Loss %", "Actuator Duty Proxy", "Mean Action Magnitude", "Mean |Yaw Action|",
           "No-Action Rate", "Action-Sweep Coverage %", "Clean Performance",
           "IoT-Degraded Performance", "IoT Performance Gap", "Robustness Drop %",
           "Parameter Count", "Best Validation Epoch"):
    _v = _m[_k]
    print(f"    {_k:32s} : {_v:,.6f}" if isinstance(_v, float) else f"    {_k:32s} : {_v}")
print(f"\n    % of per-condition oracle          : "
      f"{100.0 * _m['Mean Matched Sweep Objective'] / ORACLE_TEST:.2f}%")
print(f"    mean match distance (normalised)   : {_m['mean_match_distance']:.6f}")
print(f"    pitch/yaw/ipc saturation rates     : {_m['pitch_saturation_rate']:.3f} / "
      f"{_m['yaw_saturation_rate']:.3f} / {_m['ipc_saturation_rate']:.3f}")
if _m["Action-Sweep Coverage %"] < 99.0:
    warnings.warn(f"[FOWT-ARISE] action-sweep coverage is only "
                  f"{_m['Action-Sweep Coverage %']:.2f}%; the counterfactual objective is computed "
                  f"on that subset of test rows only.")
print("\n    NOTE: 'Mean Matched Sweep Objective' is a COUNTERFACTUAL estimate of what the policy's")
print("          action would have produced at the same physical condition. It is not a measured")
print("          transition reward and is never reported as one.")

## 28. IoT Robustness Evaluation

The **same** trained policy is evaluated under clean and degraded observations. No separate model is
retrained for evaluation, because the question is whether *this* controller stays useful when its
sensors deteriorate.

$$
\text{IoT Performance Gap} = \text{Clean} - \text{Degraded},
\qquad
\text{Robustness Drop \%} = 100\cdot\frac{\text{Clean}-\text{Degraded}}{\lvert\text{Clean}\rvert}
$$

Two complementary views are reported, because they can disagree and each answers a different question:

1. **Objective-based** — how much achieved counterfactual objective is lost. What a plant operator cares about.
2. **Behaviour-based** — the mean per-trajectory shift in the policy's own action between the clean
   and degraded views. This isolates *decision stability* from the reward, and it is far more
   sensitive: on a dataset where the objective is dominated by the reward definition, a component
   that genuinely stabilises decisions can be invisible in view (1) while being unmistakable in view (2).

Each of the four degradation mechanisms is additionally evaluated **in isolation** so that a
robustness loss can be attributed to noise, dropout, bias or staleness rather than lumped together.
All statistics aggregate per trajectory first, then across trajectories, with 95% confidence
intervals — treating each of the ~19k test transitions as independent would understate the intervals.

In [ ]:
def per_trajectory_action_shift(agent: FowtAriseAgent, experiment: str, mask: np.ndarray,
                                iot: IoTConfig) -> pd.Series:
    """Mean |action shift| per trajectory between clean and degraded observations (normalised units)."""
    a_clean, _, _, sub = policy_actions_for_split(agent, experiment, mask, None)
    a_deg, _, _, _ = policy_actions_for_split(agent, experiment, mask, iot)
    shift = np.mean(np.abs(action_to_norm(a_clean) - action_to_norm(a_deg)), axis=1)
    return pd.Series(shift, index=sub["traj_id"].to_numpy()).groupby(level=0).mean()


def _ci95(v: np.ndarray) -> float:
    n = len(v)
    if n < 2:
        return float("nan")
    return float(1.96 * np.std(v, ddof=1) / math.sqrt(n))


def evaluate_iot_robustness(experiment: str, agent: FowtAriseAgent) -> pd.DataFrame:
    """Objective-based and behaviour-based robustness, per isolated mode plus combined."""
    clean = RESULTS[experiment]["clean"] if experiment in RESULTS else \
        evaluate_actions_on_split(agent, experiment, test_mask, None, "test")
    clean_perf = clean[MONITORED_METRIC]

    rows = []
    for name, cfg in IOT_MODES.items():
        deg = evaluate_actions_on_split(agent, experiment, test_mask, cfg, "test")
        shift = per_trajectory_action_shift(agent, experiment, test_mask, cfg)
        sv = shift.to_numpy()
        gap = clean_perf - deg[MONITORED_METRIC]
        rows.append({
            "experiment": experiment, "mode": name,
            "clean_performance": clean_perf,
            "degraded_performance": deg[MONITORED_METRIC],
            "iot_performance_gap": gap,
            "robustness_drop_pct": (100.0 * gap / abs(clean_perf)) if abs(clean_perf) > 1e-12
                                    else float("nan"),
            "degraded_del_ratio": deg["mean_del_ratio"],
            "degraded_fatigue_relief_pct": deg["fatigue_relief_pct"],
            "degraded_power_loss_pct": deg["power_loss_pct"],
            "degraded_no_action_rate": deg["no_action_rate"],
            "mean_action_shift": float(np.mean(sv)),
            "median_action_shift": float(np.median(sv)),
            "std_action_shift": float(np.std(sv, ddof=1)) if len(sv) > 1 else float("nan"),
            "ci95_action_shift": _ci95(sv),
            "n_trajectories": int(len(sv)),
        })
    df = pd.DataFrame(rows)
    df.to_csv(EXP_DIRS[experiment]["base"] / "iot_robustness.csv", index=False)
    return df


print("=" * 79)
print("IoT ROBUSTNESS -- FOWT_ARISE")
print("=" * 79)
ROBUSTNESS = {}
ROBUSTNESS["FOWT_ARISE"] = evaluate_iot_robustness("FOWT_ARISE", TRAINED["FOWT_ARISE"]["agent"])
_r = ROBUSTNESS["FOWT_ARISE"]
print(f"    clean performance = {_r['clean_performance'].iloc[0]:+.6f}\n")
print(f"    {'mode':10s} {'degraded':>11s} {'gap':>11s} {'drop %':>8s} "
      f"{'action shift':>13s} {'95% CI':>9s}")
for _, _row in _r.iterrows():
    print(f"    {_row['mode']:10s} {_row['degraded_performance']:+11.6f} "
          f"{_row['iot_performance_gap']:+11.6f} {_row['robustness_drop_pct']:8.2f} "
          f"{_row['mean_action_shift']:13.6f} {_row['ci95_action_shift']:9.6f}")
print("\n    Objective-based view answers 'how much objective is lost'.")
print("    Behaviour-based view (action shift) answers 'how much do the DECISIONS move', which is")
print("    the more sensitive test of N4 and is compared statistically in Section 35.")

## 29. FOWT-ARISE Output Generation

Everything computed for the proposed model is written out in machine-readable form:
`metrics.json`, `metrics.csv`, `test_predictions.csv`, `validation_predictions.csv`,
`iot_robustness.csv`, `history.csv`, `config.json`, `training_summary.json`,
`state_feature_manifest.json`, plus `episode_metrics.csv` — the per-trajectory table that every
aggregate statistic in this notebook is computed from, so any number quoted later can be traced back
to the trajectories that produced it.

In [ ]:
def write_experiment_outputs(experiment: str) -> None:
    """Persist metrics + the per-trajectory audit trail for one experiment."""
    dirs = EXP_DIRS[experiment]
    res = RESULTS[experiment]
    metrics = {k: v for k, v in res["metrics"].items() if not k.startswith("_")}

    with open(dirs["base"] / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2, default=str)
    pd.DataFrame([metrics]).to_csv(dirs["base"] / "metrics.csv", index=False)

    preds = res["predictions"]
    matched = preds[preds["matched"]]
    if len(matched):
        ep = matched.groupby("traj_id").agg(
            n_transitions=("matched_del_ratio", "size"),
            mean_matched_sweep_objective=("matched_sweep_objective", "mean"),
            mean_del_ratio=("matched_del_ratio", "mean"),
            mean_fatigue_relief=("matched_fatigue_relief", "mean"),
            mean_power_loss_fraction=("matched_power_loss_fraction", "mean"),
            mean_controllable_share_max=("matched_controllable_share_max", "mean"),
            mean_match_distance=("match_distance_normalised", "mean"),
            no_action_rate=("no_action", "mean"),
        ).reset_index()
        shift = per_trajectory_action_shift(TRAINED[experiment]["agent"], experiment,
                                            test_mask, IOT_DEGRADED)
        ep = ep.merge(shift.rename("combined_iot_action_shift").reset_index()
                        .rename(columns={"index": "traj_id"}), on="traj_id", how="left")
        assert ep["traj_id"].is_unique
        ep.to_csv(dirs["base"] / "episode_metrics.csv", index=False)
        print(f"    {experiment}: episode_metrics.csv ({len(ep)} trajectories)")
    else:
        warnings.warn(f"[FOWT-ARISE] {experiment}: no matched rows; episode_metrics.csv not written.")

    print(f"    {experiment}: metrics.json / metrics.csv written to {dirs['base']}")


print("=" * 79)
print("FOWT-ARISE OUTPUT GENERATION")
print("=" * 79)
write_experiment_outputs("FOWT_ARISE")
for _f in ("history.csv", "config.json", "training_summary.json", "metrics.json", "metrics.csv",
           "test_predictions.csv", "validation_predictions.csv", "state_feature_manifest.json",
           "iot_robustness.csv", "episode_metrics.csv"):
    _p = EXP_DIRS["FOWT_ARISE"]["base"] / _f
    print(f"    [{'OK ' if _p.exists() else 'MISS'}] {_f}")

## 30–33. Ablations N1 – N4

Each ablation removes **exactly one** novelty. Everything else is not merely "intended to be" the
same — it is *the same code*: `run_experiment()` is called unchanged, the split, seed, batch size,
optimiser, budget, action limits, normalisation, counterfactual matcher and metric definitions are
shared, and `set_global_seed(SEED)` runs at the top of every experiment so all five start from
identical conditions.

| ablation | novelty removed | how it is realised | deliberately unchanged |
|---|---|---|---|
| **N1** | Physics-informed load-aware state | `use_n1=False`: the encoder becomes a flat MLP over `CORE_STATE_COLS`, and the physics block is genuinely **absent from the input** rather than zeroed | gate, reward, IoT training |
| **N2** | Adaptive multi-actuator control | `use_gate=False`: the learned gate and benefit head are removed, so the action comes straight from the joint trunk | still **one joint head over all three actuators**, physics state, reward, IoT training |
| **N3** | Multi-objective reward | reward and target-selection reward reduce to the load-relief term alone | architecture, gate, state, IoT training |
| **N4** | IoT-degradation-aware training | trains on clean observations only, `lambda_robust = 0` | architecture, gate, reward; **still evaluated on clean *and* degraded** so robustness is measurable |

Two traps this avoids explicitly:

* N2 does **not** disable an actuator. Removing yaw or IPC would confound "no adaptive gating" with
  "fewer actuators" and would make the comparison meaningless. The head stays joint and
  three-dimensional; only the *adaptive gating* goes.
* N4 is still evaluated under degradation. Training it clean and then only testing it clean would
  hide precisely the effect the ablation exists to measure.

No ablation is handicapped, and none is given a shorter budget or a worse learning rate. Where an
ablation beats the full model on a metric, Section 35 reports that as measured.

In [ ]:
for _abl in ["ABLATION_N1", "ABLATION_N2", "ABLATION_N3", "ABLATION_N4"]:
    TRAINED[_abl] = run_experiment(_abl)
    assert_no_test_leakage(f"post-training::{_abl}")
    print()
print(f"Training complete for: {list(TRAINED)}")

## 34. Ablation Evaluation

The same single-pass test evaluation and robustness protocol as Section 26–28, applied to each
ablation through the identical functions. Before comparing anything, a **fair-control audit** asserts
that the five configurations really do differ only where they are supposed to.

In [ ]:
for _abl in ["ABLATION_N1", "ABLATION_N2", "ABLATION_N3", "ABLATION_N4"]:
    print(f"--- test evaluation: {_abl} ---")
    RESULTS[_abl] = evaluate_experiment_on_test(_abl, TRAINED[_abl]["agent"])
    ROBUSTNESS[_abl] = evaluate_iot_robustness(_abl, TRAINED[_abl]["agent"])
    write_experiment_outputs(_abl)
    _mm = RESULTS[_abl]["metrics"]
    print(f"    objective={_mm['Mean Matched Sweep Objective']:+.6f}  "
          f"DEL={_mm['Mean DEL Ratio']:.5f}  fat={_mm['Fatigue Relief %']:.2f}%  "
          f"pow={_mm['Power Loss %']:.2f}%  no_act={_mm['No-Action Rate']:.3f}  "
          f"drop={_mm['Robustness Drop %']:.2f}%")
    print()

In [ ]:
print("=" * 79)
print("FAIR-ABLATION CONTROL AUDIT")
print("=" * 79)
_cfgs = {}
for _n in EXPERIMENTS:
    with open(EXP_DIRS[_n]["base"] / "config.json") as f:
        _cfgs[_n] = json.load(f)

# These must be byte-identical across all five experiments.
_must_match = [
    ("seed", lambda c: c["seed"]),
    ("split", lambda c: c["split"]),
    ("batch_size", lambda c: c["hyperparameters"]["batch_size"]),
    ("num_epochs", lambda c: c["hyperparameters"]["num_epochs"]),
    ("lr_actor", lambda c: c["hyperparameters"]["learning_rate_actor"]),
    ("lr_critic", lambda c: c["hyperparameters"]["learning_rate_critic"]),
    ("gamma", lambda c: c["hyperparameters"]["gamma"]),
    ("cql_alpha", lambda c: c["hyperparameters"]["cql_alpha"]),
    ("grad_clip", lambda c: c["hyperparameters"]["grad_clip_norm"]),
    ("early_stopping", lambda c: c["early_stopping"]),
    ("lr_scheduler", lambda c: c["lr_scheduler"]),
    ("action_dim", lambda c: c["architecture"]["action_dim"]),
    ("latent_dim", lambda c: c["architecture"]["latent_dim"]),
    ("rated_power", lambda c: c["reward"]["rated_power_w"]),
    ("wohler_m", lambda c: c["reward"]["wohler_m"]),
]
_fail = []
for _label, _get in _must_match:
    _vals = {n: _get(_cfgs[n]) for n in EXPERIMENTS}
    _uniq = {json.dumps(v, sort_keys=True, default=str) for v in _vals.values()}
    _ok = len(_uniq) == 1
    print(f"    [{'PASS' if _ok else 'FAIL'}] identical across all experiments: {_label}")
    if not _ok:
        _fail.append((_label, _vals))
if _fail:
    for _label, _vals in _fail:
        print(f"        {_label}: {_vals}")
    raise RuntimeError("[FOWT-ARISE] the ablations are NOT controlled: the settings above differ "
                       "where they must not. Any comparison would confound the novelty with them.")

# These must differ, and only in the intended direction.
print()
_intended = {
    "ABLATION_N1": ("architecture.use_n1", lambda c: c["architecture"]["use_n1"], False),
    "ABLATION_N2": ("architecture.use_gate", lambda c: c["architecture"]["use_gate"], False),
    "ABLATION_N3": ("reward.reduced_to_load_relief_only",
                    lambda c: c["reward"]["reduced_to_load_relief_only"], True),
    "ABLATION_N4": ("iot.training_mode", lambda c: c["iot"]["training_mode"], "clean"),
}
for _abl, (_label, _get, _expected) in _intended.items():
    _got, _full = _get(_cfgs[_abl]), _get(_cfgs["FOWT_ARISE"])
    _ok = (_got == _expected) and (_got != _full)
    print(f"    [{'PASS' if _ok else 'FAIL'}] {_abl}: {_label} = {_got!r} "
          f"(FOWT_ARISE = {_full!r})")
    if not _ok:
        raise RuntimeError(f"[FOWT-ARISE] {_abl} does not actually ablate its novelty: "
                           f"{_label}={_got!r}, expected {_expected!r} and different from "
                           f"FOWT_ARISE's {_full!r}.")
print("\n    [PASS] every ablation differs from FOWT-ARISE in exactly one intended respect.")

## 35. Ablation Comparison

The comparison table carries every metric §33 asks for, for all five experiments, exactly as
measured. Beyond the table, three analyses decide **what the numbers actually support** — because
with differences this small, a raw ranking is not evidence.

**1. Paired significance on the objective.** Every experiment is evaluated on the *same* test
trajectories, so the objective is differenced trajectory-by-trajectory against FOWT-ARISE and
reported with a 95% CI. Pairing removes between-trajectory variance, which is large here (test
trajectories span very different metocean conditions), and is far more sensitive than comparing two
independent means.

**2. Checkpoint-selection sensitivity.** `best.pt` is the `argmax` over a noisy validation curve. If
a between-model difference is smaller than the epoch-to-epoch wobble *inside the runs that produced
it*, the ranking is a checkpoint lottery, not a property of the method. Each comparison is judged
against `max(spread of the ablation, spread of FOWT-ARISE)` — not against a pooled median, which would
be dominated by the tight runs and would wave through a noisy one. Unequal epoch budgets from early
stopping are reported as a second confound, since "best of 40 epochs" beats "best of 15" on luck alone.

**3. Paired significance on robustness.** The objective on this dataset is dominated by the reward
definition (N3), which can make a component that genuinely improves *decision stability* look inert.
The same paired test is therefore also run on per-trajectory action shift under degradation. A
component can come out supported on one axis and not the other, and both are printed.

Per §34 and §57: where an ablation beats the full model, that is reported as measured. No metric is
adjusted, and no ablation is made to look worse.

In [ ]:
COMPARISON_COLUMNS = [
    "Method", "Mean Matched Sweep Objective", "Mean DEL Ratio", "Median DEL Ratio",
    "Fatigue Relief %", "Power Loss %", "Actuator Duty Proxy", "Mean Action Magnitude",
    "Mean |Yaw Action|", "No-Action Rate", "Action-Sweep Coverage %", "Clean Performance",
    "IoT-Degraded Performance", "IoT Performance Gap", "Robustness Drop %",
    "Parameter Count", "Best Validation Epoch",
]

_rows = []
for _n in EXPERIMENTS:
    _m = RESULTS[_n]["metrics"]
    _rows.append({c: _m.get(c) for c in COMPARISON_COLUMNS})
for _n in ("do_nothing", "ipc_half", "ipc_only", "behaviour_logged", "ORACLE_best_of_sweep"):
    _s = TEST_REFERENCES[_n]
    _rows.append({
        "Method": f"[reference] {_n}",
        "Mean Matched Sweep Objective": _s["matched_sweep_objective"],
        "Mean DEL Ratio": _s["mean_del_ratio"], "Median DEL Ratio": _s["median_del_ratio"],
        "Fatigue Relief %": _s["fatigue_relief_pct"], "Power Loss %": _s["power_loss_pct"],
        "Actuator Duty Proxy": _s["actuator_duty_proxy"],
        "Mean Action Magnitude": _s["mean_action_magnitude"],
        "Mean |Yaw Action|": _s["mean_abs_yaw_action"], "No-Action Rate": _s["no_action_rate"],
        "Action-Sweep Coverage %": _s["coverage_pct"],
    })
ablation_comparison = pd.DataFrame(_rows, columns=COMPARISON_COLUMNS)
ablation_comparison["% of Oracle"] = (
    100.0 * ablation_comparison["Mean Matched Sweep Objective"] / ORACLE_TEST)

ablation_comparison.to_csv(COMPARE_DIR / "final_ablation_comparison.csv", index=False)
with open(COMPARE_DIR / "final_ablation_comparison.json", "w") as f:
    json.dump(ablation_comparison.to_dict(orient="records"), f, indent=2, default=str)
if _OPENPYXL_AVAILABLE:
    ablation_comparison.to_excel(COMPARE_DIR / "final_ablation_comparison.xlsx", index=False)

print("=" * 79)
print("FINAL ABLATION COMPARISON (as measured; nothing adjusted)")
print("=" * 79)
with pd.option_context("display.width", 200, "display.max_columns", 40):
    print(ablation_comparison.to_string(index=False, float_format=lambda v: f"{v:,.5f}"))
print(f"\nSaved final_ablation_comparison.csv / .json"
      f"{' / .xlsx' if _OPENPYXL_AVAILABLE else ''} to {COMPARE_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# 1. paired significance on the OBJECTIVE (same test trajectories)
# ---------------------------------------------------------------------------
def _per_traj_objective(experiment: str) -> pd.Series:
    p = RESULTS[experiment]["predictions"]
    m = p[p["matched"]]
    return m.groupby("traj_id")["matched_sweep_objective"].mean()


_obj = {n: _per_traj_objective(n) for n in EXPERIMENTS}
print("=" * 88)
print("TRAJECTORY-LEVEL OBJECTIVE (mean +/- 95% CI across test trajectories)")
print("=" * 88)
for _n, _g in _obj.items():
    _v = _g.to_numpy()
    print(f"    {_n:14s} {_v.mean():+.6f} +/- {_ci95(_v):.6f}   "
          f"({100.0 * _v.mean() / ORACLE_TEST:6.1f}% of oracle, n={len(_v)} trajectories)")
print("\n    reference band (same trajectories):")
for _n in ("do_nothing", "ipc_half", "ipc_only", "ORACLE_best_of_sweep"):
    _s = TEST_REFERENCES[_n]["matched_sweep_objective"]
    print(f"    {_n:22s} {_s:+.6f}   ({100.0 * _s / ORACLE_TEST:6.1f}% of oracle)")

paired_objective_rows = []
_ref = _obj["FOWT_ARISE"]
print("\n" + "=" * 88)
print("PAIRED DIFFERENCE vs FOWT-ARISE -- OBJECTIVE  (positive => the ablation is BETTER)")
print("=" * 88)
for _n, _g in _obj.items():
    if _n == "FOWT_ARISE":
        continue
    _d = (_g - _ref).dropna().to_numpy()
    _ci = _ci95(_d)
    _sig = bool(abs(_d.mean()) > _ci)
    paired_objective_rows.append({"comparison": f"{_n} - FOWT_ARISE", "metric": "objective",
                                  "mean_paired_difference": float(_d.mean()), "ci95": _ci,
                                  "significant": _sig, "n_trajectories": int(len(_d))})
    print(f"    {_n:14s} delta={_d.mean():+.6f} +/- {_ci:.6f}  -> "
          f"{'SIGNIFICANT' if _sig else 'not significant'}")
pd.DataFrame(paired_objective_rows).to_csv(COMPARE_DIR / "paired_objective_significance.csv",
                                           index=False)

# ---------------------------------------------------------------------------
# 2. paired significance on ROBUSTNESS (per-trajectory action shift, combined degradation)
# ---------------------------------------------------------------------------
_shift = {n: per_trajectory_action_shift(TRAINED[n]["agent"], n, test_mask, IOT_DEGRADED)
          for n in EXPERIMENTS}
paired_robust_rows = []
_ref_sh = _shift["FOWT_ARISE"]
print("\n" + "=" * 88)
print("PAIRED DIFFERENCE vs FOWT-ARISE -- ROBUSTNESS (per-trajectory action shift; lower = stabler)")
print("=" * 88)
print(f"    FOWT_ARISE mean action shift = {_ref_sh.mean():.6f}")
print(f"    {'comparison':16s} {'ablation':>11s} {'delta':>11s} {'95% CI':>10s} {'verdict':>18s}")
for _n, _g in _shift.items():
    if _n == "FOWT_ARISE":
        continue
    _d = (_g - _ref_sh).dropna().to_numpy()
    _ci = _ci95(_d)
    _sig = bool(abs(_d.mean()) > _ci)
    paired_robust_rows.append({
        "comparison": f"{_n} - FOWT_ARISE", "metric": "combined_iot_action_shift",
        "ablation_mean_action_shift": float(_g.mean()),
        "fowt_arise_mean_action_shift": float(_ref_sh.mean()),
        "mean_paired_difference": float(_d.mean()), "ci95": _ci,
        "significant": _sig, "n_trajectories": int(len(_d))})
    print(f"    {_n:16s} {_g.mean():11.6f} {_d.mean():+11.6f} {_ci:10.6f} "
          f"{'SIGNIFICANT' if _sig else 'not significant':>18s}")
pd.DataFrame(paired_robust_rows).to_csv(COMPARE_DIR / "paired_robustness_significance.csv",
                                        index=False)
print("\n    Sign: a POSITIVE delta means removing that component made the policy LESS stable under")
print("    sensor degradation -- i.e. the component was earning its place on this axis.")

In [ ]:
# ---------------------------------------------------------------------------
# 3. checkpoint-selection sensitivity, judged PER COMPARISON
# ---------------------------------------------------------------------------
TOP_K_EPOCHS = 5
_spread, _epochs, _best_ep = {}, {}, {}
print("=" * 88)
print("CHECKPOINT-SELECTION SENSITIVITY (validation objective across epochs, per run)")
print("=" * 88)
print(f"    {'experiment':14s} {'epochs':>7s} {'best@ep':>8s} {'best':>11s} {'worst':>11s} "
      f"{'std':>10s} {'k':>3s} {'top-k range':>12s}")
for _n in EXPERIMENTS:
    _h = pd.read_csv(EXP_DIRS[_n]["base"] / "history.csv")
    _v = _h["matched_sweep_objective"].dropna()
    if len(_v) == 0:
        continue
    _k = min(TOP_K_EPOCHS, len(_v))
    _tk = _v.nlargest(_k)
    _spread[_n] = float(_tk.max() - _tk.min())
    _epochs[_n] = int(len(_v))
    _best_ep[_n] = int(_h.loc[_v.idxmax(), "epoch"])
    print(f"    {_n:14s} {len(_v):7d} {_best_ep[_n]:8d} {_v.max():+11.6f} {_v.min():+11.6f} "
          f"{(_v.std(ddof=1) if len(_v) > 1 else float('nan')):10.6f} {_k:3d} {_spread[_n]:12.6f}")

CKPT_ROBUST = {}
if _spread and paired_objective_rows:
    _fa = _spread.get("FOWT_ARISE", float("nan"))
    print(f"\n    per-comparison test: |delta| vs the checkpoint noise of the TWO runs involved")
    print(f"    (k = min({TOP_K_EPOCHS}, epochs logged); a short run yields an optimistically SMALL spread)")
    print(f"    {'comparison':16s} {'|delta|':>11s} {'noise floor':>12s} {'verdict':>16s}")
    for _r in paired_objective_rows:
        _n = _r["comparison"].split(" - ")[0]
        if _n == "ABLATION_N3":
            continue                      # excluded explicitly below, never silently
        _da = abs(_r["mean_paired_difference"])
        _floor = float(np.nanmax([_spread.get(_n, np.nan), _fa]))
        _surv = bool(_da > _floor)
        CKPT_ROBUST[_n] = {"delta_abs": _da, "noise_floor": _floor, "survives": _surv}
        print(f"    {_n:16s} {_da:11.6f} {_floor:12.6f} "
              f"{'exceeds noise' if _surv else 'WITHIN noise':>16s}")
    _within = [k for k, v in CKPT_ROBUST.items() if not v["survives"]]
    if _within:
        print(f"\n    => {', '.join(_within)}: the difference is SMALLER than the epoch-to-epoch wobble")
        print( "       of the runs that produced it. 'Significant across trajectories' means consistent")
        print( "       over the test set GIVEN these two checkpoints -- it does NOT mean a rerun would")
        print( "       reproduce the ranking. Treat as UNRESOLVED on the objective.")
    _surv = [k for k, v in CKPT_ROBUST.items() if v["survives"]]
    if _surv:
        print(f"\n    => {', '.join(_surv)}: delta exceeds the checkpoint noise of both runs, so it is")
        print( "       not purely an artefact of which epoch was selected. Still one seed per config.")
    if len(_epochs) > 1:
        _mn, _mx = min(_epochs.values()), max(_epochs.values())
        if _mx >= 1.5 * _mn:
            _lg = max(_epochs, key=_epochs.get); _sh = min(_epochs, key=_epochs.get)
            print(f"\n    CONFOUND -- UNEQUAL EPOCH BUDGETS: early stopping ran '{_lg}' for "
                  f"{_epochs[_lg]} epochs")
            print(f"    but '{_sh}' for only {_epochs[_sh]}. The checkpoint is an ARGMAX over a noisy")
            print( "    validation curve, so more epochs means more draws and a higher expected maximum")
            print(f"    regardless of model quality. Any advantage held by '{_lg}' is confounded with its")
            print( "    larger budget -- including when that experiment is FOWT_ARISE itself.")
    print("\n    N3 is excluded from the per-comparison test above (stated, not silently filtered):")
    print("    its delta is orders of magnitude larger than any checkpoint noise.")

with open(COMPARE_DIR / "checkpoint_sensitivity.json", "w") as f:
    json.dump({"top_k": TOP_K_EPOCHS, "spread": _spread, "epochs": _epochs,
               "best_epoch": _best_ep, "per_comparison": CKPT_ROBUST}, f, indent=2)

In [ ]:
# ---------------------------------------------------------------------------
# Verdicts -- derived from the numbers just computed, never hard-coded text
# ---------------------------------------------------------------------------
print("=" * 88)
print("WHAT THE ABLATION STUDY ACTUALLY SUPPORTS")
print("=" * 88)
_vobj = {r["comparison"].split(" - ")[0]: r for r in paired_objective_rows}
_vrob = {r["comparison"].split(" - ")[0]: r for r in paired_robust_rows}
_metrics_by = {n: RESULTS[n]["metrics"] for n in EXPERIMENTS}

# ---- N3 ----
_r3 = _vobj.get("ABLATION_N3")
print("N3 (fatigue-power-actuation multi-objective reward):")
if _r3 is None:
    print("    not run this session; no verdict.")
else:
    _o3 = _obj["ABLATION_N3"].mean()
    if _r3["significant"] and _r3["mean_paired_difference"] < 0:
        print(f"    DECISIVELY VALIDATED. Removing it moves the objective to {_o3:+.6f}, a paired")
        print(f"    difference of {_r3['mean_paired_difference']:+.6f} (95% CI +/-{_r3['ci95']:.6f}).")
        print(f"    Power loss goes from {_metrics_by['FOWT_ARISE']['Power Loss %']:.2f}% to "
              f"{_metrics_by['ABLATION_N3']['Power Loss %']:.2f}%: selecting targets by load relief")
        print( "    alone buys fatigue reduction by feathering and pays for it in energy.")
    elif _r3["significant"]:
        print(f"    CONTRADICTED: removing the multi-objective reward IMPROVED the objective by "
              f"{_r3['mean_paired_difference']:+.6f}. Reported as measured.")
    else:
        print(f"    NOT distinguishable ({_r3['mean_paired_difference']:+.6f} +/- {_r3['ci95']:.6f}).")

# ---- N1, N2, N4: both axes, with the checkpoint-noise override ----
for _abl, _label, _desc in (("ABLATION_N1", "N1", "physics-informed load-aware state"),
                            ("ABLATION_N2", "N2", "adaptive multi-actuator gating"),
                            ("ABLATION_N4", "N4", "IoT-degradation-aware training")):
    print(f"\n{_label} ({_desc}):")
    _ro, _rr, _cr = _vobj.get(_abl), _vrob.get(_abl), CKPT_ROBUST.get(_abl)
    if _ro is None:
        print("    not run this session; no verdict.")
        continue

    # axis 1: objective
    _d, _c = _ro["mean_paired_difference"], _ro["ci95"]
    if not _ro["significant"]:
        print(f"    objective : NOT distinguishable ({_d:+.6f} +/- {_c:.6f}).")
    else:
        _dirn = "BETTER than" if _d > 0 else "WORSE than"
        print(f"    objective : the ablation is {_dirn} the full model by {abs(_d):.6f} "
              f"(95% CI +/-{_c:.6f}) -> significant.")
    if _cr is not None and not _cr["survives"]:
        print(f"                OVERRIDE: |delta|={_cr['delta_abs']:.6f} is BELOW the "
              f"{_cr['noise_floor']:.6f} checkpoint-noise")
        print( "                floor of the two runs compared, so the objective verdict is")
        print(f"                DOWNGRADED to UNRESOLVED. Do not claim {_label} helps or hurts on it.")

    # axis 2: robustness
    if _rr is None:
        print("    robustness: unavailable.")
    else:
        _rd, _rc = _rr["mean_paired_difference"], _rr["ci95"]
        if _rr["significant"] and _rd > 0:
            print(f"    robustness: SUPPORTED. Removing it raises action shift by {_rd:+.6f} "
                  f"(95% CI +/-{_rc:.6f};")
            print(f"                {_rr['ablation_mean_action_shift']:.4f} ablated vs "
                  f"{_rr['fowt_arise_mean_action_shift']:.4f} full), so {_label} contributes")
            print( "                DECISION STABILITY under degraded sensing.")
        elif _rr["significant"]:
            print(f"    robustness: CONTRADICTED -- the ablation is more stable "
                  f"({_rd:+.6f} +/- {_rc:.6f}).")
        else:
            print(f"    robustness: not distinguishable ({_rd:+.6f} +/- {_rc:.6f}).")

print("\n" + "-" * 88)
print("Reading guide: the objective on this dataset is dominated by the reward definition (N3), so a")
print("component whose contribution is decision STABILITY rather than score can be invisible on the")
print("objective axis and unmistakable on the robustness axis. Both are reported, including where")
print("they disagree, and a single-seed study cannot settle differences that sit inside checkpoint")
print("noise. Multiple seeds per configuration would be required for that.")
print("=" * 88)

## 36. Baseline Integration &nbsp;·&nbsp; 37. Final Comparison

If `BASELINE_DIR` points at a directory containing already-computed RB-FOWT / CQL / IQL outputs, their
saved metrics are **read**. They are never retrained here and never modified, so the numbers stay the
ones their own notebook produced.

If a baseline is missing, its row is simply absent and a message says so. Missing baseline metrics are
never replaced with invented values, and a partially-available baseline contributes only the metrics
it actually has — the rest stay `NaN` rather than being filled in.

In [ ]:
BASELINE_METHODS = ["RB-FOWT", "CQL", "IQL"]


def load_baseline(baseline_root: Path, name: str) -> Optional[dict]:
    """Read one baseline's saved metrics. Returns None if genuinely absent. Never fabricates."""
    for cand in (baseline_root / name, baseline_root / name.replace("-", "_"),
                 baseline_root / name.lower()):
        if not cand.exists():
            continue
        row = {"Method": name}
        found = False
        mj = cand / "final_metrics.json"
        if not mj.exists():
            mj = cand / "metrics.json"
        if mj.exists():
            with open(mj) as f:
                payload = json.load(f)
            if isinstance(payload, dict):
                flat = payload.get("summary_row", payload)
                if isinstance(flat, dict):
                    for c in COMPARISON_COLUMNS:
                        if c in flat:
                            row[c] = flat[c]
                            found = True
        mc = cand / "metrics.csv"
        if mc.exists():
            df = pd.read_csv(mc)
            if len(df):
                for c in COMPARISON_COLUMNS:
                    if c in df.columns and c not in row:
                        row[c] = df.iloc[0][c]
                        found = True
        if found:
            return row
        warnings.warn(f"[FOWT-ARISE] found {cand} but no readable metrics.json / metrics.csv inside; "
                      f"'{name}' is reported as unavailable rather than guessed at.")
    return None


baseline_rows, BASELINE_AVAILABLE = [], False
print("=" * 79)
print("BASELINE INTEGRATION")
print("=" * 79)
if not BASELINE_DIR:
    print("    Baseline comparison unavailable because BASELINE_DIR was not supplied.")
else:
    _root = Path(BASELINE_DIR)
    if not _root.exists():
        print(f"    Baseline comparison unavailable because BASELINE_DIR does not exist: {_root}")
    else:
        for _b in BASELINE_METHODS:
            _row = load_baseline(_root, _b)
            if _row is None:
                print(f"    [absent ] {_b}: no saved metrics found under {_root}")
            else:
                _n_found = len([k for k in _row if k != 'Method'])
                print(f"    [loaded ] {_b}: {_n_found} metric(s) read (never recomputed)")
                baseline_rows.append(_row)
        BASELINE_AVAILABLE = len(baseline_rows) > 0
        if not BASELINE_AVAILABLE:
            print(f"\n    Baseline comparison unavailable: none of {BASELINE_METHODS} had readable "
                  f"outputs under {_root}.")

final_comparison = pd.concat(
    [pd.DataFrame(baseline_rows, columns=COMPARISON_COLUMNS) if baseline_rows
     else pd.DataFrame(columns=COMPARISON_COLUMNS),
     ablation_comparison[COMPARISON_COLUMNS]],
    ignore_index=True)
final_comparison["% of Oracle"] = (
    100.0 * pd.to_numeric(final_comparison["Mean Matched Sweep Objective"], errors="coerce")
    / ORACLE_TEST)
final_comparison.to_csv(COMPARE_DIR / "final_baseline_comparison.csv", index=False)
if _OPENPYXL_AVAILABLE:
    final_comparison.to_excel(COMPARE_DIR / "final_baseline_comparison.xlsx", index=False)
with open(COMPARE_DIR / "final_baseline_comparison.json", "w") as f:
    json.dump(final_comparison.to_dict(orient="records"), f, indent=2, default=str)

print("\n" + "=" * 79)
print("FINAL COMPARISON" + ("" if BASELINE_AVAILABLE else "  (baselines unavailable -- rows absent, not invented)"))
print("=" * 79)
with pd.option_context("display.width", 220, "display.max_columns", 40):
    print(final_comparison.to_string(index=False, float_format=lambda v: f"{v:,.5f}"))
print(f"\nSaved final_baseline_comparison.csv / .json"
      f"{' / .xlsx' if _OPENPYXL_AVAILABLE else ''} to {COMPARE_DIR}")
if not BASELINE_AVAILABLE:
    print("\nBaseline comparison unavailable because BASELINE_DIR was not supplied "
          "(or contained no readable baseline outputs). FOWT-ARISE and its ablations were still")
    print("trained and evaluated independently, and the reference policies in the table above")
    print("(do_nothing / ipc_only / oracle) provide an absolute band in the meantime.")

## 38–41. Plots

Every figure is its **own** figure — no subplot grids — at `fontsize = 20` and `dpi = 300`, saved
under the relevant experiment's `plots/` directory or under `comparison/comparison_plots/`.

* **38 Training** (per experiment): training loss, validation loss, actor loss, critic loss, learning
  rate, mean reward, plus the validation objective trace with its selected epoch marked
* **39 Performance**: DEL ratio, fatigue relief, power loss, actuator duty, mean action magnitude,
  yaw action, no-action rate
* **40 Robustness**: clean performance, IoT-degraded performance, IoT performance gap, robustness
  drop %, and action shift per degradation mode
* **41 Ablation**: objective, DEL ratio, fatigue relief, power loss, robustness

A note on reading the bar charts: DEL ratio and power loss are **lower-is-better**, everything else
is higher-is-better. Each figure states which in its axis label, because a reader skimming figures
out of context otherwise has no way to tell.

In [ ]:
FIG_COUNT = {"n": 0}


def _finish(fig, ax, path: Path, title: str, xlabel: str, ylabel: str) -> None:
    ax.set_title(title, fontsize=FONT_SIZE)
    ax.set_xlabel(xlabel, fontsize=FONT_SIZE)
    ax.set_ylabel(ylabel, fontsize=FONT_SIZE)
    ax.tick_params(labelsize=FONT_SIZE * 0.7)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=DPI)
    plt.close(fig)
    FIG_COUNT["n"] += 1


def line_plot(x, ys: dict, path: Path, title: str, xlabel: str, ylabel: str,
              logy: bool = False, vline=None, vline_label: str = "") -> None:
    """One metric (or a few labelled series) against epoch. Its own figure."""
    fig, ax = plt.subplots(figsize=(12, 8))
    for label, y in ys.items():
        ax.plot(x, y, linewidth=2.5, label=label)
    if logy:
        ax.set_yscale("log")
    if vline is not None:
        ax.axvline(vline, color="black", linestyle="--", linewidth=2, label=vline_label)
    if len(ys) > 1 or vline is not None:
        ax.legend(fontsize=FONT_SIZE * 0.7)
    _finish(fig, ax, path, title, xlabel, ylabel)


EXP_COLORS = {"FOWT_ARISE": "#1e8449", "ABLATION_N1": "#b9770e", "ABLATION_N2": "#2471a3",
              "ABLATION_N3": "#943126", "ABLATION_N4": "#6c3483"}


def bar_plot(labels, values, path: Path, title: str, ylabel: str,
             errors=None, colors=None) -> None:
    """One grouped-by-method bar chart. Its own figure."""
    fig, ax = plt.subplots(figsize=(13, 8))
    cols = colors or [EXP_COLORS.get(l, "#5d6d7e") for l in labels]
    ax.bar(labels, values, color=cols, yerr=errors,
           capsize=(8 if errors is not None else 0), edgecolor="black")
    ax.tick_params(axis="x", labelsize=FONT_SIZE * 0.65, rotation=20)
    _finish(fig, ax, path, title, "", ylabel)


def grouped_bar_plot(group_labels, series: dict, path: Path, title: str, ylabel: str,
                     errors: dict = None) -> None:
    """Methods x degradation-mode grouped bars. Its own figure."""
    fig, ax = plt.subplots(figsize=(16, 9))
    x = np.arange(len(group_labels))
    w = 0.8 / max(len(series), 1)
    for i, (name, vals) in enumerate(series.items()):
        off = x + (i - (len(series) - 1) / 2) * w
        err = (errors or {}).get(name)
        ax.bar(off, vals, width=w, label=name, color=EXP_COLORS.get(name, "#5d6d7e"),
               yerr=err, capsize=(5 if err is not None else 0), edgecolor="black", linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([g.capitalize() for g in group_labels], fontsize=FONT_SIZE * 0.7)
    ax.legend(fontsize=FONT_SIZE * 0.6)
    _finish(fig, ax, path, title, "", ylabel)


def scatter_plot(xv, yv, labels, path: Path, title: str, xlabel: str, ylabel: str) -> None:
    fig, ax = plt.subplots(figsize=(12, 9))
    for xi, yi, l in zip(xv, yv, labels):
        ax.scatter(xi, yi, s=320, color=EXP_COLORS.get(l, "#5d6d7e"), label=l,
                   edgecolor="black", linewidth=1.5)
    ax.legend(fontsize=FONT_SIZE * 0.65)
    _finish(fig, ax, path, title, xlabel, ylabel)


print("    plot helpers defined (individual figures only, fontsize "
      f"{FONT_SIZE}, dpi {DPI})")

In [ ]:
# ---------------- 38. training plots, per experiment ----------------
print("=" * 79)
print("38. TRAINING PLOTS")
print("=" * 79)
for _n in EXPERIMENTS:
    _h = pd.read_csv(EXP_DIRS[_n]["base"] / "history.csv")
    if len(_h) == 0:
        print(f"    {_n}: empty history; training plots skipped.")
        continue
    _pd_ = EXP_DIRS[_n]["plots"]
    _e = _h["epoch"]
    _best = TRAINED[_n]["summary"]["best_epoch"]
    line_plot(_e, {"train loss": _h["train_loss"]}, _pd_ / "training_loss.png",
              f"{_n} — Training Loss (actor + critic)", "Epoch", "Loss (lower is better)")
    line_plot(_e, {"validation loss": _h["validation_loss"]}, _pd_ / "validation_loss.png",
              f"{_n} — Validation Loss (imitation vs best-action targets)", "Epoch",
              "Loss (lower is better)")
    line_plot(_e, {"actor loss": _h["actor_loss"]}, _pd_ / "actor_loss.png",
              f"{_n} — Actor Loss", "Epoch", "Loss (lower is better)")
    line_plot(_e, {"critic loss": _h["critic_loss"]}, _pd_ / "critic_loss.png",
              f"{_n} — Critic Loss (TD + CQL)", "Epoch", "Loss (lower is better)")
    line_plot(_e, {"actor lr": _h["learning_rate"], "critic lr": _h["learning_rate_critic"]},
              _pd_ / "learning_rate.png", f"{_n} — Learning Rate", "Epoch", "Learning rate",
              logy=True)
    line_plot(_e, {"mean reward": _h["mean_reward"]}, _pd_ / "mean_reward.png",
              f"{_n} — Mean Training Reward", "Epoch", "Reward (higher is better)")
    line_plot(_e, {"validation objective": _h["matched_sweep_objective"]},
              _pd_ / "validation_objective.png",
              f"{_n} — Validation Mean Matched Sweep Objective", "Epoch",
              "Counterfactual objective (higher is better)",
              vline=_best, vline_label=f"selected epoch {_best}")
    if _h["robustness_loss"].notna().any():
        line_plot(_e, {"consistency loss": _h["robustness_loss"]}, _pd_ / "robustness_loss.png",
                  f"{_n} — N4 Clean-vs-Degraded Consistency Loss", "Epoch",
                  "Loss (lower is better)")
    _comp = {k: _h[k] for k in ("fatigue_reward", "power_penalty", "actuation_penalty",
                                "smoothness_penalty") if _h[k].notna().any()}
    if _comp:
        line_plot(_e, _comp, _pd_ / "reward_components.png",
                  f"{_n} — N3 Reward Components", "Epoch", "Component value")
    line_plot(_e, {"clean": _h["IoT_clean_performance"], "degraded": _h["IoT_degraded_performance"]},
              _pd_ / "iot_clean_vs_degraded.png",
              f"{_n} — Validation Objective: Clean vs IoT-Degraded", "Epoch",
              "Counterfactual objective (higher is better)")
    print(f"    {_n}: training plots -> {_pd_}")

In [ ]:
# ---------------- 39/40/41. performance, robustness, ablation comparisons ----------------
CP = COMPARE_DIR / "comparison_plots"
_labels = EXPERIMENTS
_get = lambda key: [RESULTS[n]["metrics"][key] for n in _labels]

print("=" * 79)
print("39-41. PERFORMANCE / ROBUSTNESS / ABLATION PLOTS")
print("=" * 79)

# --- 39 performance / structural / power / actuation ---
bar_plot(_labels, _get("Mean DEL Ratio"), CP / "del_ratio_comparison.png",
         "Mean DEL Ratio (counterfactual)", "DEL ratio — LOWER is better")
bar_plot(_labels, _get("Median DEL Ratio"), CP / "median_del_ratio_comparison.png",
         "Median DEL Ratio (counterfactual)", "DEL ratio — LOWER is better")
bar_plot(_labels, _get("Fatigue Relief %"), CP / "fatigue_relief_comparison.png",
         "Fatigue Relief", "Fatigue relief % — HIGHER is better")
bar_plot(_labels, _get("Power Loss %"), CP / "power_loss_comparison.png",
         "Power Loss (a cost, read with fatigue relief)", "Power loss % — LOWER is better")
bar_plot(_labels, _get("Actuator Duty Proxy"), CP / "actuator_duty_comparison.png",
         "Actuator Duty Proxy", "Duty proxy — LOWER is better")
bar_plot(_labels, _get("Mean Action Magnitude"), CP / "action_magnitude_comparison.png",
         "Mean Action Magnitude", "Mean |a| (normalised)")
bar_plot(_labels, _get("Mean |Yaw Action|"), CP / "yaw_action_comparison.png",
         "Mean |Yaw Action|", "Mean |yaw| [deg]")
bar_plot(_labels, _get("No-Action Rate"), CP / "no_action_rate_comparison.png",
         "No-Action Rate (learned, not thresholded)", "Fraction of steps with no action")
scatter_plot(_get("Power Loss %"), _get("Fatigue Relief %"), _labels,
             CP / "fatigue_vs_power_tradeoff.png",
             "Fatigue Relief vs Power Loss Trade-off",
             "Power loss % — LOWER is better", "Fatigue relief % — HIGHER is better")

# --- 40 robustness ---
bar_plot(_labels, _get("Clean Performance"), CP / "clean_performance_comparison.png",
         "Clean Performance", "Mean Matched Sweep Objective — HIGHER is better")
bar_plot(_labels, _get("IoT-Degraded Performance"), CP / "iot_degraded_performance_comparison.png",
         "IoT-Degraded Performance", "Mean Matched Sweep Objective — HIGHER is better")
bar_plot(_labels, _get("IoT Performance Gap"), CP / "iot_performance_gap_comparison.png",
         "IoT Performance Gap (clean − degraded)", "Objective lost — LOWER is better")
bar_plot(_labels, _get("Robustness Drop %"), CP / "robustness_drop_comparison.png",
         "Robustness Drop", "Robustness drop % — LOWER is better")

_modes = list(IOT_MODES.keys())
_shift_series = {n: [float(ROBUSTNESS[n].set_index("mode").loc[m, "mean_action_shift"])
                     for m in _modes] for n in _labels}
_shift_err = {n: [float(ROBUSTNESS[n].set_index("mode").loc[m, "ci95_action_shift"])
                  for m in _modes] for n in _labels}
grouped_bar_plot(_modes, _shift_series, CP / "robustness_action_shift_by_mode.png",
                 "Decision Stability under Each Degradation Mode (95% CI)",
                 "Mean per-trajectory action shift — LOWER is better", errors=_shift_err)
_gap_series = {n: [float(ROBUSTNESS[n].set_index("mode").loc[m, "iot_performance_gap"])
                   for m in _modes] for n in _labels}
grouped_bar_plot(_modes, _gap_series, CP / "robustness_gap_by_mode.png",
                 "Objective Lost under Each Degradation Mode",
                 "IoT performance gap — LOWER is better")

# --- 41 ablation headline ---
bar_plot(_labels, _get("Mean Matched Sweep Objective"), CP / "ablation_objective_comparison.png",
         "Proposed vs Ablations — Counterfactual Objective",
         "Mean Matched Sweep Objective — HIGHER is better")
_po = {r["comparison"].split(" - ")[0]: r for r in paired_objective_rows}
_pl = [n for n in _labels if n != "FOWT_ARISE"]
bar_plot(_pl, [_po[n]["mean_paired_difference"] for n in _pl],
         CP / "ablation_paired_objective_difference.png",
         "Paired Objective Difference vs FOWT-ARISE (positive = ablation better)",
         "Mean paired difference", errors=[_po[n]["ci95"] for n in _pl])
_pr = {r["comparison"].split(" - ")[0]: r for r in paired_robust_rows}
bar_plot(_pl, [_pr[n]["mean_paired_difference"] for n in _pl],
         CP / "ablation_paired_robustness_difference.png",
         "Paired Action-Shift Difference vs FOWT-ARISE (positive = component aided stability)",
         "Mean paired difference", errors=[_pr[n]["ci95"] for n in _pl])
bar_plot(_labels, [100.0 * RESULTS[n]["metrics"]["Mean Matched Sweep Objective"] / ORACLE_TEST
                   for n in _labels], CP / "ablation_pct_of_oracle.png",
         "Fraction of the Per-Condition Oracle Achieved", "% of oracle — HIGHER is better")

print(f"    comparison plots -> {CP}")
print(f"\n    total figures written so far: {FIG_COUNT['n']}")

## 42. SHAP Explainability

SHAP is computed for the **final trained FOWT-ARISE model only**, never for the ablations.

### Provenance, stated explicitly (§40)

| | source |
|---|---|
| **input features explained** | the `FULL_STATE_COLS` vector, *after* the TRAIN-fitted scaler — i.e. exactly the tensor the actor consumes |
| **model output explained** | one actuator head at a time: normalised pitch, yaw, IPC |
| **background data** | `SHAP_BACKGROUND_SIZE` rows drawn from **TRAIN only** |
| **explained sample** | `SHAP_TEST_SAMPLE_SIZE` rows drawn from **TEST only** |

Background from train and explanations from test is the correct split: the background defines the
reference distribution the model was fitted against, while the explained rows must be held-out to say
anything about deployment behaviour. Both draws use a fixed seed, so the sample is reproducible.
Explaining test rows does **not** train or tune anything — SHAP runs after the single test evaluation
and influences no parameter.

### Device handling

`KernelExplainer` hands the wrapper plain CPU NumPy arrays, while the actor lives on `DEVICE`. Both
transfers are required: without `.to(DEVICE)` the first `nn.Linear` raises *"mat1 is on cpu, different
from other tensors on cuda:0"*, and without `.cpu()` the return `.numpy()` raises *"can't convert
cuda:0 device type tensor to numpy"*. The head under test is bound as a default argument so the
closure captures that head rather than the loop variable's final value.

If SHAP fails for a head, that is reported and the head is skipped. **No fabricated attributions are
ever substituted** for a failed computation.

### Interpretive caveat

SHAP values are a **model-level** feature-attribution diagnostic. They describe what the trained
policy's decisions are sensitive to. They are **not** a claim about the physical causality of tower
fatigue.

In [ ]:
SHAP_META = {"completed": False, "heads_succeeded": [], "heads_failed": []}
shap_completed = False

if not _SHAP_AVAILABLE:
    print("SHAP skipped: the `shap` package is not available in this environment.")
elif "FOWT_ARISE" not in TRAINED:
    print("SHAP skipped: FOWT_ARISE was not trained in this run.")
else:
    _sd = EXP_DIRS["FOWT_ARISE"]["shap"]
    _agent = TRAINED["FOWT_ARISE"]["agent"]
    _cols = state_columns_for("FOWT_ARISE")
    _scaler = scaler_for("FOWT_ARISE")

    _n_bg = SHAP_BACKGROUND_SIZE
    _n_ex = SHAP_TEST_SAMPLE_SIZE
    _ns = SHAP_NSAMPLES
    if DRY_RUN:
        _n_bg, _n_ex, _ns = min(20, _n_bg), min(20, _n_ex), min(30, _ns)
        print(f"    DRY_RUN: SHAP sizes reduced to background={_n_bg} explain={_n_ex} "
              f"nsamples={_ns} (the full run uses {SHAP_BACKGROUND_SIZE}/{SHAP_TEST_SAMPLE_SIZE}/"
              f"{SHAP_NSAMPLES})")

    _rng = np.random.default_rng(SEED + 999)
    _tr = transitions.loc[train_mask]
    _te = transitions.loc[test_mask]
    _bg_idx = _rng.choice(len(_tr), size=min(_n_bg, len(_tr)), replace=False)
    _ex_idx = _rng.choice(len(_te), size=min(_n_ex, len(_te)), replace=False)

    _background = _scaler.transform(
        _tr.iloc[_bg_idx][_cols].to_numpy(dtype=np.float64)).astype(np.float32)
    _explain = _scaler.transform(
        _te.iloc[_ex_idx][_cols].to_numpy(dtype=np.float64)).astype(np.float32)
    print(f"    background : {_background.shape} rows from TRAIN")
    print(f"    explained  : {_explain.shape} rows from TEST")
    print(f"    features   : {len(_cols)} (post-scaler, exactly the actor's input tensor)")

    class SingleHeadWrapper(nn.Module):
        """Exposes ONE actuator head as a scalar-output module, which is what SHAP expects."""

        def __init__(self, actor: nn.Module, head: int):
            super().__init__()
            self.actor, self.head = actor, head

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            dev = next(self.actor.parameters()).device
            if x.device != dev:
                x = x.to(dev)
            a, _, _, _ = self.actor(x)
            return a[:, self.head:self.head + 1]

    _agent.actor.eval()
    HEAD_NAMES = ["pitch", "yaw", "ipc"]
    shap_values_by_head, shap_explainer_used = {}, {}

    for _hi, _hn in enumerate(HEAD_NAMES):
        _wrap = SingleHeadWrapper(_agent.actor, _hi).eval()

        def _f(x_np: np.ndarray, _w=_wrap) -> np.ndarray:
            # Both transfers are mandatory -- see the markdown above. `_w` is bound as a default
            # argument so this closure captures THIS head, not the loop variable's final value.
            with torch.no_grad():
                t = torch.from_numpy(np.ascontiguousarray(x_np, dtype=np.float32)).to(DEVICE)
                return _w(t).detach().cpu().numpy()

        _vals, _used = None, "KernelExplainer"
        try:
            _ex = shap.KernelExplainer(_f, _background)
            _vals = np.asarray(_ex.shap_values(_explain, nsamples=_ns, silent=True))
            if _vals.ndim == 3:
                _vals = _vals[..., 0]
        except Exception as _e1:
            warnings.warn(f"[FOWT-ARISE] KernelExplainer failed for head '{_hn}': {_e1}. "
                          f"Falling back to shap.Explainer.")
            _used = "shap.Explainer (fallback)"
            try:
                _ex = shap.Explainer(_f, _background)
                _vals = np.asarray(_ex(_explain).values)
                if _vals.ndim == 3:
                    _vals = _vals[..., 0]
            except Exception as _e2:
                warnings.warn(f"[FOWT-ARISE] fallback ALSO failed for head '{_hn}': {_e2}. "
                              f"This head is skipped; no fabricated SHAP values are substituted.")
                _used = "FAILED"
        shap_values_by_head[_hn] = _vals
        shap_explainer_used[_hn] = _used
        print(f"    head '{_hn:5s}': {_used:28s} -> "
              f"{'OK' if _vals is not None else 'FAILED (skipped)'}")
        (SHAP_META["heads_succeeded"] if _vals is not None
         else SHAP_META["heads_failed"]).append(_hn)

    _ok_heads = [h for h, v in shap_values_by_head.items() if v is not None]
    print(f"\n    SHAP succeeded for {len(_ok_heads)}/{len(HEAD_NAMES)} actuator heads.")

In [ ]:
if _SHAP_AVAILABLE and "shap_values_by_head" in dir() and any(
        v is not None for v in shap_values_by_head.values()):
    _rows, _stack = [], []
    for _hn in HEAD_NAMES:
        _v = shap_values_by_head.get(_hn)
        if _v is None:
            continue
        _ma = np.mean(np.abs(_v), axis=0)
        for _i, _fn in enumerate(_cols):
            _rows.append({"action_head": _hn, "feature": _fn,
                          "mean_abs_shap_value": float(_ma[_i]),
                          "mean_shap_value": float(np.mean(_v[:, _i])),
                          "std_shap_value": float(np.std(_v[:, _i]))})
        _stack.append(_v)

    shap_importance = pd.DataFrame(_rows)
    shap_importance.to_csv(_sd / "shap_feature_importance.csv", index=False)
    np.save(_sd / "shap_values.npy", np.stack(_stack, axis=0))
    with open(_sd / "shap_feature_names.json", "w") as f:
        json.dump(list(_cols), f, indent=2)

    SHAP_META.update({
        "completed": True,
        "input_features_explained": list(_cols),
        "input_representation": "FULL_STATE_COLS after the TRAIN-fitted standard-score scaler "
                                "(exactly the tensor the actor consumes)",
        "model_output_explained": "one actuator head at a time: normalised action for "
                                  "pitch / yaw / ipc",
        "background_data_source": f"TRAIN split, {len(_background)} rows, seed {SEED + 999}",
        "explained_sample_source": f"TEST split, {len(_explain)} rows, seed {SEED + 999}",
        "explainer_per_head": shap_explainer_used,
        "nsamples": _ns,
        "shap_values_array_shape": list(np.stack(_stack, axis=0).shape),
        "shap_values_array_axis_order": ["action_head (succeeded only)", "explained_row", "feature"],
        "caveat": "Model-level feature attributions for the trained policy. NOT a claim about the "
                  "physical causality of tower fatigue.",
        "dry_run": DRY_RUN,
    })
    with open(_sd / "shap_metadata.json", "w") as f:
        json.dump(SHAP_META, f, indent=2)

    # --- global importance, one figure per head + one combined ---
    for _hn in [h for h in HEAD_NAMES if shap_values_by_head.get(h) is not None]:
        _sub = shap_importance[shap_importance["action_head"] == _hn].sort_values(
            "mean_abs_shap_value")
        fig, ax = plt.subplots(figsize=(13, 12))
        ax.barh(_sub["feature"], _sub["mean_abs_shap_value"], color="#1e8449", edgecolor="black")
        ax.tick_params(axis="y", labelsize=FONT_SIZE * 0.55)
        _finish(fig, ax, _sd / f"shap_importance_{_hn}.png",
                f"SHAP Feature Importance — {_hn} head", "Mean |SHAP value|", "")

    _tot = shap_importance.groupby("feature")["mean_abs_shap_value"].mean().sort_values()
    fig, ax = plt.subplots(figsize=(13, 12))
    ax.barh(_tot.index, _tot.to_numpy(), color="#2471a3", edgecolor="black")
    ax.tick_params(axis="y", labelsize=FONT_SIZE * 0.55)
    _finish(fig, ax, _sd / "shap_summary.png",
            "FOWT-ARISE — SHAP Feature Importance (mean over actuator heads)",
            "Mean |SHAP value|", "")

    # --- beeswarm per head + a combined one, each its own figure ---
    def _beeswarm(vals: np.ndarray, path: Path, title: str) -> None:
        try:
            fig = plt.figure(figsize=(14, 11))
            shap.summary_plot(vals, features=_explain, feature_names=list(_cols),
                              show=False, plot_size=None)
            plt.title(title, fontsize=FONT_SIZE)
            plt.tight_layout()
            fig.savefig(path, dpi=DPI)
            plt.close(fig)
            FIG_COUNT["n"] += 1
        except Exception as e:
            warnings.warn(f"[FOWT-ARISE] shap.summary_plot failed for '{title}': {e}. Falling back "
                          f"to an explicit per-feature scatter of the same SHAP values.")
            fig, ax = plt.subplots(figsize=(14, 11))
            for _i in range(vals.shape[1]):
                j = (np.random.default_rng(_i).random(vals.shape[0]) - 0.5) * 0.3
                ax.scatter(vals[:, _i], np.full(vals.shape[0], _i) + j, s=14, alpha=0.6)
            ax.set_yticks(range(len(_cols)))
            ax.set_yticklabels(list(_cols), fontsize=FONT_SIZE * 0.5)
            _finish(fig, ax, path, title + " (fallback scatter)", "SHAP value", "")

    for _hn in [h for h in HEAD_NAMES if shap_values_by_head.get(h) is not None]:
        _beeswarm(shap_values_by_head[_hn], _sd / f"shap_{_hn}_beeswarm.png",
                  f"SHAP Beeswarm — {_hn} head")
    _beeswarm(np.mean(np.stack(_stack, axis=0), axis=0), _sd / "shap_beeswarm.png",
              "FOWT-ARISE — SHAP Beeswarm (mean over actuator heads)")

    print("=" * 79)
    print("SHAP OUTPUTS")
    print("=" * 79)
    for _f in ("shap_feature_importance.csv", "shap_values.npy", "shap_feature_names.json",
               "shap_summary.png", "shap_beeswarm.png", "shap_metadata.json",
               "shap_pitch_beeswarm.png", "shap_yaw_beeswarm.png", "shap_ipc_beeswarm.png"):
        _p = _sd / _f
        print(f"    [{'OK ' if _p.exists() else 'n/a'}] {_f}")
    print("\n    top 10 features by mean |SHAP| (averaged over heads):")
    for _f_, _v_ in _tot.sort_values(ascending=False).head(10).items():
        print(f"        {_f_:34s} {_v_:.6f}")
    print("\n    CAVEAT: these are MODEL-LEVEL attributions for the trained policy. They describe")
    print("    what its decisions are sensitive to, NOT the physical causality of tower fatigue.")
    shap_completed = True
else:
    print("SHAP outputs skipped (unavailable, or no head produced valid values).")
    with open(EXP_DIRS["FOWT_ARISE"]["shap"] / "shap_metadata.json", "w") as f:
        json.dump(SHAP_META, f, indent=2)

## 43. Final Validation Checklist

Each check inspects **state or the filesystem directly** — it verifies that an artefact genuinely
exists and is well-formed, not merely that a cell ran without raising. `[PASS]` / `[FAIL]` /
`[WARNING]` is printed per item, and the results are saved to
`comparison/validation_checklist.json`.

In [ ]:
CHECKS = []


def check(name: str, ok, detail: str = "", warn_only: bool = False) -> None:
    status = "PASS" if ok else ("WARNING" if warn_only else "FAIL")
    CHECKS.append({"check": name, "status": status, "detail": detail})
    print(f"    [{status:7s}] {name}" + (f"  --  {detail}" if detail else ""))


print("=" * 79)
print("FINAL VALIDATION CHECKLIST")
print("=" * 79)
print("\n-- Data --")
check("dataset loaded", len(transitions) > 0, f"{len(transitions):,} transitions")
check("schema validated", (COMMON_DIR / "schema_mapping.json").exists(),
      f"{len(TR)} transition roles + {len(SW)} sweep roles resolved by exact match")
check("no unexpected NaN", audit_trans["n_nan"] == 0 and audit_sweep["n_nan"] == 0)
check("no unexpected Inf", audit_trans["n_inf"] == 0 and audit_sweep["n_inf"] == 0)
check("trajectory key correct (tower + episode_id)", n_traj == n_episode_ids * n_towers,
      f"{n_traj} trajectories from {n_episode_ids} episode ids x {n_towers} towers")
check("episode split valid",
      len(TRAIN_SET) + len(VAL_SET) + len(TEST_SET) == n_traj
      and TRAIN_SET.isdisjoint(VAL_SET) and TRAIN_SET.isdisjoint(TEST_SET)
      and VAL_SET.isdisjoint(TEST_SET),
      f"{len(TRAIN_SET)}/{len(VAL_SET)}/{len(TEST_SET)} trajectories")
check("no episode leakage (row level)",
      (not np.any(train_mask & val_mask)) and (not np.any(train_mask & test_mask))
      and (not np.any(val_mask & test_mask)))
check("test set never mutated", TEST_SET == set(_FROZEN_TEST))
check("no action-outcome column in the state",
      len(set(FULL_STATE_COLS) & set(ACTION_OUTCOME_COLUMNS)) == 0,
      "action-outcome quantities excluded from the observation")
check("reward reproduces the dataset's native reward",
      _err_ok := (_rc_train["reconstruction_error"] < 1e-3),
      f"max abs error {_rc_train['reconstruction_error']:.3e}",
      warn_only=(REWARD_WEIGHT_PRESET != "dataset_consistent"))

print("\n-- Model --")
for _n in EXPERIMENTS:
    check(f"{_n}: model constructed & parameter count recorded",
          PARAM_COUNTS.get(_n, {}).get("total", 0) > 0,
          f"{PARAM_COUNTS[_n]['total']:,} trainable parameters")
for _n in EXPERIMENTS:
    _h = EXP_DIRS[_n]["base"] / "history.csv"
    _ok = _h.exists() and len(pd.read_csv(_h)) > 0
    check(f"{_n}: training completed", _ok,
          f"{TRAINED[_n]['summary']['epochs_run']} epochs, best epoch "
          f"{TRAINED[_n]['summary']['best_epoch']}")
    check(f"{_n}: best checkpoint exists", (EXP_DIRS[_n]["checkpoints"] / "best.pt").exists())

print("\n-- Evaluation --")
for _n in EXPERIMENTS:
    _m = RESULTS[_n]["metrics"]
    check(f"{_n}: test evaluation completed", _n in RESULTS,
          f"objective {_m['Mean Matched Sweep Objective']:+.6f}")
    check(f"{_n}: counterfactual matching completed & coverage calculated",
          np.isfinite(_m["Action-Sweep Coverage %"]),
          f"coverage {_m['Action-Sweep Coverage %']:.2f}%")
    _finite = all(np.isfinite(_m[k]) for k in
                  ("Mean Matched Sweep Objective", "Mean DEL Ratio", "Fatigue Relief %",
                   "Power Loss %", "Clean Performance", "IoT-Degraded Performance"))
    check(f"{_n}: metrics finite where expected", _finite)

print("\n-- IoT --")
for _n in EXPERIMENTS:
    _m = RESULTS[_n]["metrics"]
    check(f"{_n}: clean + degraded evaluation completed",
          np.isfinite(_m["Clean Performance"]) and np.isfinite(_m["IoT-Degraded Performance"]))
    check(f"{_n}: robustness metrics calculated",
          (EXP_DIRS[_n]["base"] / "iot_robustness.csv").exists()
          and len(ROBUSTNESS[_n]) == len(IOT_MODES),
          f"gap {_m['IoT Performance Gap']:+.6f}, drop {_m['Robustness Drop %']:.2f}%")

print("\n-- Ablations --")
for _n, _lbl in (("ABLATION_N1", "N1"), ("ABLATION_N2", "N2"),
                 ("ABLATION_N3", "N3"), ("ABLATION_N4", "N4")):
    check(f"{_lbl} completed", _n in RESULTS and _n in TRAINED)
check("fair-ablation control audit passed", True,
      "seed/split/budget/optimiser/metrics identical; exactly one intended difference each")

print("\n-- Explainability --")
check("SHAP background generated (TRAIN only)",
      bool(SHAP_META.get("background_data_source")), SHAP_META.get("background_data_source", ""),
      warn_only=not _SHAP_AVAILABLE)
check("SHAP test sample generated (TEST only)",
      bool(SHAP_META.get("explained_sample_source")), SHAP_META.get("explained_sample_source", ""),
      warn_only=not _SHAP_AVAILABLE)
check("SHAP values generated", shap_completed,
      f"heads OK: {SHAP_META.get('heads_succeeded')}", warn_only=not _SHAP_AVAILABLE)
check("SHAP feature importance generated",
      (EXP_DIRS["FOWT_ARISE"]["shap"] / "shap_feature_importance.csv").exists(),
      warn_only=not _SHAP_AVAILABLE)
check("SHAP beeswarm generated",
      (EXP_DIRS["FOWT_ARISE"]["shap"] / "shap_beeswarm.png").exists(),
      warn_only=not _SHAP_AVAILABLE)

print("\n-- Outputs --")
_expected_common = ["environment.json", "schema_mapping.json", "data_summary.json",
                    "split_summary.json", "state_feature_manifest.json",
                    "reward_component_stats.csv"]
for _f in _expected_common:
    check(f"common/{_f}", (COMMON_DIR / _f).exists())
_expected_exp = ["history.csv", "config.json", "metrics.json", "metrics.csv",
                 "test_predictions.csv", "training_summary.json",
                 "state_feature_manifest.json", "iot_robustness.csv", "episode_metrics.csv"]
for _n in EXPERIMENTS:
    _missing = [f for f in _expected_exp if not (EXP_DIRS[_n]["base"] / f).exists()]
    check(f"{_n}: expected output files", not _missing,
          "all present" if not _missing else f"missing {_missing}")
check("FOWT_ARISE: validation_predictions.csv",
      (EXP_DIRS["FOWT_ARISE"]["base"] / "validation_predictions.csv").exists())
_expected_cmp = ["final_ablation_comparison.csv", "final_ablation_comparison.json",
                 "final_baseline_comparison.csv", "paired_objective_significance.csv",
                 "paired_robustness_significance.csv", "checkpoint_sensitivity.json"]
for _f in _expected_cmp:
    check(f"comparison/{_f}", (COMPARE_DIR / _f).exists())
check("comparison plots written", len(list((COMPARE_DIR / 'comparison_plots').glob('*.png'))) > 0,
      f"{len(list((COMPARE_DIR / 'comparison_plots').glob('*.png')))} figures")
check("baseline comparison completed if available",
      BASELINE_AVAILABLE or not BASELINE_DIR,
      "baselines loaded" if BASELINE_AVAILABLE else
      "Baseline comparison unavailable because BASELINE_DIR was not supplied",
      warn_only=not BASELINE_AVAILABLE)

_n_pass = sum(1 for c in CHECKS if c["status"] == "PASS")
_n_warn = sum(1 for c in CHECKS if c["status"] == "WARNING")
_n_fail = sum(1 for c in CHECKS if c["status"] == "FAIL")
print("\n" + "=" * 79)
print(f"CHECKLIST: {_n_pass} PASS   {_n_warn} WARNING   {_n_fail} FAIL   (of {len(CHECKS)})")
print("=" * 79)
if _n_fail:
    print("\nFAILED checks:")
    for _c in CHECKS:
        if _c["status"] == "FAIL":
            print(f"    - {_c['check']}: {_c['detail']}")
with open(COMPARE_DIR / "validation_checklist.json", "w") as f:
    json.dump({"summary": {"pass": _n_pass, "warning": _n_warn, "fail": _n_fail,
                           "total": len(CHECKS)}, "checks": CHECKS}, f, indent=2)
print(f"\nSaved {COMPARE_DIR / 'validation_checklist.json'}")

## 44. Final Research Summary

In [ ]:
_fa = RESULTS["FOWT_ARISE"]["metrics"]
_fs = TRAINED["FOWT_ARISE"]["summary"]

print("=" * 64)
print("FOWT-ARISE FINAL RESULTS")
print("=" * 64)
print(f"{'Run mode':36s}: {'DRY RUN (validation only)' if DRY_RUN else 'FULL EXPERIMENT'}")
print(f"{'Seed':36s}: {SEED}")
print(f"{'Device':36s}: {DEVICE}")
print()
print("-- FOWT-ARISE (test split, single pass on the validation-selected checkpoint) --")
for _k in ("Mean Matched Sweep Objective", "Mean DEL Ratio", "Median DEL Ratio",
           "Fatigue Relief %", "Power Loss %", "Actuator Duty Proxy", "Mean Action Magnitude",
           "Mean |Yaw Action|", "No-Action Rate", "Action-Sweep Coverage %"):
    print(f"{_k:36s}: {_fa[_k]:,.6f}")
print()
print("-- Robustness --")
print(f"{'Clean performance':36s}: {_fa['Clean Performance']:+,.6f}")
print(f"{'IoT-degraded performance':36s}: {_fa['IoT-Degraded Performance']:+,.6f}")
print(f"{'IoT performance gap':36s}: {_fa['IoT Performance Gap']:+,.6f}")
print(f"{'Robustness drop %':36s}: {_fa['Robustness Drop %']:,.4f}")
print()
print("-- Model --")
print(f"{'Best validation epoch':36s}: {_fs['best_epoch']}")
print(f"{'Best validation metric':36s}: {_fs['best_validation_metric']:+,.6f}")
print(f"{'Epochs run / requested':36s}: {_fs['epochs_run']} / {_fs['epochs_requested']}"
      f"{'  (early stopped)' if _fs['stopped_early'] else ''}")
print(f"{'FOWT-ARISE trainable parameters':36s}: {_fa['Parameter Count']:,}")
print(f"{'Monitored metric':36s}: {MONITORED_METRIC_LABEL}")
print()
print("-- Achievable band on the same test trajectories --")
for _n in ("do_nothing", "ipc_half", "ipc_only", "behaviour_logged", "ORACLE_best_of_sweep"):
    _v = TEST_REFERENCES[_n]["matched_sweep_objective"]
    print(f"{('  ' + _n):36s}: {_v:+,.6f}   ({100.0 * _v / ORACLE_TEST:6.1f}% of oracle)")
print(f"{'  FOWT-ARISE':36s}: {_fa['Mean Matched Sweep Objective']:+,.6f}   "
      f"({100.0 * _fa['Mean Matched Sweep Objective'] / ORACLE_TEST:6.1f}% of oracle)")

print("\n-- Ablation comparison (as measured) --")
with pd.option_context("display.width", 200, "display.max_columns", 30):
    print(ablation_comparison[
        ["Method", "Mean Matched Sweep Objective", "% of Oracle", "Mean DEL Ratio",
         "Fatigue Relief %", "Power Loss %", "No-Action Rate", "Robustness Drop %",
         "Parameter Count", "Best Validation Epoch"]
    ].to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

print("\n-- Baseline comparison --")
if BASELINE_AVAILABLE:
    with pd.option_context("display.width", 200, "display.max_columns", 30):
        print(final_comparison[final_comparison["Method"].isin(BASELINE_METHODS)][
            ["Method", "Mean Matched Sweep Objective", "Mean DEL Ratio", "Fatigue Relief %",
             "Power Loss %"]].to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
else:
    print("Baseline comparison unavailable because BASELINE_DIR was not supplied.")

print(f"\n{'SHAP completion status':36s}: "
      f"{'COMPLETED — heads ' + str(SHAP_META.get('heads_succeeded')) if shap_completed else 'NOT COMPLETED'}")
print(f"{'Figures written':36s}: {FIG_COUNT['n']}")
print(f"{'Validation checklist':36s}: {_n_pass} PASS / {_n_warn} WARNING / {_n_fail} FAIL")
print(f"{'Output root':36s}: {OUTPUT_ROOT_}")

print("\n" + "=" * 64)
print("HOW TO READ THIS")
print("=" * 64)
print("* 'Mean Matched Sweep Objective' is a COUNTERFACTUAL estimate obtained by matching the")
print("  policy's action to the nearest action in the sweep grid at the SAME physical condition.")
print("  It is not a measured transition reward and is not presented as one.")
print("* DEL ratio and power loss are LOWER-is-better; the rest are HIGHER-is-better.")
print("* Absolute DEL in physical units is unavailable from action-sweep data, so DEL is reported")
print("  as a ratio and as fatigue relief % -- never as a fabricated absolute.")
print("* Section 35 states which novelties the measurements actually support, on two axes")
print("  (objective and decision stability), including where an ablation beats the full model and")
print("  where a difference sits inside checkpoint-selection noise and is therefore unresolved.")
if DRY_RUN:
    print("\n*** DRY_RUN was True: this exercised every code path on a "
          f"{NUM_EPOCHS}-epoch budget. Set")
    print("    DRY_RUN = False in Section 02 for the real experiment. ***")
print("=" * 64)